# Answer-only (AO) 蒸留学生の学習ノートブック（Google Colab 用）

研究課題「CoT蒸留による数学能力向上は、教師のどの能力の転移によるものか」を検証するため、
公開 CoT 学生（`jasonrqh/*_Math-CoT-20k_lr5e-5_ep8_bs256`）と比較する **Answer-only (AO) 学生** を、
**公式実装 `verl.trainer.fsdp_sft_trainer_ours` を無変更のまま** 使って学習する。

| 役割 | 使用資産 |
| --- | --- |
| 教師 | `Qwen/Qwen3-32B`（記録のみ。再生成はしない） |
| 学生の学習データ | `jasonrqh/Math-CoT-20k`（20,480 行）。AO の target は `response` 列から抽出した教師の最終回答 `\boxed{...}` |
| Base / CoT 学生 | 設定セルの `SUPPORTED_MODELS` を参照（まず Qwen3 系列） |
| 学習コード | Fork `nshiki08/rethink_sft_generalization_AnsOnlyDistill`（参照元 commit `71a442ea…`） |

## 実行順

| # | セクション | 内容 | 実行するカーネル | GPU |
| --- | --- | --- | --- | --- |
| 0 | 認証 | Colab Secrets（`HF_TOKEN`, 任意で `WANDB_API_KEY`）を読む。値は表示しない | Colab | 不要 |
| 1 | 環境構築 | Fork の clone（`upstream` は push 無効）、Python 3.12 の公式環境を作り公式 pin を導入、公式環境カーネルを起動 | Colab | 不要 |
| 設定 | 設定セル | モデル・revision・保存先・実行モード・学習設定・探索候補・HF 設定をここで編集 | 公式環境 | 不要 |
| 2 | 環境記録・研究条件 | 環境の記録、HF ログイン、1 GPU の FSDP2 勾配確認、対応モデル・AO target の説明 | 公式環境 | 確認のみ |
| 3 | データ取得・AO 抽出・全行監査 | 元 20,480 行・行順・`message`・`advantage` を維持し `teacher_answer` 列を追加 | 公式環境 | 不要 |
| 4 | 長さ・target・loss mask・実効設定 | 公式 tokenizer / chat template / dataset クラスで確認。`data.max_length` の提示 | 公式環境 | 不要 |
| 5 | ドライラン（既定） | 起動コマンド・想定 step 数・保存回数・HF 保存先を確認。学習も HF 書き込みも開始しない | 公式環境 | 不要 |
| 6 | 短い試走 | 少数 step の学習 → HF 保存 → プロセス終了 → HF から取得 → 新プロセスで再開 | 公式環境 | 必要 |
| 7 | baseline | 公式 CoT 設定（lr 5e-5, 8 epoch）の AO 学習 | 公式環境 | 必要 |
| 8 | LR / epoch 探索 | 候補ごとに同じ Base から独立に学習 | 公式環境 | 必要 |
| 再開 | 切断後の再開 | HF の再開可能な checkpoint 一覧 → 選択 → 検証 → `resume_path` で再開 | 公式環境 | 必要 |
| 9 | 評価 | 論文と同じ評価（公式 `math_eval_budget.py`: MATH500 avg@3, AIME24 avg@10）を各 step で実行（論文と同じく選択はしない） | 評価用環境（vLLM） | 必要 |
| 10 | 最終モデル | `verl.model_merger` で HF 形式へ変換、HF 保存、読み込み・生成確認、Model Card | 公式環境 | 必要 |

## 2 つのカーネル

Colab の Python は 3.13（2026-08 から）で、公式 `requirements.txt` の pin（numpy 1.26.4, ray 2.43.0 など）は Python 3.12 までしか入らない。
セル 1 が Python 3.12 の別環境（`/content/ao_work/train_env`）に公式の pin を入れ、その Python で 2 つ目のカーネルを起動する。
設定セル以降は先頭の `%%ao` によりそのカーネルで実行される（出力・エラー・停止ボタンは中継される）。Colab の「セッションを再起動」をしたらセル 0 から実行し直す。

| GPU | 環境 |
| --- | --- |
| A100（推奨） | 公式 `requirements.txt` のまま（torch 2.6.0, flash-attn 2.7.4.post1） |
| G4（RTX PRO 6000 Blackwell） | torch 2.6.0 が動かない。セル 1 の `ALLOW_BLACKWELL_TORCH_DEVIATION=True` で torch 2.7.1+cu128（と torch が要求する sympy 1.13.3）を使い、公式との差として記録する |
| T4 など（compute capability < 8.0） | flash-attn 2 が動かないので不可 |

## 公式 CoT 学習との条件合わせ（GPU 台数）

公式 CoT 学生は 8 GPU で学習された。公式 trainer は 1 GPU 内の micro batch の勾配を和で蓄積し、GPU 間は平均する。
Colab の GPU 台数 N（1, 2, 4）でそのまま動かすと、micro batch（4 行）の組み合わせと勾配の大きさ（8/N 倍）が公式と変わる。
`EMULATE_OFFICIAL_WORLD_SIZE=True`（既定）では、公式コードを変えずに次の 2 点で公式 8 GPU の計算を再現する。

- 学習用 parquet の行を並べ替え、各 step の各 micro batch を公式 8 GPU と同じ 4 行にする（元の AO parquet は変更しない）
- `trainer.importance_sampling_mode=adv-only` と `advantage=N/8` で loss を N/8 倍し、勾配を公式と同じ大きさにする（ログの `train/loss` も N/8 倍になるので、表示では公式スケールに戻す）

これで学習手順の違いは target 列だけになる。CPU 上で公式 trainer と小型モデルを使った検証では、fp32 で公式 8 プロセスの run と移動量比 1e-6 で一致した。bf16（公式設定）では勾配の加算順序などによる丸め差が残る（6 step 後に 1 GPU で 1.5〜3.6%、並べ替えなしの 1 GPU は 7〜14%）。残る差の一覧はドライラン（セクション 5）と実験記録に出る。

checkpoint は公式と同じ 10 step ごとに保存する。HF へは、再開用に 80 step ごと（`RESUME_CKPT_EVERY`）と最終 step を optimizer 状態込みで、公開 CoT 学生と同じ残りの step（10, 20, 40）を重みだけ転送する（1.7B の baseline で約 93 GB）。

## 注意

- 既定の `RUN_MODE` は `"dry_run"`。学習と HF 書き込みは `"trial"` / `"train"` にしないと始まらない。
- Colab の切断でプロセスは消える。学習中の checkpoint は監視スレッドが保存完了ごとに HF へ転送し、過去 step は消さない。
- 公式実装（`verl/`, `training_scripts/`）は変更しない。追加処理はすべてこのノートブックにある。
- `git push` 先は Fork のみ。オリジナル (`Nebularaid2000/...`) は fetch 専用で push URL を無効化する。
- このノートブックの静的検証は CPU 環境で行った。GPU を要する項目（試走・学習・再開・変換）は各セルの実行ログが確認根拠になる。未実行の項目を「動作確認済み」とは書かない。

## 0. 認証

このセルとセル 1 は Colab のカーネルで実行する。

In [ ]:
# @title 0. 認証（最初のセル、Colab のカーネルで実行）: Colab Secrets / 環境変数からトークンを読む。値は表示しない
# 読んだトークンは環境変数に入れ、セル 1 が起動する公式環境のカーネルへ引き継ぐ。HF へのログイン確認はセクション 2 で行う。
import os


def _get_secret(name):
    value = None
    try:
        from google.colab import userdata  # type: ignore

        try:
            value = userdata.get(name)
        except Exception:
            value = None
    except ImportError:
        pass
    if not value:
        value = os.environ.get(name)
    return value or None


for _name, _note in (("HF_TOKEN", "HF へのアップロード・再開は無効。Colab Secrets に HF_TOKEN（write）を登録して再実行する"),
                     ("WANDB_API_KEY", "公式スクリプトと同じ offline mode を使う")):
    _v = _get_secret(_name)
    if _v:
        os.environ[_name] = _v
        print(f"{_name}: found (masked)")
    else:
        print(f"{_name}: なし。{_note}")
del _v

## 1. 環境構築（Fork の clone・公式環境の作成・公式環境カーネルの起動）

Colab の Python（3.13）には公式の pin が入らないので、Python 3.12 の別環境に公式の pin を入れ、その環境で 2 つ目のカーネルを起動する。**設定セル以降の `%%ao` で始まるセルは公式環境のカーネルで実行される。** Colab の『セッションを再起動』をした場合はセル 0 から実行し直す。

In [ ]:
# @title 1. 環境構築（Colab のカーネルで実行）: Fork の clone と公式環境（Python 3.12 + 公式 pin）の準備
# Colab の Python（2026-08 から 3.13）には公式の pin（numpy 1.26.4, ray 2.43.0 など。cp312 まで）が入らない。
# そこで公式の pin を別の Python 3.12 環境（uv venv）に入れ、その Python で 2 つ目の Jupyter カーネルを起動する。
# 設定セル以降の %%ao で始まるセルはそのカーネルで実行される（出力・エラー・停止ボタンは中継する）。
import os, sys, json, subprocess, shutil, hashlib, platform, time, signal, queue

# =============================================================================
# 環境の設定（通常は変更しない）
# =============================================================================
FORK_REPO_URL = "https://github.com/nshiki08/rethink_sft_generalization_AnsOnlyDistill"
FORK_REF = "main"                 # ブランチ名または commit。実行時に HEAD の commit を記録する
UPSTREAM_REPO_URL = "https://github.com/Nebularaid2000/rethink_sft_generalization"   # 参照専用（push 無効化）
UPSTREAM_REFERENCE_COMMIT = "71a442ea8f0adc4a1df4529d3c43393ac6e504fd"               # 再現用の参照 commit

TRAIN_PYTHON_VERSION = "3.12"     # 公式 pin の wheel がある最新版（numpy 1.26.4 / ray 2.43.0 は cp312 まで）
UV_VERSION = "0.8.17"

# 公式 requirements.txt の pin から、学習に必要な範囲を抜き出したもの（vllm/sglang/評価系は含めない）。torch 系は GPU_PROFILES で決める
PIP_PINNED = [
    "transformers==4.52.4", "tokenizers==0.21.4", "accelerate==1.10.1", "datasets==4.0.0",
    "tensordict==0.9.1", "torchdata==0.11.0", "peft==0.17.1",
    "hydra-core==1.3.2", "omegaconf==2.3.0", "wandb==0.21.1", "ray[default]==2.43.0",
    "codetiming==1.4.0", "dill==0.3.8", "pyarrow==21.0.0", "numpy==1.26.4",
    "math-verify==0.7.0", "latex2sympy2_extended==1.10.1", "pylatexenc==2.10",
    "sympy==1.13.1", "antlr4-python3-runtime==4.9.3",   # math-verify の判定結果（AO 抽出の照合）に影響するので公式と揃える
    "huggingface_hub==0.34.4", "hf-xet==1.1.9", "safetensors==0.6.2", "einops==0.8.1", "sentencepiece==0.2.1",
    "regex==2025.7.34", "word2number==1.1",   # evaluation/math_eval/utils/parser.py（既存の評価 parser）が使う
    "Jinja2==3.1.6",                          # chat template の描画に使われ、プロンプトの token に影響し得るので公式と揃える
    "pandas==2.3.3",                          # 公式 requirements.txt に pin が無い（setup.py は無指定）。導入時期で 3.x に変わらないよう 2.x の最終版に固定する
]
# 直接の pin に加え、公式 requirements.txt（clone した Fork にある）を制約 -c として使い、間接的な依存も公式と同じ版にそろえる。
# requirements.txt 自体に矛盾する pin が 2 つある（protobuf 5.29.5 と opentelemetry-proto の <5、typer 0.9.4 と fastapi-cli の >=0.15.1）ので外す
CONSTRAINT_EXCLUDE = ["protobuf", "typer"]
KERNEL_PACKAGES = ["ipykernel==6.29.5"]   # 公式環境側でカーネルを動かすためだけに使う（学習には関与しない）
EXTRA_PIP_PACKAGES = []                   # import 確認で不足が出た場合に追加

_FA = "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1"
FLASH_ATTN_VERSION = "2.7.4.post1"        # 公式 pin。両プロファイルで同じ版
GPU_PROFILES = {
    # A100 / L4 / H100 など（compute capability 8.x, 9.x）: 公式 requirements.txt のまま
    "official": dict(torch=["torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0"], index_url=None, pin_overrides={},
                     flash_attn_wheel=f"{_FA}+cu12torch2.6cxx11abiFALSE-cp312-cp312-linux_x86_64.whl", cxx11abi=False, nccl=[2, 21, 5], constraint_exclude=[],
                     deviation=None),
    # G4（RTX PRO 6000 Blackwell, sm_120）など compute capability 10 以上: torch 2.6.0 には sm_100/sm_120 のコードが無く動かない。
    # sm_120 に対応した最初の公式 torch は 2.7.0（cu128）。flash-attn は同じ 2.7.4.post1 の torch2.7 用 wheel（sm_120 を含む）。
    # torch 2.7.1 は sympy>=1.13.3 を要求するので、公式の sympy==1.13.1 と両立しない（1.13.3 に上げる）。
    "blackwell": dict(torch=["torch==2.7.1+cu128", "torchvision==0.22.1+cu128", "torchaudio==2.7.1+cu128"],
                      index_url="https://download.pytorch.org/whl/cu128", pin_overrides={"sympy": "sympy==1.13.3"},
                      constraint_exclude=["torch", "torchvision", "torchaudio", "triton", "sympy", "xformers", "vllm", "nvidia-*"],
                      flash_attn_wheel=f"{_FA}+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl", cxx11abi=True, nccl=[2, 26, 2],
                      deviation="torch 2.6.0 → 2.7.1+cu128（torchvision/torchaudio も対応版）、flash-attn は同版の torch2.7 用 wheel、"
                                "sympy 1.13.1 → 1.13.3（torch 2.7.1 の要求。math-verify の照合結果はセクション 3 のゲートで確認）。"
                                "勾配の GPU 間平均・累積・clip・state_dict 読み込みの実装は 2.6 と同じことをソースで確認済み。"
                                "NCCL 2.21.5→2.26.2, cuBLAS 12.4→12.8 などカーネル実装は変わるので丸め誤差の範囲の差が出る"),
}
ALLOW_BLACKWELL_TORCH_DEVIATION = False   # G4 を使うときだけ True にする（公式 torch からの変更として記録される）

# 論文の数学評価（evaluation/math_eval/math_eval_budget.py, vLLM）用の環境。セクション 9 で必要になったときに作る（学習用の公式環境とは別）。
# vLLM 0.8.5 は公式 requirements.txt の pin（torch==2.6.0 を要求）。依存が多いので学習用の環境には入れない。
# 公式 requirements.txt と同じ版: 下記すべて（matplotlib と pandas は requirements.txt に無い。評価スクリプトが import するので追加）
EVAL_PIP_PINNED = [
    "vllm==0.8.5", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "xformers==0.0.29.post2", "triton==3.2.0",
    "transformers==4.52.4", "tokenizers==0.21.4", "numpy==1.26.4", "datasets==4.0.0", "pyarrow==21.0.0",
    "math-verify==0.7.0", "latex2sympy2_extended==1.10.1", "sympy==1.13.1", "antlr4-python3-runtime==4.9.3", "word2number==1.1",
    "regex==2025.7.34", "compressed-tensors==0.9.3", "xgrammar==0.1.18", "outlines==0.1.11", "huggingface_hub==0.34.4", "hf-xet==1.1.9",
    "pandas==2.3.3", "matplotlib==3.10.3",
]

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
WORK_DIR = os.environ.get("AO_WORK_DIR") or ("/content/ao_work" if IN_COLAB else os.path.abspath("./ao_work"))
REPO_DIR = f"{WORK_DIR}/repo"
TRAIN_ENV_DIR = f"{WORK_DIR}/train_env"   # 公式環境（Python 3.12 venv）
RECORD_DIR = f"{WORK_DIR}/records"
for _d in (WORK_DIR, RECORD_DIR):
    os.makedirs(_d, exist_ok=True)


def sh(cmd, cwd=None, check=True, env=None):
    """subprocess の薄いラッパ。失敗時はコマンドと出力の末尾を表示する"""
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True, env=env)
    if check and r.returncode != 0:
        print("COMMAND FAILED:", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
        print((r.stdout or "")[-4000:], (r.stderr or "")[-4000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return (r.stdout or "").strip()


# ---- 1.1 GPU の確認とプロファイルの決定 -----------------------------------------------------------
def gpu_info():
    if shutil.which("nvidia-smi") is None:
        return []
    out = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version,compute_cap", "--format=csv,noheader,nounits"], check=False)
    gpus = []
    for line in out.splitlines():
        p = [x.strip() for x in line.split(",")]
        if len(p) >= 5:
            gpus.append(dict(index=int(p[0]), name=p[1], vram_mb=int(float(p[2])), driver=p[3], compute_cap=p[4]))
    return gpus


GPUS = gpu_info()
print(f"GPU x{len(GPUS)}:", GPUS or "なし（dry_run とデータ処理のみ可能）")
_caps = sorted({tuple(int(v) for v in g["compute_cap"].split(".")) for g in GPUS})
if len(_caps) > 1:
    raise SystemExit(f"種類の違う GPU が混在している: {_caps}")
GPU_CAP = _caps[0] if _caps else None
if GPU_CAP is not None and GPU_CAP < (8, 0):
    raise SystemExit(f"GPU compute capability {GPU_CAP} では flash-attn 2 が動かない（Ampere 以上が必要）。"
                     "公式 trainer は flash_attention_2 固定なので、ランタイムを A100 などに変更する（コードは変更しない）")
if GPU_CAP is not None and GPU_CAP >= (10, 0):
    if not ALLOW_BLACKWELL_TORCH_DEVIATION:
        raise SystemExit(
            f"GPU {GPUS[0]['name']}（compute capability {GPU_CAP[0]}.{GPU_CAP[1]}）では公式の torch 2.6.0 が動かない（sm_100/sm_120 のコードが無い）。\n"
            "  - 公式環境のまま学習する: ランタイムを A100 に変更する（推奨）\n"
            "  - このまま使う: ALLOW_BLACKWELL_TORCH_DEVIATION=True にして再実行する。torch 2.7.1+cu128 を使い、公式との差として記録する\n"
            f"    変更内容: {GPU_PROFILES['blackwell']['deviation']}")
    GPU_PROFILE = "blackwell"
else:
    GPU_PROFILE = "official"
PROFILE = GPU_PROFILES[GPU_PROFILE]
print("GPU プロファイル:", GPU_PROFILE, "| 公式からの変更:", PROFILE["deviation"] or "なし")

# ---- 1.2 Fork の clone と remote 設定 -------------------------------------------------------------
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("cloning", FORK_REPO_URL, "->", REPO_DIR)
    sh(["git", "clone", "--quiet", FORK_REPO_URL, REPO_DIR])
_origin = sh(["git", "remote", "get-url", "origin"], cwd=REPO_DIR)
assert _origin.rstrip("/").removesuffix(".git") == FORK_REPO_URL.rstrip("/").removesuffix(".git"), f"origin が Fork ではない: {_origin}"
if "upstream" not in sh(["git", "remote"], cwd=REPO_DIR).split():
    sh(["git", "remote", "add", "upstream", UPSTREAM_REPO_URL], cwd=REPO_DIR)
sh(["git", "remote", "set-url", "--push", "upstream", "DISABLED_no_push_to_upstream"], cwd=REPO_DIR)   # オリジナルへの push を無効化
assert "DISABLED_no_push_to_upstream" in sh(["git", "remote", "get-url", "--push", "upstream"], cwd=REPO_DIR)
print(sh(["git", "remote", "-v"], cwd=REPO_DIR))
sh(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
try:
    sh(["git", "fetch", "--quiet", "upstream"], cwd=REPO_DIR)
    UPSTREAM_FETCHED = True
except RuntimeError:
    UPSTREAM_FETCHED = False
    print("WARN: upstream の fetch に失敗（参照 commit との差分確認は Fork 内の履歴で行う）")
try:
    _target = sh(["git", "rev-parse", "--verify", f"origin/{FORK_REF}^{{commit}}"], cwd=REPO_DIR)
except RuntimeError:
    _target = sh(["git", "rev-parse", "--verify", f"{FORK_REF}^{{commit}}"], cwd=REPO_DIR)
sh(["git", "checkout", "--quiet", "--detach", _target], cwd=REPO_DIR)
FORK_COMMIT = sh(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
print("Fork HEAD:", FORK_COMMIT, "(ref:", FORK_REF + ")")
try:
    _is_anc = subprocess.run(["git", "merge-base", "--is-ancestor", UPSTREAM_REFERENCE_COMMIT, "HEAD"], cwd=REPO_DIR).returncode == 0
    OFFICIAL_DIFF_STAT = sh(["git", "diff", "--stat", UPSTREAM_REFERENCE_COMMIT, "HEAD", "--", "verl", "training_scripts"], cwd=REPO_DIR)
    OFFICIAL_CODE_UNCHANGED = (OFFICIAL_DIFF_STAT == "")
    print("参照 commit は HEAD の祖先か:", _is_anc, "| 公式コード (verl/, training_scripts/) の差分:", "なし" if OFFICIAL_CODE_UNCHANGED else "\n" + OFFICIAL_DIFF_STAT)
except RuntimeError:
    OFFICIAL_DIFF_STAT, OFFICIAL_CODE_UNCHANGED = "unknown (reference commit not available)", None
    print("WARN: 参照 commit がローカルに無く差分を確認できない")

# ---- 1.3 前回の公式環境カーネルと学習プロセスの終了 ----------------------------------------------------
# Colab の「セッションを再起動」やセル 1 の再実行では、前回の公式環境カーネル（GPU メモリを持っている可能性）と、
# そこから別の process group で起動した学習プロセス（torchrun）が残る。pid ファイルに記録した process group を、
# cmdline が期待どおりのプロセスだけ終了する（pid の再利用で無関係なプロセスを止めない。自分自身は止めない）。
def _group_members(pgid):
    out = []
    for d in os.listdir("/proc"):
        if not d.isdigit():
            continue
        try:
            stat = open(f"/proc/{d}/stat").read()
            if int(stat.rsplit(")", 1)[1].split()[2]) == pgid:
                out.append((int(d), open(f"/proc/{d}/cmdline", "rb").read().replace(b"\0", b" ").decode(errors="replace")))
        except (OSError, IndexError, ValueError):
            pass
    return out


def kill_recorded_group(pidfile, must_contain, label):
    try:
        pgid = int(open(pidfile).read().strip())
    except (OSError, ValueError):
        return
    members = [(p, c) for p, c in _group_members(pgid) if p != os.getpid()]
    if pgid == os.getpgid(0) or not members or not any(all(m in c for m in must_contain) for _, c in members):
        return   # 既に終了している、または pid が別のプロセスに再利用されている
    try:
        os.killpg(pgid, signal.SIGKILL)
        print(f"前回の{label}（process group {pgid}, {len(members)} プロセス）を終了した")
    except ProcessLookupError:
        pass


TRAIN_PY = f"{TRAIN_ENV_DIR}/bin/python"
kill_recorded_group(f"{WORK_DIR}/ao-trainer.pgid", ["torch.distributed.run"], "学習プロセス")
kill_recorded_group(f"{WORK_DIR}/ao-train-env.pid", ["ipykernel_launcher", TRAIN_ENV_DIR], "公式環境カーネル")

# ---- 1.4 公式環境（Python 3.12 venv）の作成と公式 pin の導入 ------------------------------------------
# Colab は uv 用の制約ファイルを UV_* 環境変数で指定していることがある（Colab 自身の numpy 2.x などに固定される）。公式環境には使わない
_uv_env = {k: v for k, v in os.environ.items() if not k.startswith("UV_")}
_uv_env["UV_CACHE_DIR"] = os.environ.get("AO_UV_CACHE_DIR", f"{WORK_DIR}/uv_cache")
_uv_env["UV_PYTHON_INSTALL_DIR"] = f"{WORK_DIR}/uv_python"
# UV_LINK_MODE は既定（Linux では hardlink）。cache と venv が同じディスクなので CUDA 系 wheel を二重に置かない
try:
    sh([sys.executable, "-m", "uv", "--version"])
except RuntimeError:
    sh([sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"])
UV = [sys.executable, "-m", "uv", "--no-config"]
_pins = [PROFILE["pin_overrides"].get(p.split("==")[0].split("[")[0].lower(), p) for p in PIP_PINNED + EXTRA_PIP_PACKAGES]

def write_constraints(path, exclude):
    """公式 requirements.txt の == 行を制約ファイルにする（exclude は小文字・ハイフン区切りの名前か glob）"""
    import fnmatch
    keep, dropped = [], []
    for line in open(f"{REPO_DIR}/requirements.txt"):
        line = line.split("#")[0].strip()
        if "==" not in line:
            continue
        name = line.split("==")[0].strip().lower().replace("_", "-")
        (dropped if any(fnmatch.fnmatch(name, pat) for pat in exclude) else keep).append(line)
    with open(path, "w") as f:
        f.write("\n".join(keep) + "\n")
    return keep, dropped


TRAIN_CONSTRAINTS = f"{WORK_DIR}/constraints_train_{GPU_PROFILE}.txt"
_cons_keep, _cons_dropped = write_constraints(TRAIN_CONSTRAINTS, CONSTRAINT_EXCLUDE + PROFILE["constraint_exclude"])
EVAL_CONSTRAINTS = f"{WORK_DIR}/constraints_eval.txt"          # 評価用環境（vLLM 0.8.5、official プロファイルのみ）
write_constraints(EVAL_CONSTRAINTS, CONSTRAINT_EXCLUDE)
print(f"公式 requirements.txt を制約に使う: {len(_cons_keep)} 件（除外 {len(_cons_dropped)} 件: {sorted(set(l.split('==')[0] for l in _cons_dropped))}）")
_torch_index = os.environ.get("AO_TORCH_INDEX_URL") or PROFILE["index_url"]   # AO_TORCH_INDEX_URL はローカル（CPU）検証用
_want = dict(python=TRAIN_PYTHON_VERSION, torch=PROFILE["torch"], index=_torch_index, pins=_pins + KERNEL_PACKAGES,
             flash_attn=PROFILE["flash_attn_wheel"] if GPU_CAP else None, constraints=_cons_keep)
_want_hash = hashlib.sha256(json.dumps(_want, sort_keys=True).encode()).hexdigest()[:16]
_marker = f"{TRAIN_ENV_DIR}/ao_env_ok.json"
if os.path.isfile(_marker) and json.load(open(_marker)).get("hash") == _want_hash and os.path.isfile(TRAIN_PY):
    print("公式環境は作成済み（同じ pin）:", TRAIN_ENV_DIR)
else:
    t0 = time.time()
    shutil.rmtree(TRAIN_ENV_DIR, ignore_errors=True)
    sh(UV + ["venv", "--quiet", "--python", TRAIN_PYTHON_VERSION, "--python-preference", "only-managed", TRAIN_ENV_DIR], env=_uv_env)
    _idx = ["--index-url", _torch_index, "--extra-index-url", "https://pypi.org/simple", "--index-strategy", "unsafe-best-match"] if _torch_index else []
    print(f"公式 pin を導入中（{len(_want['pins']) + 3} packages。数分かかる）...")
    sh(UV + ["pip", "install", "--quiet", "--python", TRAIN_PY, "-c", TRAIN_CONSTRAINTS] + _idx + PROFILE["torch"] + _want["pins"], env=_uv_env)
    if GPU_CAP:
        # flash-attn: 公式 trainer は attn_implementation='flash_attention_2' 固定。依存（torch, einops）は導入済みなので --no-deps
        sh(UV + ["pip", "install", "--quiet", "--python", TRAIN_PY, "--no-deps", PROFILE["flash_attn_wheel"]], env=_uv_env)
    json.dump(dict(hash=_want_hash, want=_want, created_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())), open(_marker, "w"), indent=2)
    print(f"公式環境を作成した: {TRAIN_ENV_DIR}（{time.time() - t0:.0f} 秒）")
TRAIN_ENV_FREEZE = f"{RECORD_DIR}/train_env_freeze.txt"   # 公式環境の全パッケージの版（記録）
with open(TRAIN_ENV_FREEZE, "w") as f:
    f.write(sh(UV + ["pip", "freeze", "--python", TRAIN_PY], env=_uv_env) + "\n")

# ---- 1.5 公式環境の確認（Python, torch, GPU アーキテクチャ, flash-attn, 公式 trainer の import） -------------------
_TRAIN_ENV_VARS = {k: v for k, v in os.environ.items() if not k.startswith("UV_") and k not in ("PYTHONPATH", "PYTHONHOME", "MPLBACKEND", "VIRTUAL_ENV")}
_TRAIN_ENV_VARS.update(PYTHONPATH=REPO_DIR, VIRTUAL_ENV=TRAIN_ENV_DIR, PATH=f"{TRAIN_ENV_DIR}/bin" + os.pathsep + os.environ.get("PATH", ""),
                       AO_WORK_DIR=WORK_DIR)
_probe_src = r'''
import json, sys, importlib, platform
mods = ["torch", "transformers", "tokenizers", "jinja2", "accelerate", "datasets", "tensordict", "torchdata", "peft", "hydra", "omegaconf",
        "wandb", "ray", "codetiming", "numpy", "pandas", "pyarrow", "math_verify", "huggingface_hub", "triton", "flash_attn", "ipykernel"]
vers = {}
for m in mods:
    try:
        vers[m] = getattr(importlib.import_module(m), "__version__", "n/a")
    except ImportError:
        vers[m] = "not installed"
import torch
p = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, abi=bool(torch._C._GLIBCXX_USE_CXX11_ABI),
         cuda_available=torch.cuda.is_available(), arch_list=torch.cuda.get_arch_list() if torch.cuda.is_available() else [],
         cap=list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
         cudnn=torch.backends.cudnn.version() if torch.cuda.is_available() else None,
         nccl=list(torch.cuda.nccl.version()) if torch.cuda.is_available() else None)
if p["cuda_available"]:
    import flash_attn, flash_attn.bert_padding  # noqa: F401
    from flash_attn import flash_attn_func
    x = torch.randn(64, 64, device="cuda", dtype=torch.bfloat16)
    p["cuda_matmul_ok"] = bool(torch.isfinite(x @ x).all())   # その GPU 用のカーネルが無ければここで失敗する
    q, k, v = (torch.randn(2, 128, 4, 64, device="cuda", dtype=torch.bfloat16, requires_grad=True) for _ in range(3))
    o = flash_attn_func(q, k, v, causal=True)
    o.float().sum().backward()                                 # flash-attn の forward / backward カーネルを GPU で実行する
    p["flash_attn_kernel_ok"] = bool(torch.isfinite(o).all() and torch.isfinite(q.grad).all())
import verl.trainer.fsdp_sft_trainer_ours  # noqa: F401
import verl.model_merger.fsdp_model_merger  # noqa: F401
print("PROBE_OK " + json.dumps(dict(probe=p, packages=vers)))
'''
_r = subprocess.run([TRAIN_PY, "-c", _probe_src], capture_output=True, text=True, env=_TRAIN_ENV_VARS, cwd=REPO_DIR)
_ok = [l for l in _r.stdout.splitlines() if l.startswith("PROBE_OK ")]
if _r.returncode != 0 or not _ok:
    print(_r.stdout[-3000:], _r.stderr[-5000:])
    raise RuntimeError("公式環境の確認に失敗。不足パッケージは EXTRA_PIP_PACKAGES に追加して再実行する")
_probe = json.loads(_ok[-1][len("PROBE_OK "):])
TORCH_PROBE, PKG_VERSIONS = _probe["probe"], _probe["packages"]
assert TORCH_PROBE["python"].startswith(TRAIN_PYTHON_VERSION + "."), TORCH_PROBE["python"]
if GPU_CAP:
    # 同じ major の小さい minor 向けコード（例: L4 の sm_89 で sm_86/sm_80）は動く。PTX（compute_XY）は XY <= 実機なら JIT で動く
    def _arch_ok(arch_list, cap):
        for a in arch_list:
            kind, _, num = a.partition("_")
            if not num.isdigit():
                continue
            major, minor = int(num[:-1]), int(num[-1])
            if kind == "sm" and major == cap[0] and minor <= cap[1]:
                return True
            if kind == "compute" and (major, minor) <= tuple(cap):
                return True
        return False

    assert _arch_ok(TORCH_PROBE["arch_list"], GPU_CAP), f"torch {TORCH_PROBE['torch']} に compute capability {GPU_CAP} 用のコードが無い: {TORCH_PROBE['arch_list']}"
    assert TORCH_PROBE.get("cuda_matmul_ok"), "GPU 上の行列積に失敗"
    assert TORCH_PROBE.get("flash_attn_kernel_ok"), "flash-attn のカーネル実行に失敗"
    assert TORCH_PROBE["abi"] == PROFILE["cxx11abi"], f"torch の C++ ABI {TORCH_PROBE['abi']} が flash-attn wheel（cxx11abi={PROFILE['cxx11abi']}）と合わない"
    if TORCH_PROBE["nccl"] != PROFILE["nccl"]:
        print(f"WARN: NCCL {TORCH_PROBE['nccl']} が想定 {PROFILE['nccl']} と異なる（記録する）")
print("公式環境:", {k: TORCH_PROBE[k] for k in ("python", "torch", "cuda", "cap", "cudnn", "nccl")})
print("packages:", PKG_VERSIONS)

ENV_BOOTSTRAP = dict(
    kernel_python=platform.python_version(), train_python=TORCH_PROBE["python"], train_env_dir=TRAIN_ENV_DIR, train_env_hash=_want_hash,
    gpu_profile=GPU_PROFILE, gpu_profile_deviation=PROFILE["deviation"], torch_pins=PROFILE["torch"], torch_index_url=_torch_index,
    flash_attn_version=FLASH_ATTN_VERSION, flash_attn_wheel=_want["flash_attn"], pip_pinned=_pins, pin_overrides=PROFILE["pin_overrides"],
    gpus=GPUS, torch_probe=TORCH_PROBE, packages=PKG_VERSIONS, in_colab=IN_COLAB, work_dir=WORK_DIR, repo_dir=REPO_DIR,
    fork_repo=FORK_REPO_URL, fork_ref=FORK_REF, fork_commit=FORK_COMMIT, upstream_repo=UPSTREAM_REPO_URL,
    upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, upstream_fetched=UPSTREAM_FETCHED,
    official_code_unchanged=OFFICIAL_CODE_UNCHANGED, official_diff_stat=OFFICIAL_DIFF_STAT,
    # セクション 9 が評価用の環境を作るときに使う（uv は Colab のカーネルの Python に入っている）
    uv_cmd=UV, uv_cache_dir=_uv_env["UV_CACHE_DIR"], uv_python_install_dir=_uv_env["UV_PYTHON_INSTALL_DIR"],
    eval_env_dir=f"{WORK_DIR}/eval_env", eval_pip_pinned=EVAL_PIP_PINNED, eval_constraints=EVAL_CONSTRAINTS,
    train_constraints=TRAIN_CONSTRAINTS, constraint_excluded=_cons_dropped, train_env_freeze=TRAIN_ENV_FREEZE,
)
ENV_BOOTSTRAP_PATH = f"{RECORD_DIR}/env_bootstrap.json"
json.dump(ENV_BOOTSTRAP, open(ENV_BOOTSTRAP_PATH, "w"), indent=2, ensure_ascii=False)


# ---- 1.6 公式環境のカーネルを起動し、%%ao マジックを登録する --------------------------------------------
class TrainEnvKernel:
    """公式環境（別 Python）で動く Jupyter カーネル。コードを送り、出力・エラー・停止を中継する"""

    def __init__(self, python_exe, workdir, env, cwd, name="ao-train-env"):
        self.python_exe, self.workdir, self.env, self.cwd, self.name = python_exe, workdir, env, cwd, name
        self.pidfile = os.path.join(workdir, f"{name}.pid")
        self.km = self.kc = None

    def start(self):
        from jupyter_client import KernelManager
        from jupyter_client.kernelspec import KernelSpecManager
        spec_dir = os.path.join(self.workdir, "kernelspec", self.name)
        os.makedirs(spec_dir, exist_ok=True)
        with open(os.path.join(spec_dir, "kernel.json"), "w") as f:
            json.dump(dict(argv=[self.python_exe, "-m", "ipykernel_launcher", "-f", "{connection_file}"], display_name=self.name, language="python"), f)
        ksm = KernelSpecManager()
        ksm.kernel_dirs = [os.path.dirname(spec_dir)]
        # IPC（Unix socket）で通信する。パス長の上限（108 byte）があるので短いパスにする
        self.km = KernelManager(kernel_name=self.name, kernel_spec_manager=ksm, transport="ipc", ip=f"/tmp/aok{os.getpid()}")
        # 新しい process group（後で子プロセスごと終了できる）。カーネルの fd 直書き出力（subprocess など）は既に中継されるのでログファイルへ
        self._log = open(os.path.join(self.workdir, f"{self.name}.log"), "ab")
        self.km.start_kernel(env=dict(self.env), cwd=self.cwd, start_new_session=True, stdout=self._log, stderr=self._log)
        with open(self.pidfile, "w") as f:
            f.write(str(self.km.provisioner.pid if getattr(self.km, "provisioner", None) else self.km.kernel.pid))
        self.kc = self.km.client()
        self.kc.start_channels()
        self.kc.wait_for_ready(timeout=180)
        return self

    def alive(self):
        return self.km is not None and self.km.is_alive()

    def run(self, code):
        if not self.alive():
            raise RuntimeError("公式環境カーネルが動いていない（メモリ不足で落ちた可能性）。セル 1（環境構築）を再実行する")
        msg_id = self.kc.execute(code, store_history=False, allow_stdin=False)
        error, interrupted = None, False
        while True:
            try:   # 停止ボタン（KeyboardInterrupt）がループのどこで届いても公式環境カーネルに割り込みを送り、終了を待つ
                try:
                    msg = self.kc.get_iopub_msg(timeout=1)
                except queue.Empty:
                    if not self.alive():
                        raise RuntimeError("公式環境カーネルが終了した（メモリ不足など）。セル 1（環境構築）から実行し直す")
                    continue
                if msg.get("parent_header", {}).get("msg_id") != msg_id:
                    continue
                t, c = msg["msg_type"], msg["content"]
                if t == "stream":
                    (sys.stderr if c["name"] == "stderr" else sys.stdout).write(c["text"])
                elif t in ("display_data", "execute_result"):
                    try:
                        from IPython.display import publish_display_data
                        publish_display_data(c["data"], c.get("metadata", {}))
                    except Exception:
                        print(c["data"].get("text/plain", ""))
                elif t == "error":
                    error = c
                    sys.stderr.write("\n".join(c["traceback"]) + "\n")
                elif t == "status" and c["execution_state"] == "idle":
                    break
            except KeyboardInterrupt:
                interrupted = True
                self.km.interrupt_kernel()
        sys.stdout.flush(), sys.stderr.flush()
        if interrupted:
            raise KeyboardInterrupt
        if error is not None:
            raise RuntimeError(f"公式環境カーネルでエラー: {error['ename']}: {error['evalue']}")


_kernel_env = dict(_TRAIN_ENV_VARS, AO_ENV_BOOTSTRAP=ENV_BOOTSTRAP_PATH)   # HF_TOKEN / WANDB_API_KEY（セル 0 で設定）もここで引き継ぐ
AO_KERNEL = TrainEnvKernel(TRAIN_PY, WORK_DIR, _kernel_env, cwd=WORK_DIR).start()
AO_KERNEL.run("import json, os\n"
              "ENV_BOOTSTRAP = json.load(open(os.environ['AO_ENV_BOOTSTRAP']))\n"
              "import sys, platform; print('公式環境カーネル:', sys.executable, platform.python_version())")

try:
    from IPython import get_ipython
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def ao(line, cell):
        """%%ao: このセルを公式環境のカーネルで実行する"""
        AO_KERNEL.run(cell)

    print("%%ao を登録した。設定セル以降は公式環境のカーネルで実行される")
except Exception as _e:   # IPython の外（スクリプト実行）では登録しない
    print("WARN: %%ao を登録できない:", _e)

## 設定

In [ ]:
%%ao
# @title 設定セル（実験の設定はここだけ。コードの revision と環境の pin はセル 1）
import os, sys, json, re, time, datetime, pathlib

# =============================================================================
# 実行モード
#   dry_run : データ・設定・コマンド・想定 step 数を確認するだけ。学習も HF 書き込みも行わない（既定）
#   trial   : 少数 step の学習で、メモリ・保存・HF 転送・プロセス終了→再開を確認する
#   train   : baseline または探索候補を所定の条件で学習する
# =============================================================================
RUN_MODE = "dry_run"          # "dry_run" | "trial" | "train"
RUN_KIND = "baseline"         # train 時に "baseline" | "search"
RUN_ID_SUFFIX = ""            # 同名 run と衝突する場合に付ける識別子（例 "-v2"）。空なら付けない

# =============================================================================
# 対象モデル（まず Qwen3 系列。サイズはここで選ぶ）
#   1.7B: 小規模な動作確認候補 / 4B: 本実験候補（Colab で実行可能かは試走で実測する）
#   GPU メモリの推定（公式 trainer のコードから算出。bf16 の重み・勾配・AdamW 2 状態 = 8 x パラメータ数 + logits 約 5.2 GiB など）:
#   1.7B 約 22 GiB / 4B 約 41 GiB / 8B 約 79 GiB。メモリ不足は optimizer 状態が作られる 2 step 目から出る
# =============================================================================
MODEL_KEY = "Qwen3-1.7B"

SUPPORTED_MODELS = {
    # revision は 2026-09-24 に HF API で確認した commit sha。ノートブック実行時にも再記録する
    "Qwen3-1.7B": dict(status="supported",
                       base_repo="Qwen/Qwen3-1.7B-Base", base_revision="ea980cb0a6c2ae4b936e82123acc929f1cec04c1",
                       cot_repo="jasonrqh/Qwen3-1.7B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="e4fcd58c7b104d3444e0e1d1b470f5107fcc2618",
                       cot_subfolder="step640", official_script="Qwen3-1.7B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="推定ピーク約 22 GiB（micro 4, max_length 1536）。A100 80GB / G4 96GB に収まる見込み"),
    "Qwen3-4B": dict(status="supported",
                     base_repo="Qwen/Qwen3-4B-Base", base_revision="906bfd4b4dc7f14ee4320094d8b41684abff8539",
                     cot_repo="jasonrqh/Qwen3-4B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="aaf1c51a6a2ee319750cafbd7cb8be0df10018b8",
                     cot_subfolder="step640", official_script="Qwen3-4B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="本実験候補。推定ピーク約 41 GiB。A100 80GB / G4 96GB に収まる見込み（試走で実測する）"),
    "Qwen3-8B": dict(status="supported",
                     base_repo="Qwen/Qwen3-8B-Base", base_revision="49e3418fbbbca6ecbdf9608b4d22e5a407081db4",
                     cot_repo="jasonrqh/Qwen3-8B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="522decd73c0b46f7029c2a315b3cc177c23a3498",
                     cot_subfolder="step640", official_script="Qwen3-8B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="推定ピーク約 79 GiB。A100 80GB ではほぼ確実にメモリ不足、G4 96GB なら収まる見込み（torch の版の変更を伴う）"),
    "Qwen3-14B": dict(status="supported",
                      base_repo="Qwen/Qwen3-14B-Base", base_revision="0b0bd3732e2c374d483664439ea334928b65f304",
                      cot_repo="jasonrqh/Qwen3-14B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="e8e6542249d647ac82c556a93410a9c0cafd41cb",
                      cot_subfolder="step640", official_script="Qwen3-14B_Math-CoT-20k_lr5e-5_ep8_bs256.sh", note="Colab 単一 GPU では full SFT が収まらない可能性が高い"),
    # Qwen2.5 系列: 公式スクリプトは Qwen/Qwen2.5-{size}-no-sys-prompt を指定する（HF API は 401 = 非公開または存在しない）。
    # 公開 Base の chat template は system message が無いと "You are a helpful assistant." を自動挿入するため tokenization が公式と一致しない。
    # 関係・準備方法を確認するまで黙って置き換えない → blocked。
    "Qwen2.5-1.5B": dict(status="blocked", base_repo="Qwen/Qwen2.5-1.5B", base_revision="8faed761d45a263340a0528343f099c05c9a4323",
                         cot_repo="jasonrqh/Qwen2.5-1.5B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="dd3c28b7846607e026f4ecc48eeb20e8d48035f1",
                         cot_subfolder=None, official_script="Qwen2.5-1.5B_Math-CoT-20k_lr5e-5_ep8_bs256.sh",
                         note="公式は Qwen/Qwen2.5-1.5B-no-sys-prompt（非公開）。公開 Base との差（chat template の既定 system prompt）が未解決"),
    "InternLM2.5-20B": dict(status="blocked", base_repo="internlm/internlm2_5-20b", base_revision="dc786ab20a7fd882ccef61df28e13a940edca24f",
                            cot_repo="jasonrqh/InternLM2.5-20B_Math-CoT-20k_lr5e-5_ep8_bs256", cot_revision="fa6f45a03525bb5a52046e3217f4ea5082aa4c75",
                            cot_subfolder=None, official_script="InternLM2.5-20B_Math-CoT-20k_lr5e-5_ep8_bs256.sh",
                            note="今回の対象外（trust_remote_code モデル。公式 micro_bsz=2）"),
}
TEACHER = dict(repo="Qwen/Qwen3-32B", revision="9216db5781bf21249d130ec9da846c4624c16137")  # 記録用。ダウンロードしない

# =============================================================================
# データの revision（コードの revision と環境の pin はセル 1 で設定する）
# =============================================================================
DATASET_REPO = "jasonrqh/Math-CoT-20k"
DATASET_REVISION = "1435fb21d4fecc8ad4966a26f22a874cf2b527f1"
DATASET_FILE = "Math-CoT-20k.parquet"
DATASET_EXPECTED_SHA256 = "3166f342ec838ebe0979be43923abb33fa1c80f33043aa6acc04e5d82161142a"
DATASET_EXPECTED_ROWS = 20480
DATASET_ORIGIN_POOL = "open-r1/OpenR1-Math-220k"   # 元問題プール（記録のみ。DeepSeek-R1 出力は抽出元にしない）

# =============================================================================
# AO データの抽出
#   teacher_answer 列 = response（教師 Qwen3-32B の CoT+回答）から抽出した最終回答
#   last_boxed_verbatim（既定）: 最後の \boxed{...} を教師の原文のまま保持（CoT・説明文は含めない）
#   boxed_content            : \boxed{} の中身のみ（教師原文の形を崩すので既定にしない）
#   last_boxed_line          : 最後の \boxed{} を含む原文の行（説明文が混ざり得るので既定にしない）
# =============================================================================
AO_TARGET_STYLE = "last_boxed_verbatim"
# math-verify で参照正解 (answer 列) と一致しなかった行のうち、内容を目視確認して「教師の最終回答として保持する」と決めた行番号。
# 2026-09-24 の監査では 3 行。いずれも同値な方程式の別表記（例 x^2/5 + 3y^2/5 = 1 と x^2+3y^2=5）で math-verify の関係式比較が失敗する。
# 実際の不一致集合がこのリストと異なれば本学習は開始しない（未解決の抽出問題として扱う）。
AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS = [12364, 16249, 16611]

# =============================================================================
# 学習設定
#   公式基準（公開 CoT スクリプト）: AdamW β=(0.9,0.999), lr 5e-5, 8 epoch, global batch 256, cosine + warmup 10%,
#   weight decay 0.01, grad clip 1.0, bf16, max_length 20000, truncation right, shuffle False, token-mean, grad ckpt, FSDP
#   → 実際の値は公式スクリプトと sft_trainer.yaml を実行時に読み取って表示する（ここに書いた値は目安）
# =============================================================================
OFFICIAL_BASELINE = {"lr": 5e-5, "epochs": 8, "status": "confirmed: 公開 CoT スクリプトの値"}

# 論文が実施した最適化条件（論文 v2。いずれも Qwen3-14B-Base × Math-CoT-20k。1.7B/4B/8B は default のみで、公開 CoT 学生も default のみ）
#   default: Sec. 2.1 / Tab. 3。短い学習・小さい LR: Sec. 3.1, App. C.1 Tab. 4。過学習の検証: Sec. 3.4 Setting 2〜4, App. C.7
#   warmup は全条件で総 step の 10%（App. B.2。constant でも warmup あり）
PAPER_OPTIMIZATION_CONDITIONS = [
    dict(lr=5e-5, epochs=8, scheduler="cosine", paper="default（Sec. 2.1, Tab. 3）"),
    dict(lr=5e-5, epochs=1, scheduler="cosine", paper="Tab. 4（短い学習）"),
    dict(lr=1e-5, epochs=1, scheduler="cosine", paper="Tab. 4（小さい LR）"),
    dict(lr=1e-5, epochs=2, scheduler="cosine", paper="Tab. 4（小さい LR）"),
    dict(lr=5e-5, epochs=16, scheduler="cosine", paper="Sec. 3.4 Setting 2"),
    dict(lr=5e-5, epochs=16, scheduler="constant", paper="Sec. 3.4 Setting 3"),
    dict(lr=1e-4, epochs=16, scheduler="constant", paper="Sec. 3.4 Setting 4"),
]
# 探索は保留: まず baseline だけを学習し、性能が出ないときに探索する。探索対象は optim.lr, trainer.total_epochs, optim.lr_scheduler。
SEARCH_GRID_PROPOSED = [(c["lr"], c["epochs"], c["scheduler"]) for c in PAPER_OPTIMIZATION_CONDITIONS[1:]]   # 探索するときの候補（論文の条件。保留中）
SEARCH_GRID_CONFIRMED = None      # 探索すると決めたら [(lr, epochs, scheduler), ...] を入れる
SEARCH_RUN_LIST = []              # このセッションで実行する候補 [(lr, epochs) or (lr, epochs, scheduler), ...]。scheduler 省略時は "cosine"

# 最大系列長: "auto_fit" = 全行が切り詰められずに収まる最小の長さ（MAX_LENGTH_ROUND_TO の倍数に切り上げ）
#             "official" = 公式の 20000 / 整数 = 明示指定
MAX_LENGTH_MODE = "auto_fit"
MAX_LENGTH_ROUND_TO = 64

# =============================================================================
# 公式 CoT 学生の学習条件（8 GPU）への合わせ込み
#   公式 CoT 学生は 8 GPU で学習された（README.md:101 "We trained all models on 8 H200 GPUs."）。
#   公式 trainer は micro batch ごとに loss を割らずに backward するので、1 GPU 内の勾配は micro batch の「和」になる
#   （verl/trainer/fsdp_sft_trainer_ours.py:467。:509 の /n_micro はログ用）。GPU 間は FSDP2 が「平均」する（ReduceOp.AVG）。
#   そのため GPU 台数 N が 8 と違うと、次の 2 点が公式と変わる:
#     (1) 1 micro batch（4 行）に入る行の組み合わせ。loss は micro batch ごとの token 平均なので、token の重みが変わる
#     (2) 勾配の大きさが 8/N 倍になり、clip_grad=1.0 の効き方と AdamW の eps の効き方が変わる
#   EMULATE_OFFICIAL_WORLD_SIZE=True のとき、公式コードを変えずに次の 2 点で公式 8 GPU の計算を再現する:
#     - 学習用 parquet の行を並べ替え、各 step の各 micro batch を公式 8 GPU と同じ 4 行にする（元の AO parquet は変更しない）
#     - trainer.importance_sampling_mode=adv-only と advantage 列 = N/8 で loss を N/8 倍する。
#       vanilla の loss を 2 のべき乗倍するだけなので、勾配・clip・AdamW の入力は公式 8 GPU と同じ値になる（丸め誤差の範囲。
#       CPU 上の公式 trainer で fp32 なら移動量比 1e-6 で一致、bf16 では加算順序の違いで数 % の丸め差）。ログの train/loss は N/8 倍になる
#   N は 8 の約数（1, 2, 4, 8）であること。N=8 のときは何もしない（公式と同じ）。
# =============================================================================
OFFICIAL_WORLD_SIZE = 8
EMULATE_OFFICIAL_WORLD_SIZE = True

MICRO_BATCH_OVERRIDE = None       # 公式の micro batch 4 を変えると (1) を再現できない。EMULATE_OFFICIAL_WORLD_SIZE=True では None 以外を禁止。
                                  # メモリ不足のときは、条件を保ったまま GPU を大きくするか 2/4 台にする。micro batch を変える場合は
                                  # EMULATE_OFFICIAL_WORLD_SIZE=False にし、公式との差として記録される（黙って変えない）
CPU_OFFLOAD_OVERRIDE = None       # 使用不可: 公式 trainer は fsdp2 経路でも FSDP1 用の CPUOffload を渡すため FSDP2 に無視され offload されない（trainer:300-303, 337）
REQUIRE_FSDP2_GRAD_CHECK = True  # セクション 2 の「1 GPU の FSDP2 勾配確認」が OK でなければ学習を開始しない（world size 1 の NCCL AVG の既知の不具合報告への対策）
KEEP_OFFICIAL_ENV_VARS = True     # 公式スクリプトの export（CUDA_LAUNCH_BLOCKING=1 など）をそのまま使う。False なら CUDA_LAUNCH_BLOCKING を外し記録する

SAVE_FREQ = 10                    # trainer.save_freq（正の整数。0 は使わない）。公式 CoT と同じ 10。保存は学習結果に影響しない
# HF へ転送する step: "cot_public+resume" = 論文が評価した step（PAPER_EVAL_STEPS）と RESUME_CKPT_EVERY の倍数と最終 step。
#   それ以外の step の checkpoint は保存完了後にローカルから消す（HF に無いので再開候補にもならない）。"all" = 保存した全 step を転送
# checkpoint 1 個（実測）: 1.7B = 10.34 GB（うち optimizer 状態 6.88 GB）, 4B = 24.15 GB（同 16.09 GB）。HF PRO の非公開枠は 1 TB
#   HF_ANALYSIS_STEP_CONTENT="model_only": 再開用（RESUME_CKPT_EVERY の倍数と最終 step）以外の step は optimizer 状態を送らない
#   （重み・extra・data.pt・huggingface/ は送る。分析・変換はできるが、その step からは再開できない）
#   1.7B baseline 640 step の HF 使用量: RESUME_CKPT_EVERY=80 + model_only で約 93 GB（=40 + full で約 186 GB）
HF_UPLOAD_STEPS = "cot_public+resume"
# 論文が評価・報告した step（App. D。16 epoch run は 1280 まで）。1.7B/4B の公開 CoT 学生の HF repo にあるのは 640 まで（2026-09-24 に確認）
PAPER_EVAL_STEPS = [10, 20, 40, 80, 160, 320, 480, 640, 800, 960, 1120, 1280]
RESUME_CKPT_EVERY = 80            # 再開用（optimizer 状態込み）に HF へ転送する間隔（step）。SAVE_FREQ の倍数。切断時は最大この step 数を再実行する
HF_ANALYSIS_STEP_CONTENT = "model_only"   # "model_only" | "full"（論文の評価 step も optimizer 状態込みで送る）
SEED = None                       # None = 公式既定 (trainer.seed=1 は yaml 既定。trainer 内で明示的な乱数初期化はされていない)

# =============================================================================
# Hugging Face 保存
# =============================================================================
HF_CKPT_REPO_ID = None            # None → f"{HF_ACCOUNT_NAME}/rethink-sft-ao-checkpoints"（HF の whoami から決める）
HF_CKPT_PRIVATE = True
HF_UPLOAD_CHECKPOINTS = True      # 学習中に保存完了した checkpoint を監視スレッドが HF へ転送する
CKPT_MANIFEST_SHA256 = True       # manifest に sha256 を記録し、転送後に HF 側の LFS sha256 と照合する
LOCAL_KEEP_LAST_N_VERIFIED_CKPTS = 2   # HF 転送・検証済みのローカル checkpoint をこの数だけ残し、古いものを削除（未検証は削除しない）
HF_FINAL_MODEL_REPO_ID = None     # None → f"{HF_ACCOUNT_NAME}/{run_id}"（推論用 HF モデル）
HF_FINAL_MODEL_PRIVATE = True

WANDB_MODE = "offline"            # 公式スクリプトと同じ offline。"online" は WANDB_API_KEY がある場合のみ
WANDB_PROJECT = "sft_generalization_ao"

# =============================================================================
# 試走（RUN_MODE="trial"）
#   AO データの先頭 TRIAL_NUM_ROWS 行だけを使い、TRIAL_EPOCHS epoch 学習する。
#   steps_per_epoch = TRIAL_NUM_ROWS // 256。TRIAL_KILL_AFTER_STEP の checkpoint が HF へ転送・検証された時点でプロセスを終了し、
#   新しいプロセスで HF から取得して再開する。試走の run_id は "trial-" で始まり、本学習と混ざらない。
# =============================================================================
TRIAL_NUM_ROWS = 512
TRIAL_EPOCHS = 4                  # 2 steps/epoch x 4 = 8 step。kill step の後に十分な step が残るようにする
TRIAL_SAVE_FREQ = 3
TRIAL_KILL_AFTER_STEP = 3         # この step の checkpoint 保存完了を検知したら学習プロセスを一時停止し、HF 転送・検証後に終了する
TRIAL_RUN_REFERENCE = True        # 中断なしの参照 run も実行し、再開後の lr / loss / データ位置を比較する

# =============================================================================
# 切断後の再開（「再開」セルで使う）
# =============================================================================
RESUME_RUN_ID = None              # 例 "Qwen3-1.7B_Math-AO-20k_lr5e-5_ep8_bs256_baseline"
RESUME_STEP = "latest"            # "latest"（最新の完了済み）または整数 step

# =============================================================================
# 評価（セクション 9）: 論文と同じ評価（論文 v2 Sec. 2.2, App. B.4/B.5）
#   公式スクリプト evaluation/math_eval/math_eval_budget.py を無変更で実行する（vLLM 0.8.5, temperature 0.6, top_p 0.95, 最大 32768 token,
#   MATH500 は 3 サンプルの平均正答率 avg@3、AIME24 は avg@10、math-verify で採点）。論文の数学評価はこの 2 つだけ。
#   学習用の公式環境とは別の Python 3.12 環境（vLLM 0.8.5 と公式 pin）を作って実行する。GPU は A100 など（vLLM 0.8.5 は G4 に非対応）
#   論文との差: 論文は 2 GPU（tensor parallel 2）、ここは 1 GPU。数値の丸め程度の差（記録する）
# =============================================================================
PAPER_EVAL_ENABLED = False        # True で実行（学習が終わってから。GPU メモリの 85% を使う）
PAPER_EVAL_DATASETS = ["MATH500", "AIME24"]
PAPER_EVAL_AO_RUNS = None         # 評価する AO run の run_id のリスト。None → このセッションで学習した run
PAPER_EVAL_AO_STEPS = "paper"     # "paper" → PAPER_EVAL_STEPS のうち HF にある step（論文と同じ報告方法: 各 step の推移）/ 整数のリスト
PAPER_EVAL_BASE = False           # Base（step 0）も評価する。既定 False: Base は論文の値（PAPER_REFERENCE）を使うと決定済み
PAPER_EVAL_COT_STEPS = []         # 公開 CoT 学生のうち再評価する step（例 [640]）。既定 []: CoT は論文の値（PAPER_REFERENCE）を使うと決定済み（評価環境の差は許容）
PAPER_EVAL_DELETE_MERGED = True   # 評価後に変換済みの AO 重みをローカルから消す（ディスク節約。HF の checkpoint は残る）

# 論文の値（App. D Table 18〜32、数学のみ。1.7B / 4B / 8B / 14B の Base・CoT・NoCoT。MATH500 avg@3 / AIME24 avg@10、%）。比較の表示に使う。NoCoT は step 20 の報告なし
PAPER_REFERENCE = {
    "Qwen3-1.7B": {
        "base": {"MATH500": 58.9, "AIME24": 5.7},
        "Math-CoT": {10: (55.7, 4.3), 20: (53.5, 3.3), 40: (42.9, 1.7), 80: (49.6, 2.7), 160: (51.6, 4.0), 320: (56.5, 5.0), 480: (59.1, 3.7), 640: (56.2, 5.0)},
        "Math-NoCoT": {10: (55.2, 3.0), 40: (58.9, 5.7), 80: (58.1, 4.7), 160: (58.0, 4.3), 320: (60.7, 4.0), 480: (61.3, 5.7), 640: (59.5, 5.7)},
        "source": "Table 18, 19, 21",
    },
    "Qwen3-4B": {
        "base": {"MATH500": 70.8, "AIME24": 11.0},
        "Math-CoT": {10: (64.9, 9.0), 20: (62.9, 13.0), 40: (71.5, 15.3), 80: (77.1, 19.0), 160: (78.8, 22.3), 320: (86.3, 29.0), 480: (87.1, 27.3), 640: (86.2, 30.7)},
        "Math-NoCoT": {10: (66.7, 7.7), 40: (73.2, 10.0), 80: (72.9, 9.7), 160: (74.3, 9.0), 320: (75.9, 11.3), 480: (74.5, 11.3), 640: (74.3, 11.7)},
        "source": "Table 18, 22, 24",
    },
    "Qwen3-8B": {
        "base": {"MATH500": 76.8, "AIME24": 12.0},
        "Math-CoT": {10: (72.7, 11.0), 20: (44.4, 8.0), 40: (78.2, 21.7), 80: (81.9, 23.0), 160: (88.5, 32.0), 320: (89.8, 36.0), 480: (92.3, 43.7), 640: (90.1, 47.7)},
        "Math-NoCoT": {10: (71.3, 9.3), 40: (76.1, 11.7), 80: (75.8, 9.7), 160: (78.0, 12.7), 320: (77.9, 13.7), 480: (76.3, 15.7), 640: (77.4, 13.3)},
        "source": "Table 18, 25, 27",
    },
    "Qwen3-14B": {
        "base": {"MATH500": 77.8, "AIME24": 14.7},
        "Math-CoT": {10: (79.2, 11.3), 20: (68.2, 25.7), 40: (88.3, 38.0), 80: (91.3, 50.7), 160: (93.7, 55.3), 320: (95.1, 60.3), 480: (94.9, 68.0), 640: (95.1, 66.0)},
        "Math-NoCoT": {10: (75.0, 15.3), 40: (80.5, 15.0), 80: (80.7, 14.3), 160: (81.1, 16.3), 320: (81.5, 18.0), 480: (81.5, 17.0), 640: (82.4, 17.0)},
        "source": "Table 18, 30, 32",
    },
}

# =============================================================================
# 最終モデル（セクション 10）
# =============================================================================
FINAL_RUN_ID = None               # None → このセッションで学習した run
FINAL_STEP = "last"               # "last" または整数 step

# =============================================================================
# 作業ディレクトリ（セル 1 で決めた値を使う。ここでは変更しない）
# =============================================================================
IN_COLAB = ENV_BOOTSTRAP["in_colab"]
WORK_DIR = ENV_BOOTSTRAP["work_dir"]
REPO_DIR = ENV_BOOTSTRAP["repo_dir"]
DATA_DIR = f"{WORK_DIR}/data"
MODELS_DIR = f"{WORK_DIR}/models"
CKPT_DIR = f"{WORK_DIR}/ckpt"
LOG_DIR = f"{WORK_DIR}/log"
RECORD_DIR = f"{WORK_DIR}/records"
for _d in (WORK_DIR, DATA_DIR, MODELS_DIR, CKPT_DIR, LOG_DIR, RECORD_DIR):
    os.makedirs(_d, exist_ok=True)

# ---- 設定の妥当性 ----
assert RUN_MODE in ("dry_run", "trial", "train"), RUN_MODE
assert RUN_KIND in ("baseline", "search"), RUN_KIND
assert isinstance(SAVE_FREQ, int) and SAVE_FREQ > 0, "trainer.save_freq は正の整数。0 は使わない"
assert HF_UPLOAD_STEPS in ("all", "cot_public+resume"), HF_UPLOAD_STEPS
assert HF_ANALYSIS_STEP_CONTENT in ("model_only", "full"), HF_ANALYSIS_STEP_CONTENT
assert RESUME_CKPT_EVERY % SAVE_FREQ == 0 and all(s % SAVE_FREQ == 0 for s in PAPER_EVAL_STEPS) or HF_UPLOAD_STEPS == "all", \
    "RESUME_CKPT_EVERY と PAPER_EVAL_STEPS は SAVE_FREQ の倍数にする（保存されない step は転送できない）"
assert isinstance(TRIAL_SAVE_FREQ, int) and TRIAL_SAVE_FREQ > 0
assert MAX_LENGTH_MODE in ("auto_fit", "official") or isinstance(MAX_LENGTH_MODE, int)
assert AO_TARGET_STYLE in ("last_boxed_verbatim", "boxed_content", "last_boxed_line")
assert CPU_OFFLOAD_OVERRIDE is None, "CPU_OFFLOAD_OVERRIDE は fsdp2 経路で効かないので使わない（公式値 False のまま）"
assert not (EMULATE_OFFICIAL_WORLD_SIZE and MICRO_BATCH_OVERRIDE is not None), "公式 8 GPU の再現には micro batch 4 が必要。MICRO_BATCH_OVERRIDE は None にする"
assert MODEL_KEY in SUPPORTED_MODELS, f"未対応の MODEL_KEY: {MODEL_KEY}"
MODEL_INFO = SUPPORTED_MODELS[MODEL_KEY]

SESSION_STARTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print("RUN_MODE:", RUN_MODE, "| RUN_KIND:", RUN_KIND, "| MODEL_KEY:", MODEL_KEY, "| status:", MODEL_INFO["status"])
print("WORK_DIR:", WORK_DIR)

## 2. 環境の記録・HF ログイン・1 GPU の FSDP2 勾配確認・研究条件

In [ ]:
%%ao
# @title 2. 環境の記録・HF ログイン・1 GPU の FSDP2 勾配確認・研究条件
import json, os, sys, platform, shutil, subprocess, time
import huggingface_hub

# ---- 2.1 セル 1 の結果（公式環境・Fork の revision・GPU）を記録する ------------------------------------------
FORK_REPO_URL, FORK_REF, FORK_COMMIT = ENV_BOOTSTRAP["fork_repo"], ENV_BOOTSTRAP["fork_ref"], ENV_BOOTSTRAP["fork_commit"]
UPSTREAM_REPO_URL, UPSTREAM_REFERENCE_COMMIT = ENV_BOOTSTRAP["upstream_repo"], ENV_BOOTSTRAP["upstream_reference_commit"]
UPSTREAM_FETCHED = ENV_BOOTSTRAP["upstream_fetched"]
OFFICIAL_CODE_UNCHANGED, OFFICIAL_DIFF_STAT = ENV_BOOTSTRAP["official_code_unchanged"], ENV_BOOTSTRAP["official_diff_stat"]
GPUS, TORCH_PROBE, PKG_VERSIONS = ENV_BOOTSTRAP["gpus"], ENV_BOOTSTRAP["torch_probe"], dict(ENV_BOOTSTRAP["packages"])
N_GPUS = len(GPUS)
GPU_PROFILE, GPU_PROFILE_DEVIATION = ENV_BOOTSTRAP["gpu_profile"], ENV_BOOTSTRAP["gpu_profile_deviation"]
FLASH_ATTN_VERSION = ENV_BOOTSTRAP["flash_attn_version"]
FLASH_ATTN_OK = bool(TORCH_PROBE.get("cuda_available")) and PKG_VERSIONS.get("flash_attn") == FLASH_ATTN_VERSION
assert platform.python_version() == TORCH_PROBE["python"] and sys.prefix == ENV_BOOTSTRAP["train_env_dir"], \
    f"このセルは公式環境のカーネルで実行する（%%ao）。python={platform.python_version()} prefix={sys.prefix}"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# ipykernel はこのカーネルの MPLBACKEND を inline 用の値にする。subprocess（学習・評価）の環境には matplotlib_inline が無いので、描画しない Agg にする
os.environ["MPLBACKEND"] = "Agg"

_mem = {}
for line in open("/proc/meminfo"):
    _mem[line.split(":")[0]] = int(line.split(":")[1].strip().split()[0])
_disk = shutil.disk_usage(WORK_DIR)
print(f"GPU x{N_GPUS}: {GPUS} | profile={GPU_PROFILE}")
print(f"RAM total {_mem.get('MemTotal', 0) / 1e6:.1f} GB | disk free {_disk.free / 1e9:.1f} GB | kernel python {ENV_BOOTSTRAP['kernel_python']} | 公式環境 python {TORCH_PROBE['python']}")
print(f"torch {TORCH_PROBE['torch']} (CUDA {TORCH_PROBE['cuda']}, cuDNN {TORCH_PROBE['cudnn']}, NCCL {TORCH_PROBE['nccl']}) | flash-attn {PKG_VERSIONS.get('flash_attn')} | transformers {PKG_VERSIONS.get('transformers')}")
if GPU_PROFILE_DEVIATION:
    print(f"*** 公式環境からの変更（GPU {GPUS[0]['name']} のため）: {GPU_PROFILE_DEVIATION} ***")
print("公式コード (verl/, training_scripts/) の参照 commit からの差分:", "なし" if OFFICIAL_CODE_UNCHANGED else OFFICIAL_DIFF_STAT)

# ---- 2.2 HF ログイン（公式環境の huggingface_hub で行う。アカウント名は whoami から取る） -------------------------
HF_LOGGED_IN, HF_ACCOUNT_NAME = False, None   # GitHub のユーザー名からは推測しない
if os.environ.get("HF_TOKEN"):
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    HF_ACCOUNT_NAME = huggingface_hub.whoami().get("name")
    HF_LOGGED_IN = True
    print(f"HF: logged in as '{HF_ACCOUNT_NAME}' (token masked) | huggingface_hub {huggingface_hub.__version__}")
else:
    print("HF: HF_TOKEN が無い（セル 0）。HF へのアップロード・再開は無効")

# ---- 2.3 1 GPU の FSDP2 で勾配が正しく計算されるかの確認（GPU があるときだけ） ----------------------------------
# 1 GPU では FSDP2 の reduce-scatter が world size 1 の NCCL ReduceOp.AVG になる。torch には「world size 1 の AVG で一部の勾配が 0 になる」
# 未解決の報告がある（pytorch/pytorch#165399, #144045）。公式の apply_fsdp2 で包んだ小型 Qwen3 の勾配を、FSDP 無しの同じモデルの勾配と比べる。
_GRAD_CHECK_SRC = r'''
import json, os, copy, torch, torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from transformers import Qwen3Config, AutoModelForCausalLM
from verl.utils.fsdp_utils import apply_fsdp2, fsdp2_load_full_state_dict, MixedPrecisionPolicy

cuda = torch.cuda.is_available()
if not cuda and os.environ.get("AO_GRADCHECK_CPU_PATCH") == "1":   # CPU での検証用（torch 2.6 の CPU Stream に FSDP2 が使うメソッドが無い）
    torch.cpu.Stream.record_event = lambda self, event=None: torch.cpu.Event()
    torch.cpu.Stream.wait_event = lambda self, event: None
dist.init_process_group("nccl" if cuda else "gloo")
dev = torch.device("cuda", int(os.environ.get("LOCAL_RANK", 0))) if cuda else torch.device("cpu")
if cuda:
    torch.cuda.set_device(dev)
# Qwen3-1.7B と同じ語彙・tied embedding の小型モデル（重みは乱数。ダウンロードしない）
cfg = Qwen3Config(vocab_size=151936, hidden_size=256, intermediate_size=512, num_hidden_layers=2, num_attention_heads=4,
                  num_key_value_heads=2, head_dim=64, tie_word_embeddings=True, max_position_embeddings=1024)
torch.manual_seed(0)
model = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.bfloat16, attn_implementation="sdpa")
ref = copy.deepcopy(model).to(dev)
for m in (model, ref):   # 公式 trainer と同じ（enable_gradient_checkpointing=True）
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
mesh = init_device_mesh(dev.type, mesh_shape=(dist.get_world_size(),), mesh_dim_names=("fsdp",))
mp = MixedPrecisionPolicy(param_dtype=torch.bfloat16, reduce_dtype=torch.float32, cast_forward_inputs=True)   # 公式 trainer と同じ
full_state = model.state_dict()
apply_fsdp2(model, dict(mesh=mesh, mp_policy=mp, offload_policy=None, reshard_after_forward=True), {})
fsdp2_load_full_state_dict(model, full_state, mesh, None)
g = torch.Generator().manual_seed(1)
for _ in range(2):   # 公式と同じく micro batch ごとに backward して勾配を累積する
    ids = torch.randint(0, cfg.vocab_size, (4, 128), generator=g).to(dev)
    for m in (model, ref):
        with torch.autocast(device_type=dev.type, dtype=torch.bfloat16):
            logits = m(input_ids=ids).logits
        loss = torch.nn.functional.cross_entropy(logits[:, :-1].float().reshape(-1, cfg.vocab_size), ids[:, 1:].reshape(-1))
        loss.backward()
ref_params = dict(ref.named_parameters())
rows = []
for name, p in model.named_parameters():
    gf = p.grad.full_tensor().float().cpu()
    gr = ref_params[name].grad.float().cpu()
    rel = float((gf - gr).norm() / gr.norm().clamp_min(1e-30))
    zeroed = int(((gr != 0) & (gf == 0)).sum())
    rows.append(dict(name=name, rel=rel, zeroed=zeroed, numel=gr.numel(), tail_ref=gr.flatten()[-2:].tolist(), tail_fsdp=gf.flatten()[-2:].tolist()))
ok = all(r["rel"] < 1e-2 and r["zeroed"] == 0 for r in rows)
if dist.get_rank() == 0:
    print("GRADCHECK " + json.dumps(dict(ok=ok, world_size=dist.get_world_size(), torch=torch.__version__,
                                         nccl=list(torch.cuda.nccl.version()) if cuda else None, params=rows)))
dist.destroy_process_group()
'''


def run_fsdp2_grad_check(nproc=1, extra_env=None, timeout=900):
    path = f"{WORK_DIR}/tools/fsdp2_grad_check.py"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(_GRAD_CHECK_SRC)
    env = dict(os.environ, PYTHONPATH=REPO_DIR, **(extra_env or {}))
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", path]
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=REPO_DIR, timeout=timeout)
    lines = [l for l in r.stdout.splitlines() if l.startswith("GRADCHECK ")]
    if r.returncode != 0 or not lines:
        return dict(status="error", returncode=r.returncode, stderr=r.stderr[-3000:])
    res = json.loads(lines[-1][len("GRADCHECK "):])
    res["status"] = "ok" if res["ok"] else "mismatch"
    return res


FSDP2_GRAD_CHECK = dict(status="skipped (GPU なし)")
if N_GPUS >= 1:
    # 公式スクリプトの export（セクション 5 で表示）と同じ環境変数で実行する（同期の違いで結果が変わる不具合を見逃さないため）
    FSDP2_GRAD_CHECK = run_fsdp2_grad_check(nproc=1, extra_env=dict(CUDA_LAUNCH_BLOCKING="1", TORCH_NCCL_AVOID_RECORD_STREAMS="1", NCCL_DEBUG="WARN"))
    if FSDP2_GRAD_CHECK["status"] == "ok":
        _worst = max(FSDP2_GRAD_CHECK["params"], key=lambda r: r["rel"])
        print(f"FSDP2 勾配確認 (world size 1): OK。全 {len(FSDP2_GRAD_CHECK['params'])} パラメータで 0 化なし、最大相対差 {_worst['rel']:.2e} ({_worst['name']})")
    elif FSDP2_GRAD_CHECK["status"] == "mismatch":
        for _r in FSDP2_GRAD_CHECK["params"]:
            if _r["rel"] >= 1e-2 or _r["zeroed"]:
                print("  MISMATCH", _r)
        print("*** FSDP2 (world size 1) の勾配が FSDP 無しの勾配と一致しない。この環境では学習を開始しない（REQUIRE_FSDP2_GRAD_CHECK） ***")
    else:
        print("*** FSDP2 勾配確認のスクリプトが失敗した ***\n", FSDP2_GRAD_CHECK.get("stderr", ""))
else:
    print("FSDP2 勾配確認: GPU が無いのでスキップ")

ENV_RECORD = dict(ENV_BOOTSTRAP, session_started_at=SESSION_STARTED_AT, n_gpus=N_GPUS, ram_total_gb=round(_mem.get("MemTotal", 0) / 1e6, 1),
                  disk_free_gb=round(_disk.free / 1e9, 1), flash_attn_ok=FLASH_ATTN_OK, hf_account=HF_ACCOUNT_NAME,
                  fsdp2_grad_check={k: v for k, v in FSDP2_GRAD_CHECK.items() if k != "params"})
with open(f"{RECORD_DIR}/env_record.json", "w") as f:
    json.dump(ENV_RECORD, f, indent=2, ensure_ascii=False)
print("saved", f"{RECORD_DIR}/env_record.json")

# ---- 2.4 研究条件・対応モデル -------------------------------------------------------------------
print("\n=== 対応モデル ===")
for _k, _v in SUPPORTED_MODELS.items():
    print(f"  {_k:16s} {_v['status']:9s} base={_v['base_repo']}@{_v['base_revision'][:8]}  cot={_v['cot_repo']}"
          f"{'/' + _v['cot_subfolder'] if _v['cot_subfolder'] else ''}@{_v['cot_revision'][:8]}")
    print(f"  {'':16s} note: {_v['note']}")
if MODEL_INFO["status"] != "supported":
    raise SystemExit(f"MODEL_KEY={MODEL_KEY} は blocked: {MODEL_INFO['note']}。設定を確認するまで進めない")

print("\n=== 比較対象 ===")
print(f"  Teacher            : {TEACHER['repo']} @ {TEACHER['revision'][:8]}（記録のみ）")
print(f"  Base Student       : {MODEL_INFO['base_repo']} @ {MODEL_INFO['base_revision'][:8]}")
print(f"  CoT Distill Student: {MODEL_INFO['cot_repo']}/{MODEL_INFO['cot_subfolder']} @ {MODEL_INFO['cot_revision'][:8]}（公開モデル。学習しない）")
print(f"  AO Distill Student : このノートブックで学習（Base から Full-parameter SFT）")

print("\n=== 公式基準設定（公開 CoT スクリプトを実行時に解析して表示する。ここは目安） ===")
print(f"  lr={OFFICIAL_BASELINE['lr']}, epochs={OFFICIAL_BASELINE['epochs']}  [{OFFICIAL_BASELINE['status']}]")
print("  探索: 保留（baseline の性能が出ないときに行う）。候補:", SEARCH_GRID_PROPOSED)
print("  探索候補（確定）      :", SEARCH_GRID_CONFIRMED)
print("  このセッションで実行する候補:", SEARCH_RUN_LIST)

print("\n=== AO target とプロンプトの関係（実験記録に明記する） ===")
print("  入力 prompt (message 列) は公式と同一。末尾の『Please reason step by step, and put your final answer within \\boxed{}.』も変更しない。")
print("  target (teacher_answer 列) は教師 response の最後の \\boxed{...} を原文のまま用いる。CoT・説明文は含めない。")
print("  したがって AO 学生は『step by step』と指示されても推論を書かず \\boxed{答え} + EOS を出力するよう学習される。")
print("  公式の Math-NoCoT-20k（<think> を除き最終まとめ＋回答を残す）とは異なる。参照正解 answer 列は照合にだけ使い、target に代用しない。")

print(f"\n=== 公式 CoT 学生の学習条件（{OFFICIAL_WORLD_SIZE} GPU）への合わせ込み: {'ON' if EMULATE_OFFICIAL_WORLD_SIZE else 'OFF'} ===")
print(f"  公式 CoT 学生は {OFFICIAL_WORLD_SIZE} GPU で学習された（README.md:101）。公式 trainer は 1 GPU 内の micro batch の勾配を和で蓄積し、GPU 間は平均する。")
print(f"  そのまま GPU 台数 N で動かすと、micro batch（4 行）の組み合わせと勾配の大きさ（{OFFICIAL_WORLD_SIZE}/N 倍）が公式と変わる。")
if EMULATE_OFFICIAL_WORLD_SIZE:
    print("  ON のとき: 学習用 parquet の行を並べ替えて各 micro batch を公式と同じ 4 行にし、adv-only + advantage=N/8 で loss を N/8 倍して勾配を公式と同じ大きさにする。")
    print("  学習手順の違いは target 列（teacher_answer）だけになる。残る差は丸め誤差程度（GPU の種類、bf16 の加算順序など。ドライランで一覧表示）。")

## 3. データ取得・AO 抽出・全行監査

`teacher_answer` 列は `response` の `</think>` 以降の最後の `\boxed{...}` を原文のまま用いる。抽出失敗行は `null` のまま残し（削除も `answer` 代用もしない）、ゲート `AO_DATA_READY` で本学習を止める。

In [ ]:
%%ao
# @title 3. データ取得・AO 抽出・全行監査（元 20,480 行・行順・message・advantage を維持し teacher_answer 列を追加）
import hashlib, json, os, re, collections, time
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import pandas as pd
import multiprocessing
from huggingface_hub import hf_hub_download

# 既存の boxed 抽出・照合処理を流用する（verl/utils/reward_score/math_verify_ours.py: last_boxed_only_string + math-verify）
from verl.utils.reward_score.math_verify_ours import last_boxed_only_string, compute_score as mv_compute_score


def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- 3.1 取得（revision 固定、sha256 照合） ----
RAW_PARQUET = hf_hub_download(repo_id=DATASET_REPO, filename=DATASET_FILE, repo_type="dataset", revision=DATASET_REVISION,
                              local_dir=f"{DATA_DIR}/raw")
RAW_SHA256 = sha256_of(RAW_PARQUET)
print("raw parquet:", RAW_PARQUET, "\nsha256:", RAW_SHA256)
assert RAW_SHA256 == DATASET_EXPECTED_SHA256, "Math-CoT-20k.parquet の sha256 が想定と異なる。revision を確認する"

RAW_TABLE = pq.read_table(RAW_PARQUET)
assert RAW_TABLE.num_rows == DATASET_EXPECTED_ROWS, RAW_TABLE.num_rows
for _c in ("message", "response", "answer", "advantage"):
    assert _c in RAW_TABLE.column_names, _c
print("columns:", RAW_TABLE.column_names, "| rows:", RAW_TABLE.num_rows)
_raw_df = RAW_TABLE.select(["response", "answer"]).to_pandas()


# ---- 3.2 抽出（監査用に全 box を列挙し、target は既存 helper の last_boxed_only_string で取る） ----
def find_all_boxed(text):
    """監査用: \\boxed の出現を先頭から列挙する。(位置, 文字列 | 'NOBRACE' | None=閉じ括弧なし)"""
    out, start = [], 0
    while True:
        idx = text.find("\\boxed", start)
        if idx < 0:
            break
        k = idx + len("\\boxed")
        while k < len(text) and text[k] == " ":
            k += 1
        if k < len(text) and text[k] == "{":
            depth, i, right = 0, k, None
            while i < len(text):
                if text[i] == "{":
                    depth += 1
                elif text[i] == "}":
                    depth -= 1
                    if depth == 0:
                        right = i
                        break
                i += 1
            if right is None:
                out.append((idx, None)); start = k + 1
            else:
                out.append((idx, text[idx:right + 1])); start = right + 1
        else:
            out.append((idx, "NOBRACE")); start = k
    return out


def extract_ao_target(response, style):
    """response（教師 CoT+回答）から AO target を抽出する。戻り値: (target or None, status, info)"""
    info = dict(n_think_open=response.count("<think>"), n_think_close=response.count("</think>"))
    after = response.split("</think>")[-1] if info["n_think_close"] >= 1 else response
    boxes_after = find_all_boxed(after)
    info["n_boxes_after_think"] = len(boxes_after)
    info["n_boxes_total"] = len(find_all_boxed(response))
    info["nobrace_form_after_think"] = any(b[1] == "NOBRACE" for b in boxes_after)
    info["unclosed_box_after_think"] = any(b[1] is None for b in boxes_after)
    distinct = [b[1] for b in boxes_after if isinstance(b[1], str) and b[1] != "NOBRACE"]
    info["n_distinct_boxes_after_think"] = len(set(distinct))
    info["first_box_after_think"] = distinct[0] if distinct else None
    last_box = last_boxed_only_string(after)             # 既存 helper（入れ子括弧対応）
    info["last_box_equals_response_last_box"] = (last_box == last_boxed_only_string(response))
    if last_box is None:
        return None, "no_box", info
    inner = last_box[len("\\boxed{"):-1]
    if inner.strip() == "":
        return None, "empty_box", info
    if style == "last_boxed_verbatim":
        target = last_box
    elif style == "boxed_content":
        target = inner
    elif style == "last_boxed_line":
        pos = after.rfind(last_box)
        ls = after.rfind("\n", 0, pos) + 1
        le = after.find("\n", pos + len(last_box))
        target = after[ls: (le if le >= 0 else len(after))].strip()
    else:
        raise ValueError(style)
    return target, "ok", info


def _verify_row(args):
    target, gold = args
    if target is None:
        return "not_extracted"
    try:
        s = mv_compute_score(solution_str=target, ground_truth=gold)   # 既存の照合処理（gold を $..$ で包み math-verify）
    except Exception as e:  # noqa
        return "verify_error:" + type(e).__name__
    return "match" if float(s) == 1.0 else "mismatch"


t0 = time.time()
_targets, _status, _infos = [], [], []
for resp in _raw_df["response"].tolist():
    t, s, i = extract_ao_target(resp, AO_TARGET_STYLE)
    _targets.append(t); _status.append(s); _infos.append(i)
print(f"extraction done in {time.time() - t0:.1f}s")

t0 = time.time()
# セル内で定義した関数を worker で使うため fork を明示（Colab/Linux）。spawn 環境では動かない
with multiprocessing.get_context("fork").Pool(max(1, min(8, os.cpu_count() or 1))) as _pool:
    _verify = _pool.map(_verify_row, list(zip(_targets, _raw_df["answer"].tolist())), chunksize=64)
print(f"math-verify done in {time.time() - t0:.1f}s")

AUDIT = pd.DataFrame(_infos)
AUDIT.insert(0, "row", range(len(AUDIT)))
AUDIT["extraction_status"] = _status
AUDIT["teacher_answer"] = _targets
AUDIT["answer"] = _raw_df["answer"].values
AUDIT["verify_status"] = _verify
AUDIT["teacher_answer_chars"] = AUDIT["teacher_answer"].map(lambda x: len(x) if x else 0)

# ---- 3.3 監査集計とゲート ----
_summary = dict(
    rows=int(len(AUDIT)), target_style=AO_TARGET_STYLE, dataset_repo=DATASET_REPO, dataset_revision=DATASET_REVISION, raw_sha256=RAW_SHA256,
    extraction_status=AUDIT["extraction_status"].value_counts().to_dict(),
    verify_status=AUDIT["verify_status"].value_counts().to_dict(),
    think_tag_counts=dict(open=AUDIT["n_think_open"].value_counts().to_dict(), close=AUDIT["n_think_close"].value_counts().to_dict()),
    n_boxes_after_think=AUDIT["n_boxes_after_think"].value_counts().sort_index().to_dict(),
    n_distinct_boxes_after_think=AUDIT["n_distinct_boxes_after_think"].value_counts().sort_index().to_dict(),
    rows_multi_distinct_boxes=int((AUDIT["n_distinct_boxes_after_think"] > 1).sum()),
    rows_nobrace_form_after_think=int(AUDIT["nobrace_form_after_think"].sum()),
    rows_unclosed_box_after_think=int(AUDIT["unclosed_box_after_think"].sum()),
    rows_last_box_differs_from_response_last_box=int((~AUDIT["last_box_equals_response_last_box"]).sum()),
    teacher_answer_chars=dict(min=int(AUDIT["teacher_answer_chars"].min()), mean=float(AUDIT["teacher_answer_chars"].mean()), max=int(AUDIT["teacher_answer_chars"].max())),
)
_failed = AUDIT[AUDIT["extraction_status"] != "ok"]
_mismatch = AUDIT[AUDIT["verify_status"] != "match"]
_unacknowledged = sorted(set(_mismatch["row"].tolist()) - set(AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS))
_stale_ack = sorted(set(AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS) - set(_mismatch["row"].tolist()))
_summary.update(rows_extraction_failed=_failed["row"].tolist(), rows_verify_mismatch=_mismatch["row"].tolist(),
                rows_verify_mismatch_unacknowledged=_unacknowledged, acknowledged_rows_not_mismatching=_stale_ack)
print(json.dumps(_summary, indent=1, ensure_ascii=False))

pd.set_option("display.max_colwidth", 120)
if len(_mismatch):
    print("\n--- math-verify 不一致行（教師原文の box と参照正解。代用も削除もしない） ---")
    print(_mismatch[["row", "teacher_answer", "answer", "verify_status", "n_distinct_boxes_after_think"]].to_string(index=False))
if len(_failed):
    print("\n--- 抽出失敗行 ---")
    print(_failed[["row", "extraction_status", "n_boxes_after_think"]].to_string(index=False))
_multi = AUDIT[AUDIT["n_distinct_boxes_after_think"] > 1]
print(f"\n--- 複数の異なる box を持つ行: {len(_multi)}（最後の box が参照正解と一致した行数: {(int((_multi['verify_status'] == 'match').sum()))}）例 ---")
print(_multi[["row", "n_boxes_after_think", "n_distinct_boxes_after_think", "first_box_after_think", "teacher_answer", "answer", "verify_status"]].head(8).to_string(index=False))

AO_DATA_READY = (len(_failed) == 0) and (len(_unacknowledged) == 0)
if not AO_DATA_READY:
    print("\n*** AO データに未解決の抽出/照合問題がある。同一データ条件の本学習へは進まない。"
          "不一致行を確認し、教師の最終回答として保持するなら AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS に行番号を追加する ***")
else:
    print("\nAO_DATA_READY = True（抽出失敗 0、未確認の不一致 0）")

# ---- 3.4 派生データの保存（元テーブルに列を追加するだけ。行順・message・advantage はそのまま） ----
AO_TABLE = RAW_TABLE
for _name, _vals, _type in [("teacher_answer", _targets, pa.string()), ("ao_extraction_status", _status, pa.string()),
                            ("ao_verify_status", _verify, pa.string()), ("ao_source_row", list(range(len(_targets))), pa.int64()),
                            ("ao_n_boxes_after_think", AUDIT["n_boxes_after_think"].tolist(), pa.int64())]:
    AO_TABLE = AO_TABLE.append_column(_name, pa.array(_vals, type=_type))
# 抽出失敗行は teacher_answer=null のまま保存する（削除も answer 代用もしない）。ゲートで本学習を止める。
assert AO_TABLE.num_rows == RAW_TABLE.num_rows
assert AO_TABLE.column("message").equals(RAW_TABLE.column("message")) and AO_TABLE.column("advantage").equals(RAW_TABLE.column("advantage"))
AO_PARQUET = f"{DATA_DIR}/Math-AO-20k.parquet"
pq.write_table(AO_TABLE, AO_PARQUET)
AO_SHA256 = sha256_of(AO_PARQUET)
# 読み戻して元列が同一であることを確認
_rt = pq.read_table(AO_PARQUET)
for _c in RAW_TABLE.column_names:
    assert _rt.column(_c).equals(RAW_TABLE.column(_c)), f"column {_c} changed after round trip"
print("saved AO parquet:", AO_PARQUET, "sha256:", AO_SHA256)

AUDIT.to_csv(f"{RECORD_DIR}/ao_audit_rows.csv", index=False)
with open(f"{RECORD_DIR}/ao_audit_summary.json", "w") as f:
    json.dump(dict(_summary, ao_parquet=AO_PARQUET, ao_sha256=AO_SHA256, ao_data_ready=AO_DATA_READY,
                   acknowledged_verify_mismatch_rows=AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS,
                   extraction_method="verl.utils.reward_score.math_verify_ours.last_boxed_only_string on text after </think>; "
                                     "verification = math_verify_ours.compute_score(target, answer)"), f, indent=2, ensure_ascii=False)
_flag_rows = sorted(set(_failed["row"]) | set(_mismatch["row"]) | set(_multi["row"].head(200)))
with open(f"{RECORD_DIR}/ao_flagged_rows.jsonl", "w") as f:
    for _r in _flag_rows:
        _rec = {k: (v.item() if hasattr(v, "item") else v) for k, v in AUDIT.iloc[_r].to_dict().items()}
        _rec["response_tail"] = _raw_df["response"].iloc[_r][-400:]
        f.write(json.dumps(_rec, ensure_ascii=False, default=str) + "\n")
print("audit files:", f"{RECORD_DIR}/ao_audit_summary.json", f"{RECORD_DIR}/ao_audit_rows.csv", f"{RECORD_DIR}/ao_flagged_rows.jsonl")
AO_AUDIT_SUMMARY = _summary

## 4. 長さ・target・loss mask・実効設定の確認

In [ ]:
%%ao
# @title 4-a. 共通ヘルパー: 公式スクリプトの解析・run 仕様の組み立て（公式実装は変更せず既存引数だけを渡す）
import re, json, os, math, pathlib, random, hashlib, datetime
import pyarrow as pa, pyarrow.parquet as pq
from omegaconf import OmegaConf
from huggingface_hub import snapshot_download, HfApi

HF_API = HfApi() if HF_LOGGED_IN else None
if HF_CKPT_REPO_ID is None and HF_ACCOUNT_NAME:
    HF_CKPT_REPO_ID = f"{HF_ACCOUNT_NAME}/rethink-sft-ao-checkpoints"   # HF の whoami から決める（GitHub 名からは推測しない）
print("HF checkpoint repo:", HF_CKPT_REPO_ID, "(private=%s)" % HF_CKPT_PRIVATE, "| upload during training:", HF_UPLOAD_CHECKPOINTS)


def hf_repo_exists(repo_id, repo_type="model"):
    if HF_API is None or not repo_id:
        return False
    try:
        HF_API.repo_info(repo_id, repo_type=repo_type)
        return True
    except Exception:
        return False


def hf_run_paths(repo_id, run_id):
    """HF 上の runs/<run_id>/ 配下の global_step_* を列挙（完了マーカー ao_upload_verified.json の有無付き）"""
    if not hf_repo_exists(repo_id):
        return {}
    try:
        files = HF_API.list_repo_files(repo_id, repo_type="model")
    except Exception:
        return {}
    out = {}
    for f in files:
        m = re.match(rf"runs/{re.escape(run_id)}/global_step_(\d+)/(.*)$", f)
        if m:
            out.setdefault(int(m.group(1)), set()).add(m.group(2))
    return {s: dict(files=sorted(v), complete=("ao_upload_verified.json" in v)) for s, v in out.items()}


def parse_official_script(path):
    """training_scripts/*.sh から変数と torchrun の hydra 上書き引数を取り出す（$VAR は解決する）"""
    text = pathlib.Path(path).read_text()
    var = {}
    exported = []
    for m in re.finditer(r"^(export\s+)?([A-Za-z_][A-Za-z0-9_]*)=(.*)$", text, re.M):
        is_export, k, v = bool(m.group(1)), m.group(2), m.group(3).strip()
        if v.startswith("$((") or v.startswith("$(") :
            continue
        v = v.strip().strip('"').strip("'")
        var[k] = v
        if is_export:
            exported.append(k)

    def resolve(v):
        for _ in range(3):
            v = re.sub(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}|\$([A-Za-z_][A-Za-z0-9_]*)",
                       lambda mm: var.get(mm.group(1) or mm.group(2), mm.group(0)), v)
        return v

    var = {k: resolve(v) for k, v in var.items()}
    m = re.search(r"torchrun(.*?)(?:>>|$)", text, re.S)
    assert m, "torchrun ブロックが見つからない"
    tokens = m.group(1).replace("\\\n", " ").split()
    overrides, torchrun_args, module = {}, [], None
    for i, t in enumerate(tokens):
        if t.startswith("--"):
            torchrun_args.append(resolve(t))
        elif t == "-m":
            module = tokens[i + 1]
        elif "=" in t and not t.startswith("-") and t != module:
            k, v = t.split("=", 1)
            overrides[k] = resolve(v.strip('"').strip("'"))
    env = {k: var[k] for k in exported if k not in ("WANDB_API_KEY", "PYTHONPATH")}
    return dict(path=str(path), vars=var, overrides=overrides, torchrun_args=torchrun_args, module=module, env=env)


def hydra_value(v):
    """hydra の CLI 値（文字列）を Python 値へ。OmegaConf で dataset config を組むときに使う"""
    if isinstance(v, (int, float, bool)) or v is None:
        return v
    s = str(v).strip()
    if s in ("null", "None"):
        return None
    if s in ("True", "true"):
        return True
    if s in ("False", "false"):
        return False
    if re.fullmatch(r"-?\d+", s):
        return int(s)
    if re.fullmatch(r"-?\d*\.\d+(e-?\d+)?|-?\d+e-?\d+", s):
        return float(s)
    if s.startswith("[") and s.endswith("]"):
        return json.loads(s.replace("'", '"'))
    return s


def fmt_lr(lr):
    mant, exp = f"{float(lr):e}".split("e")
    mant = mant.rstrip("0").rstrip(".")
    return f"{mant}e{int(exp)}"


OFFICIAL_SCRIPT_PATH = f"{REPO_DIR}/training_scripts/{MODEL_INFO['official_script']}"
OFFICIAL = parse_official_script(OFFICIAL_SCRIPT_PATH)
YAML_DEFAULTS = OmegaConf.load(f"{REPO_DIR}/verl/trainer/config/sft_trainer.yaml")
OFFICIAL_MAX_LENGTH = hydra_value(OFFICIAL["overrides"]["data.max_length"])
OFFICIAL_TBS = hydra_value(OFFICIAL["overrides"]["data.train_batch_size"])
OFFICIAL_MICRO_BSZ = hydra_value(OFFICIAL["overrides"]["data.micro_batch_size_per_gpu"])
OFFICIAL_LR = float(OFFICIAL["overrides"]["optim.lr"])
OFFICIAL_EPOCHS = hydra_value(OFFICIAL["overrides"]["trainer.total_epochs"])

print("公式スクリプト:", OFFICIAL_SCRIPT_PATH)
print("  module:", OFFICIAL["module"], "| torchrun args:", OFFICIAL["torchrun_args"])
print("  env exports:", OFFICIAL["env"])
print("  hydra overrides:")
for _k, _v in OFFICIAL["overrides"].items():
    print(f"    {_k} = {_v}")
print("  yaml 既定（スクリプトで上書きされない主要値）: weight_decay=%s warmup_steps_ratio=%s clip_grad=%s seed=%s strategy=%s grad_ckpt=%s lora_rank=%s"
      % (YAML_DEFAULTS.optim.weight_decay, YAML_DEFAULTS.optim.warmup_steps_ratio, YAML_DEFAULTS.optim.clip_grad, YAML_DEFAULTS.trainer.seed,
         YAML_DEFAULTS.model.strategy, YAML_DEFAULTS.model.enable_gradient_checkpointing, YAML_DEFAULTS.model.lora_rank))
assert hydra_value(YAML_DEFAULTS.model.lora_rank) == 0 and "lora_rank" not in OFFICIAL["overrides"], "Full-parameter SFT のはずが LoRA 設定がある"
assert OFFICIAL_LR == OFFICIAL_BASELINE["lr"] and OFFICIAL_EPOCHS == OFFICIAL_BASELINE["epochs"], "設定セルの OFFICIAL_BASELINE と公式スクリプトが一致しない"


def get_base_model_local():
    """Base モデルを pin した revision で取得し、ローカルパスを返す（model.partial_pretrain に渡す = revision 固定）"""
    d = f"{MODELS_DIR}/{MODEL_KEY}-Base_{MODEL_INFO['base_revision'][:8]}"
    p = snapshot_download(repo_id=MODEL_INFO["base_repo"], revision=MODEL_INFO["base_revision"], local_dir=d)
    return p.rstrip("/")


def steps_per_epoch(n_rows, tbs, n_gpus):
    """DistributedSampler(drop_last) + DataLoader(drop_last) と同じ計算"""
    per_rank_rows = n_rows // n_gpus
    per_rank_bs = tbs // n_gpus
    return per_rank_rows // per_rank_bs


# ---- 公式 8 GPU の micro batch 構成と勾配の大きさを N GPU で再現する -----------------------------
TRAIN_VIEW_VERSION = 1

# 再現しても残る差（公式コードを変えずには揃えられない）。実験記録と Model Card に書く
def residual_differences(n_gpus, max_length, emulated):
    out = []
    if n_gpus != OFFICIAL_WORLD_SIZE and emulated:
        out.append("勾配の加算の順序と精度: 公式は各 GPU で 8 micro batch 分を bf16 で加算し、GPU 間の平均は fp32。"
                   f"{n_gpus} GPU では {64 // n_gpus} micro batch 分を bf16 で加算する。丸め誤差の大きさが変わる"
                   "（CPU 上の公式 trainer と小型モデルでの検証: 6 step 後のパラメータ差は公式の移動量に対して 1 GPU で 1.5〜3.6%、"
                   "2 GPU で 0.7〜1.3%。並べ替えと loss 倍率なしの 1 GPU は 7〜14%。fp32 では 1e-6）")
        out.append(f"ログの train/loss: adv-only では token 平均が fp32 で計算され、vanilla では bf16。{OFFICIAL_WORLD_SIZE}/{n_gpus} 倍に戻した値は"
                   "公式ログと micro batch ごとの bf16 丸め 1 回分（最大 0.4% 程度）ずれ得る。勾配は影響を受けない")
    out.append("GPU の種類: 公式は H200。Colab の GPU では選ばれる CUDA カーネルが変わり、丸め誤差が変わる")
    if GPU_PROFILE_DEVIATION:
        out.append(f"torch の版（GPU プロファイル {GPU_PROFILE}）: {GPU_PROFILE_DEVIATION}")
    out.append("flash-attn の backward は非決定的（公式の学習も同じ条件）")
    if int(max_length) != int(OFFICIAL_MAX_LENGTH):
        out.append(f"data.max_length {max_length}（公式 {OFFICIAL_MAX_LENGTH}）: padding の量だけが変わる。padding は loss_mask と attention mask で除外されるので"
                   "loss と勾配は数学的に同じ（行列積の形が変わるので丸め誤差は変わる）")
    if n_gpus != OFFICIAL_WORLD_SIZE:
        out.append(f"train/grad_norm のログ: 公式 trainer は各 GPU の shard ノルムの平均を記録する（全体ノルムではない。CPU 上の公式 trainer で確認）。"
                   f"公式は {OFFICIAL_WORLD_SIZE} 分割、この run は {n_gpus} 分割の平均なので、公式ログとは直接比べられない（clip は正しい全体ノルムで行われる）")
    return out


def emulate_official_order(n_rows, official_world=8, actual_world=1, global_batch=256, micro=4):
    """公式 trainer（DistributedSampler(shuffle=False, drop_last=True) + StatefulDataLoader(batch=global_batch/N) + split(micro)）を
    N GPU で動かしたとき、各 step の各 micro batch が公式 official_world GPU の run と同じ行の組になる並び順を返す。
    戻り値 order: 学習用 file の p 行目 = 元 file の order[p] 行目。
      公式:  step k, rank r, micro t, 要素 i -> 元の行 B*k + r + W*(M*t + i)
      N GPU: step k, rank q, 位置 j        -> file 位置 B*k + N*j + q（j = (r_local*T + t)*M + i, r = q*G + r_local）
    256 行の block 内だけで並べ替えるので、先頭 m*256 行の試走 subset でも全体の先頭 m step と同じ micro batch になる。
    末尾の端数行（n_rows % 256）はどちらの run でも使われないので動かさない。"""
    W, N, B, M = int(official_world), int(actual_world), int(global_batch), int(micro)
    assert W >= 1 and N >= 1 and W % N == 0, f"actual_world {N} must divide official_world {W}"
    assert B % W == 0 and (B // W) % M == 0, f"global_batch {B} / official_world {W} / micro {M} inconsistent"
    G = W // N              # 1 つの実 GPU が受け持つ公式 rank の数
    T = B // W // M         # 公式 1 rank あたりの micro batch 数（256/8/4 = 8）
    S = n_rows // B         # 1 epoch の step 数（W でも N でも同じ）
    order = list(range(n_rows))
    for k in range(S):
        base = B * k
        for q in range(N):
            for r_local in range(G):
                r = q * G + r_local
                for t in range(T):
                    for i in range(M):
                        j = (r_local * T + t) * M + i
                        order[base + N * j + q] = base + r + W * (M * t + i)
    return order


def prepare_train_file(source_path, source_sha256, n_gpus):
    """学習に渡す parquet を返す: (path, sha256, view_meta)。
    N=8 または EMULATE_OFFICIAL_WORLD_SIZE=False なら元の file をそのまま使う。
    それ以外は元の file（AO parquet または試走 subset）を読み、emulate_official_order で行を並べ替え、advantage 列を N/8 にした
    派生 file を別名で書く。元の file は変更しない。派生 file の各行は元の行と teacher_answer / message が同一で、ao_source_row で対応が分かる。"""
    off = dict(enabled=False, version=TRAIN_VIEW_VERSION, official_world=OFFICIAL_WORLD_SIZE, actual_world=int(n_gpus),
               importance_sampling_mode=OFFICIAL["overrides"].get("trainer.importance_sampling_mode", "vanilla"), advantage=1.0,
               logged_loss_scale=1.0, source_path=source_path, source_sha256=source_sha256, train_file=source_path, train_file_sha256=source_sha256)
    if int(n_gpus) == OFFICIAL_WORLD_SIZE or not EMULATE_OFFICIAL_WORLD_SIZE:
        return source_path, source_sha256, off
    W, N = OFFICIAL_WORLD_SIZE, int(n_gpus)
    assert W % N == 0, f"GPU 台数 {N} は {W} の約数でないため公式 {W} GPU の micro batch 構成を再現できない（1, 2, 4, 8 台で実行する）"
    micro = OFFICIAL_MICRO_BSZ
    assert off["importance_sampling_mode"] == "vanilla", "公式スクリプトが vanilla 以外のときはこの再現方法を使えない"
    tbl = pq.read_table(source_path)
    n = tbl.num_rows
    adv_col = tbl.column("advantage")
    assert all(v == 1.0 for v in adv_col.to_pylist()), "元の advantage 列が 1.0 以外を含む。loss の倍率で勾配を合わせる前提が崩れる"
    order = emulate_official_order(n, W, N, OFFICIAL_TBS, micro)
    adv = N / W                                     # 1/8, 1/4, 1/2: 2 のべき乗なので浮動小数点で誤差なく掛けられる
    view = tbl.take(pa.array(order, type=pa.int64()))
    view = view.set_column(view.schema.get_field_index("advantage"), view.schema.field("advantage"), pa.array([adv] * n, type=adv_col.type))
    stem = os.path.splitext(os.path.basename(source_path))[0]
    out_dir = os.path.join(os.path.dirname(source_path), "train_view")
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, f"{stem}.emul-w{W}-n{N}.parquet")
    pq.write_table(view, out)
    # 書いた file を読み戻して、行の対応と advantage を確認する
    rt = pq.read_table(out)
    assert "ao_source_row" in tbl.column_names, "元の file に ao_source_row 列が無い（セクション 3 で作った AO parquet かその先頭 subset を渡す）"
    src_ids = tbl.column("ao_source_row").to_pylist()
    assert rt.num_rows == n and rt.column("ao_source_row").to_pylist() == [src_ids[o] for o in order], "派生 file の行の対応が崩れている"
    for c in ("message", "teacher_answer", "answer"):
        if c in tbl.column_names:
            assert rt.column(c).equals(tbl.column(c).take(pa.array(order, type=pa.int64()))), f"派生 file の {c} 列が元の行と一致しない"
    assert set(rt.column("advantage").to_pylist()) == {adv}
    meta = dict(off, enabled=True, importance_sampling_mode="adv-only", advantage=adv, logged_loss_scale=adv, global_batch=OFFICIAL_TBS, micro=micro,
                n_rows=n, order_sha256=hashlib.sha256(json.dumps(order).encode()).hexdigest(), train_file=out, train_file_sha256=sha256_of(out))
    return out, meta["train_file_sha256"], meta


def official_micro_batches(file_source_rows, world, global_batch, micro):
    """公式 trainer と同じ DistributedSampler で world 台に配ったときの、各 step の micro batch（元の行番号の組）を返す。
    戻り値: steps[k] = sorted list of tuples（その step の全 rank の micro batch）"""
    import torch.utils.data as tud
    n = len(file_source_rows)
    per_rank_bs = global_batch // world
    steps = None
    for r in range(world):
        idx = list(iter(tud.DistributedSampler(range(n), num_replicas=world, rank=r, shuffle=False, drop_last=True)))
        n_steps = len(idx) // per_rank_bs
        if steps is None:
            steps = [[] for _ in range(n_steps)]
        for k in range(n_steps):
            b = idx[k * per_rank_bs:(k + 1) * per_rank_bs]
            for m in range(0, per_rank_bs, micro):
                steps[k].append(tuple(file_source_rows[i] for i in b[m:m + micro]))
    return [sorted(s) for s in steps]


def normalize_run(entry):
    """SEARCH_RUN_LIST の要素 (lr, epochs) / (lr, epochs, scheduler) を (lr, epochs, scheduler) にそろえる"""
    assert isinstance(entry, (list, tuple)) and len(entry) in (2, 3), \
        f"探索候補は (lr, epochs) か (lr, epochs, scheduler) のタプルで書く（旧形式の {{'lr': [...], 'epochs': [...]}} は使えない）: {entry!r}"
    lr, ep, *rest = entry
    sched = rest[0] if rest else "cosine"
    assert sched in ("cosine", "constant"), f"lr_scheduler は論文で使われた cosine / constant のみ: {sched}"
    return float(lr), int(ep), sched


def paper_condition_of(lr, epochs, scheduler):
    """論文の最適化条件（PAPER_OPTIMIZATION_CONDITIONS）に一致すれば、その出典を返す"""
    for c in PAPER_OPTIMIZATION_CONDITIONS:
        if abs(c["lr"] - float(lr)) < 1e-12 and c["epochs"] == int(epochs) and c["scheduler"] == scheduler:
            return c["paper"]
    return None


def make_run_id(kind, lr, epochs, tbs=None, suffix="", scheduler="cosine"):
    tbs = tbs or OFFICIAL_TBS
    base = f"{MODEL_KEY}_Math-AO-20k_lr{fmt_lr(lr)}_ep{epochs}_bs{tbs}" + ("_ConstLR" if scheduler == "constant" else "")   # 公式スクリプト名と同じ付け方
    return f"{base}_{kind}{suffix}"


def build_run_spec(kind, lr, epochs, data_path, data_sha256, n_rows, max_length, save_freq, run_id=None, n_gpus=None,
                   resume_from_path=None, total_training_steps=None, extra_note=None, upload_policy=None, lr_scheduler="cosine"):
    """公式スクリプトの上書き引数を起点に、AO 学習に必要な最小限の引数だけ差し替える。差分は changes に記録する"""
    n_gpus = n_gpus or N_GPUS
    assert n_gpus >= 1, "GPU がない"
    ov = dict(OFFICIAL["overrides"])
    changes = []

    def set_(k, v, reason):
        old = ov.get(k, "<yaml default>")
        v = str(v)
        if old != v:
            changes.append(dict(key=k, official=old, new=v, reason=reason))
        ov[k] = v

    run_id = run_id or make_run_id(kind, lr, epochs, suffix=RUN_ID_SUFFIX, scheduler=lr_scheduler)
    _paper_cond = paper_condition_of(lr, epochs, lr_scheduler)
    base_local = get_base_model_local()
    set_("model.partial_pretrain", base_local, f"同じ Base を revision {MODEL_INFO['base_revision'][:8]} で固定したローカルパス")
    train_path, train_sha, view = prepare_train_file(data_path, data_sha256, n_gpus)
    set_("data.train_files", train_path, "AO データ（teacher_answer 列を追加した Math-CoT-20k）" +
         (f"を公式 {OFFICIAL_WORLD_SIZE} GPU の micro batch 構成になるよう並べ替えた学習用 file" if view["enabled"] else ""))
    set_("data.val_files", train_path, "公式同様 train と同じ値（参照版 trainer では validation はコメントアウト）")
    set_("data.response_key", "teacher_answer", "AO target 列")
    if view["enabled"]:
        set_("trainer.importance_sampling_mode", "adv-only",
             f"loss = -advantage × log_prob。advantage 列 = {view['advantage']}（= {n_gpus}/{OFFICIAL_WORLD_SIZE}）で vanilla の loss を {view['advantage']} 倍し、"
             f"{n_gpus} GPU の勾配を公式 {OFFICIAL_WORLD_SIZE} GPU と同じ大きさにする（勾配・clip・AdamW の入力が丸め誤差の範囲で公式と同じ値になる）")
        changes.append(dict(key="学習用 file の行順 / advantage 列", official="Math-CoT-20k の行順 / 1.0",
                            new=f"公式 {OFFICIAL_WORLD_SIZE} GPU の micro batch 構成を再現する並べ替え / {view['advantage']}",
                            reason=f"{n_gpus} GPU でも各 step の各 micro batch（4 行）を公式 {OFFICIAL_WORLD_SIZE} GPU と同じ行の組にする。"
                                   "各 step で使う 256 行の集合と epoch ごとの順序は公式と同じ。元の AO parquet は変更しない"))
    set_("data.max_length", max_length, "AO は短いので全行が収まる長さへ短縮（時間短縮。公式は 20000）" if max_length != OFFICIAL_MAX_LENGTH else "公式値")
    _why = f"探索対象（論文の条件: {_paper_cond}）" if _paper_cond else "探索対象（論文に無い条件）"
    set_("optim.lr", fmt_lr(lr), _why if float(lr) != OFFICIAL_LR else "公式値")
    set_("trainer.total_epochs", epochs, _why if int(epochs) != OFFICIAL_EPOCHS else "公式値")
    set_("optim.lr_scheduler", lr_scheduler, _why if lr_scheduler != OFFICIAL["overrides"].get("optim.lr_scheduler", "cosine") else "公式値")
    set_("trainer.save_freq", save_freq, "checkpoint 周期（正整数）。保存は学習結果に影響しない" if int(save_freq) != hydra_value(OFFICIAL["overrides"]["trainer.save_freq"]) else "公式値")
    set_("trainer.checkpoint.save_contents", '["model","optimizer","extra"]', "再開に必要な状態一式を保存（公式 CoT は model のみ）")
    set_("trainer.checkpoint.load_contents", '["model","optimizer","extra"]', "再開時に一式を読む")
    set_("trainer.default_local_dir", f"{CKPT_DIR}/{run_id}", "run 別ディレクトリ")
    set_("trainer.experiment_name", run_id, "run ID")
    set_("trainer.project_name", WANDB_PROJECT, "wandb project")
    set_("trainer.nnodes", 1, "Colab 単一ノード")
    if MICRO_BATCH_OVERRIDE is not None:
        assert not view["enabled"], "公式 8 GPU の再現中は micro batch を変えられない"
        set_("data.micro_batch_size_per_gpu", MICRO_BATCH_OVERRIDE, "メモリ都合の micro batch 変更（global batch 256 は維持）。micro batch 構成と token の重みが公式と変わる")
    if resume_from_path:
        set_("trainer.resume_mode", "resume_path", "HF から取得した checkpoint から再開")
        set_("trainer.resume_from_path", resume_from_path, "再開元")
    else:
        set_("trainer.resume_mode", "disable", "新規 run（同じ Base から独立に開始）")
    if total_training_steps is not None:
        set_("trainer.total_training_steps", total_training_steps, "試走専用の step 上限（scheduler 期間も変わる。本学習では使わない）")

    tbs = hydra_value(ov["data.train_batch_size"])
    micro = hydra_value(ov["data.micro_batch_size_per_gpu"])
    assert tbs % n_gpus == 0 and (tbs // n_gpus) % micro == 0, f"train_batch_size {tbs} / n_gpus {n_gpus} / micro {micro} の整合が取れない"
    spe = steps_per_epoch(n_rows, tbs, n_gpus)
    total_steps = int(math.ceil(spe * float(epochs))) if total_training_steps is None else int(total_training_steps)
    warmup = int(total_steps * float(YAML_DEFAULTS.optim.warmup_steps_ratio))
    env = dict(OFFICIAL["env"]) if KEEP_OFFICIAL_ENV_VARS else {k: v for k, v in OFFICIAL["env"].items() if k != "CUDA_LAUNCH_BLOCKING"}
    if not KEEP_OFFICIAL_ENV_VARS:
        changes.append(dict(key="env:CUDA_LAUNCH_BLOCKING", official="1", new="<unset>", reason="KEEP_OFFICIAL_ENV_VARS=False（速度優先。公式は 1）"))
    env["WANDB_MODE"] = WANDB_MODE if (WANDB_MODE == "offline" or os.environ.get("WANDB_API_KEY")) else "offline"
    env["WANDB_DIR"] = f"{WORK_DIR}/wandb"
    env["VERL_SFT_LOGGING_LEVEL"] = "INFO"     # 再開ログ（checkpoint 読み込み等）を表示する。既存の環境変数
    changes.append(dict(key="env:WANDB_DIR / VERL_SFT_LOGGING_LEVEL / PYTHONUNBUFFERED", official="<unset>", new=f"{env['WANDB_DIR']} / INFO / 1",
                        reason="wandb 出力先・再開ログ表示・ログ即時出力（学習内容に影響しない環境変数）"))
    if n_gpus == OFFICIAL_WORLD_SIZE:
        _nproc_reason = "公式と同じ台数"
    elif view["enabled"]:
        _nproc_reason = (f"実 GPU 台数。global batch {tbs} は 1 GPU あたり {(tbs // n_gpus) // micro} micro batch の勾配蓄積で実現し、"
                         f"行の並べ替えと loss の {view['advantage']} 倍で公式 {OFFICIAL_WORLD_SIZE} GPU の勾配を再現")
    else:
        _nproc_reason = (f"実 GPU 台数。EMULATE_OFFICIAL_WORLD_SIZE=False のため、micro batch 構成と勾配の大きさ（{OFFICIAL_WORLD_SIZE / n_gpus:g} 倍）"
                         "が公式と異なる。clip_grad=1.0 の効き方も変わる")
    changes.append(dict(key="torchrun --nproc_per_node", official=f"{OFFICIAL_WORLD_SIZE} (README: 8 x H200)", new=str(n_gpus), reason=_nproc_reason))
    _saves = sorted(set([s for s in range(int(save_freq), total_steps + 1, int(save_freq))] + [total_steps]))
    upload_policy = upload_policy or HF_UPLOAD_STEPS
    if upload_policy == "all":
        _uploads = list(_saves)
        _full = list(_saves)
    else:   # 論文が評価した step + 再開用の間隔 + 最終 step
        _uploads = sorted(s for s in _saves if s in PAPER_EVAL_STEPS or s % RESUME_CKPT_EVERY == 0 or s == total_steps)
        # 再開用（optimizer 状態込み）は RESUME_CKPT_EVERY の倍数と最終 step。それ以外（公開 CoT と同じ分析用 step）は model_only なら重みだけ送る
        _full = _uploads if HF_ANALYSIS_STEP_CONTENT == "full" else [s for s in _uploads if s % RESUME_CKPT_EVERY == 0 or s == total_steps]
    residual = residual_differences(n_gpus, max_length, view["enabled"])
    if n_gpus != OFFICIAL_WORLD_SIZE and not view["enabled"]:
        residual = [f"【再現無効】micro batch 構成と勾配の大きさ（{OFFICIAL_WORLD_SIZE / n_gpus:g} 倍）が公式 {OFFICIAL_WORLD_SIZE} GPU と異なる"] + residual
    spec = dict(
        run_id=run_id, kind=kind, lr=float(lr), lr_str=fmt_lr(lr), epochs=epochs, lr_scheduler=lr_scheduler, paper_condition=_paper_cond,
        model_key=MODEL_KEY, base_repo=MODEL_INFO["base_repo"],
        base_revision=MODEL_INFO["base_revision"], base_local=base_local, cot_repo=MODEL_INFO["cot_repo"], cot_revision=MODEL_INFO["cot_revision"],
        cot_subfolder=MODEL_INFO["cot_subfolder"], teacher=TEACHER, data_path=data_path, data_sha256=data_sha256, n_rows=n_rows,
        dataset_repo=DATASET_REPO, dataset_revision=DATASET_REVISION, target_style=AO_TARGET_STYLE, max_length=int(max_length),
        official_max_length=OFFICIAL_MAX_LENGTH, tbs=tbs, micro_bsz=micro, n_gpus=n_gpus, grad_accum_micro_batches=(tbs // n_gpus) // micro,
        steps_per_epoch=spe, total_steps=total_steps, warmup_steps=warmup, save_freq=int(save_freq),
        expected_save_steps=_saves, upload_steps=_uploads, upload_full_steps=_full, upload_policy=upload_policy,
        gpu_profile=GPU_PROFILE, gpu_profile_deviation=GPU_PROFILE_DEVIATION,
        overrides=ov, changes_vs_official=changes, env=env, official_script=OFFICIAL["path"], module=OFFICIAL["module"],
        fork_commit=FORK_COMMIT, upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, keep_official_env_vars=KEEP_OFFICIAL_ENV_VARS,
        resume_from_path=resume_from_path, total_training_steps_override=total_training_steps, note=extra_note,
        train_file=train_path, train_file_sha256=train_sha, training_view=view, logged_loss_scale=view["logged_loss_scale"],
        official_world_size=OFFICIAL_WORLD_SIZE, residual_differences=residual,
        created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    )
    return spec


def print_run_spec(spec):
    print(f"run_id={spec['run_id']} kind={spec['kind']} lr={spec['lr_str']} epochs={spec['epochs']} lr_scheduler={spec.get('lr_scheduler', 'cosine')} n_gpus={spec['n_gpus']}"
          f" | 論文の条件: {spec.get('paper_condition') or '該当なし（論文に無い条件）'}")
    print(f"  data={spec['data_path']} rows={spec['n_rows']} sha256={spec['data_sha256'][:12]} max_length={spec['max_length']} (official {spec['official_max_length']})")
    v = spec["training_view"]
    if v["enabled"]:
        print(f"  公式 {v['official_world']} GPU の再現: ON（学習用 file={os.path.basename(spec['train_file'])} sha256={spec['train_file_sha256'][:12]}, "
              f"importance_sampling_mode=adv-only, advantage={v['advantage']}）。ログの train/loss は {v['logged_loss_scale']} 倍（表示時は公式スケールに戻す）")
    elif spec["n_gpus"] == spec["official_world_size"]:
        print(f"  GPU 台数が公式と同じ {spec['n_gpus']} 台。並べ替えと loss 倍率は不要")
    else:
        print(f"  *** 公式 {spec['official_world_size']} GPU の再現: OFF。micro batch 構成と勾配の大きさが公式と異なる ***")
    print(f"  global batch={spec['tbs']} micro/gpu={spec['micro_bsz']} grad-accum micro-batches/step={spec['grad_accum_micro_batches']}")
    print(f"  steps/epoch={spec['steps_per_epoch']} total_steps={spec['total_steps']} warmup={spec['warmup_steps']} save_freq={spec['save_freq']} -> saves at {spec['expected_save_steps']}")
    _full = spec.get("upload_full_steps", spec["upload_steps"])
    _light = [x for x in spec["upload_steps"] if x not in _full]
    print(f"  HF へ転送する step（{spec['upload_policy']}）: optimizer 状態込み（再開可）{_full}"
          + (f"、重みのみ（分析用・再開不可）{_light}" if _light else "") + "。それ以外はローカルで保存完了後に削除")
    print("  公式スクリプトとの差分:")
    for c in spec["changes_vs_official"]:
        print(f"    {c['key']}: {c['official']} -> {c['new']}   [{c['reason']}]")
    if not spec["keep_official_env_vars"]:
        print("    env: CUDA_LAUNCH_BLOCKING を外した（KEEP_OFFICIAL_ENV_VARS=False）")
    print("  公式と揃えられない残りの差（数値の丸め程度）:")
    for r in spec["residual_differences"]:
        print(f"    - {r}")


print("helpers ready: parse_official_script / build_run_spec / print_run_spec / get_base_model_local / fmt_lr / emulate_official_order / prepare_train_file")

In [ ]:
%%ao
# @title 4-b. 長さ・target・loss mask・実効設定の確認（公式 tokenizer / chat template / OurSFTDataset をそのまま使う）
import numpy as np, torch, json, time, math, os, glob, collections
from omegaconf import OmegaConf
from verl.utils import hf_tokenizer
from verl.utils.dataset.sft_dataset import OurSFTDataset

BASE_MODEL_LOCAL = get_base_model_local()
print("base model local:", BASE_MODEL_LOCAL)
TOKENIZER = hf_tokenizer(BASE_MODEL_LOCAL, trust_remote_code=hydra_value(OFFICIAL["overrides"].get("model.trust_remote_code", False)))
# 公式 run_sft と同じ eos 上書き処理（公式スクリプトは data.eos_token=null / eos_token_id=null）
if hydra_value(OFFICIAL["overrides"].get("data.eos_token")) is not None:
    TOKENIZER.eos_token = hydra_value(OFFICIAL["overrides"]["data.eos_token"])
if hydra_value(OFFICIAL["overrides"].get("data.eos_token_id")) is not None:
    TOKENIZER.eos_token_id = hydra_value(OFFICIAL["overrides"]["data.eos_token_id"])
print(f"tokenizer: eos={TOKENIZER.eos_token!r}({TOKENIZER.eos_token_id}) pad={TOKENIZER.pad_token!r}({TOKENIZER.pad_token_id}) bos={TOKENIZER.bos_token!r}")

# ---- 4.1 全行の Q + 回答 + EOS 長（OurSFTDataset.__getitem__ と同じ手順） ----
_msgs = RAW_TABLE.column("message").to_pylist()
_targets_all = AO_TABLE.column("teacher_answer").to_pylist()
t0 = time.time()
PROMPT_LENS = np.array([len(TOKENIZER.apply_chat_template(list(m), add_generation_prompt=True, tokenize=True)) for m in _msgs])
RESP_LENS = np.array([len(TOKENIZER(t if t is not None else "", add_special_tokens=False)["input_ids"]) + 1 for t in _targets_all])  # +1 = EOS
TOTAL_LENS = PROMPT_LENS + RESP_LENS
print(f"tokenized {len(TOTAL_LENS)} rows in {time.time() - t0:.1f}s")
for _name, _arr in (("prompt", PROMPT_LENS), ("target+EOS", RESP_LENS), ("total", TOTAL_LENS)):
    print(f"  {_name:11s} min={_arr.min()} mean={_arr.mean():.1f} p50={int(np.median(_arr))} p95={int(np.percentile(_arr, 95))} p99={int(np.percentile(_arr, 99))} max={_arr.max()}")
_example_prompt = TOKENIZER.apply_chat_template(list(_msgs[0]), add_generation_prompt=True, tokenize=False)
print("chat template 例 (row 0):", repr(_example_prompt[:120]), "...", repr(_example_prompt[-40:]))
print("target 例 (row 0):", repr(_targets_all[0]), "-> tokens", TOKENIZER.convert_ids_to_tokens(TOKENIZER(_targets_all[0], add_special_tokens=False)["input_ids"]), "+ EOS")

MAX_TOTAL_LEN = int(TOTAL_LENS.max())
AUTO_FIT_MAX_LENGTH = int(math.ceil(MAX_TOTAL_LEN / MAX_LENGTH_ROUND_TO) * MAX_LENGTH_ROUND_TO)
if MAX_LENGTH_MODE == "auto_fit":
    MAX_LENGTH = AUTO_FIT_MAX_LENGTH
elif MAX_LENGTH_MODE == "official":
    MAX_LENGTH = OFFICIAL_MAX_LENGTH
else:
    MAX_LENGTH = int(MAX_LENGTH_MODE)
_n_trunc = int((TOTAL_LENS > MAX_LENGTH).sum())
print(f"\n全行が切り詰められずに収まる最大長 = {MAX_TOTAL_LEN} tokens（{MAX_LENGTH_ROUND_TO} の倍数へ切り上げ: {AUTO_FIT_MAX_LENGTH}）")
print(f"data.max_length = {MAX_LENGTH}（mode={MAX_LENGTH_MODE}、公式 {OFFICIAL_MAX_LENGTH}）。この長さで切り詰められる行数: {_n_trunc}")
print(f"公式 20000 での padding 率: {(1 - TOTAL_LENS.mean() / OFFICIAL_MAX_LENGTH) * 100:.1f}% / 選択値での padding 率: {(1 - TOTAL_LENS.mean() / MAX_LENGTH) * 100:.1f}%")
if _n_trunc > 0:
    print("*** 切り詰められる行がある。data.truncation=right で target/EOS が失われる → max_length を上げる ***")

# ---- 4.2 公式 OurSFTDataset で loss mask を確認 ----
_data_cfg = OmegaConf.create(OmegaConf.to_container(YAML_DEFAULTS.data, resolve=True))
for _k, _v in OFFICIAL["overrides"].items():
    if _k.startswith("data."):
        OmegaConf.update(_data_cfg, _k[len("data."):], hydra_value(_v), merge=True)
OmegaConf.update(_data_cfg, "train_files", AO_PARQUET)
OmegaConf.update(_data_cfg, "val_files", AO_PARQUET)
OmegaConf.update(_data_cfg, "response_key", "teacher_answer")
OmegaConf.update(_data_cfg, "max_length", int(MAX_LENGTH))
print("\ndataset config (data.*):", json.dumps(OmegaConf.to_container(_data_cfg), ensure_ascii=False)[:600])
DS = OurSFTDataset(parquet_files=AO_PARQUET, tokenizer=TOKENIZER, config=_data_cfg)
assert len(DS) == DATASET_EXPECTED_ROWS


def check_item(i):
    it = DS[i]
    p, r = int(PROMPT_LENS[i]), int(RESP_LENS[i])
    ids, am, lm = it["input_ids"], it["attention_mask"], it["loss_mask"]
    ok = dict(
        length=(ids.shape[0] == MAX_LENGTH),
        no_truncation=(int(am.sum()) == p + r),
        loss_mask_count=(int(lm.sum()) == r),                                              # target tokens + EOS の数
        loss_mask_positions=(torch.nonzero(lm).flatten().tolist() == list(range(p - 1, p + r - 1))),  # 1 つ前の位置が次 token を予測
        prompt_masked=(int(lm[: p - 1].sum()) == 0),
        padding_masked=(int(lm[p + r - 1:].sum()) == 0 and int(am[p + r:].sum()) == 0),
        eos_supervised=(int(ids[p + r - 1]) == TOKENIZER.eos_token_id and int(lm[p + r - 2]) == 1),
        target_tokens=(ids[p: p + r - 1].tolist() == TOKENIZER(_targets_all[i], add_special_tokens=False)["input_ids"]),
        advantage=(float(it["advantages"][0]) == 1.0),
    )
    return ok, it, p, r


t0 = time.time()
_bad = collections.Counter()
FULL_MASK_SCAN = True
_null_rows = [i for i, t in enumerate(_targets_all) if t is None]   # 抽出失敗行（OurSFTDataset は str を要求するので走査から除く。ゲートは AO_DATA_READY）
_scan_rows = [i for i in (range(len(DS)) if FULL_MASK_SCAN else range(0, len(DS), 97)) if _targets_all[i] is not None]
for _i in _scan_rows:
    _ok, _, _, _ = check_item(_i)
    for _k, _v in _ok.items():
        if not _v:
            _bad[_k] += 1
print(f"\nloss mask scan over {len(_scan_rows)} rows in {time.time() - t0:.1f}s -> failures: {dict(_bad) or 'none'} | skipped null-target rows: {len(_null_rows)}")
MASK_CHECK_OK = (len(_bad) == 0) and (len(_null_rows) == 0)
for _i in [i for i in (0, int(TOTAL_LENS.argmax())) if _targets_all[i] is not None]:
    _ok, _it, _p, _r = check_item(_i)
    _sup = _it["input_ids"][_p: _p + _r]
    print(f"row {_i}: prompt_len={_p} target+eos={_r} supervised tokens -> {TOKENIZER.decode(_sup)!r}  checks={_ok}")

# ---- 4.3 実効設定 ----
_param_bytes = sum(os.path.getsize(f) for f in glob.glob(f"{BASE_MODEL_LOCAL}/*.safetensors"))
CKPT_SIZE_EST_GB = _param_bytes * 3 / 1e9   # bf16 params + AdamW の 2 状態（公式は model_dtype=bf16 で読み込む）。1.7B の実測 10.34 GB と 0.2% 以内
CKPT_MODEL_ONLY_EST_GB = _param_bytes / 1e9 + 0.02   # 重み + extra/data.pt/huggingface/（optimizer 状態なし）


def hf_usage_gb(n_full, n_model_only):
    return n_full * CKPT_SIZE_EST_GB + n_model_only * CKPT_MODEL_ONLY_EST_GB


print("\n=== 実効設定（1 GPU あたり micro batch、勾配蓄積） ===")
for _lr, _ep, _kind in [(OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], "baseline")] + [normalize_run(r)[:2] + ("search",) for r in SEARCH_RUN_LIST]:
    _spe = steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, max(N_GPUS, 1))
    _micro = MICRO_BATCH_OVERRIDE or OFFICIAL_MICRO_BSZ
    _total = int(math.ceil(_spe * _ep))
    _saves = sorted(set(list(range(SAVE_FREQ, _total + 1, SAVE_FREQ)) + [_total]))
    _up = _saves if HF_UPLOAD_STEPS == "all" else [x for x in _saves if x in PAPER_EVAL_STEPS or x % RESUME_CKPT_EVERY == 0 or x == _total]
    _nfull = len(_up) if (HF_UPLOAD_STEPS == "all" or HF_ANALYSIS_STEP_CONTENT == "full") else len([x for x in _up if x % RESUME_CKPT_EVERY == 0 or x == _total])
    print(f"  {_kind:8s} lr={fmt_lr(_lr)} ep={_ep}: steps/epoch={_spe} total={_total} warmup={int(_total * float(YAML_DEFAULTS.optim.warmup_steps_ratio))} "
          f"micro/gpu={_micro} accum={(OFFICIAL_TBS // max(N_GPUS, 1)) // _micro} HF uploads={len(_up)}（再開用 {_nfull} x {CKPT_SIZE_EST_GB:.1f} GB + "
          f"重みのみ {len(_up) - _nfull} x {CKPT_MODEL_ONLY_EST_GB:.1f} GB ≈ {hf_usage_gb(_nfull, len(_up) - _nfull):.0f} GB on HF）")
print("  loss: micro batch ごとの token 平均で backward し、1 GPU 内では micro batch の勾配を「和」で蓄積、GPU 間は FSDP2 が「平均」する（公式 training_step）。"
      "ログの train/loss は micro batch の loss の平均")
print("  optimizer: AdamW betas=%s wd=%s, scheduler=%s warmup_ratio=%s, clip_grad=%s, bf16, FSDP(%s), grad ckpt=%s, shuffle=%s, truncation=%s"
      % (OFFICIAL["overrides"].get("optim.betas"), YAML_DEFAULTS.optim.weight_decay, OFFICIAL["overrides"].get("optim.lr_scheduler"),
         YAML_DEFAULTS.optim.warmup_steps_ratio, YAML_DEFAULTS.optim.clip_grad, YAML_DEFAULTS.model.strategy, YAML_DEFAULTS.model.enable_gradient_checkpointing,
         OFFICIAL["overrides"].get("data.shuffle_train"), OFFICIAL["overrides"].get("data.truncation")))
# ---- 4.4 公式 8 GPU の micro batch 構成と勾配の大きさの再現を確認 ----
_n_check = max(N_GPUS, 1)
if EMULATE_OFFICIAL_WORLD_SIZE and OFFICIAL_WORLD_SIZE % _n_check != 0:
    _vpath, _vsha, TRAIN_VIEW_CHECK = None, None, dict(enabled=False, actual_world=_n_check)
else:
    _vpath, _vsha, TRAIN_VIEW_CHECK = prepare_train_file(AO_PARQUET, AO_SHA256, _n_check)
if EMULATE_OFFICIAL_WORLD_SIZE and OFFICIAL_WORLD_SIZE % _n_check != 0:
    VIEW_CHECK_OK = False
    print(f"\n*** GPU {_n_check} 台は {OFFICIAL_WORLD_SIZE} の約数でないため、公式 {OFFICIAL_WORLD_SIZE} GPU の micro batch 構成を再現できない。1, 2, 4, 8 台のランタイムで実行する ***")
elif _n_check == OFFICIAL_WORLD_SIZE:
    VIEW_CHECK_OK = True
    print(f"\nGPU {_n_check} 台 = 公式と同じ台数。並べ替えと loss 倍率は使わない")
elif not TRAIN_VIEW_CHECK["enabled"]:
    VIEW_CHECK_OK = False
    print(f"\n*** EMULATE_OFFICIAL_WORLD_SIZE=False: GPU {_n_check} 台では micro batch 構成と勾配の大きさ（{OFFICIAL_WORLD_SIZE / _n_check:g} 倍）が公式と異なる ***")
else:
    t0 = time.time()
    _src_rows_view = pq.read_table(_vpath, columns=["ao_source_row"]).column("ao_source_row").to_pylist()
    _official_mb = official_micro_batches(list(range(DATASET_EXPECTED_ROWS)), OFFICIAL_WORLD_SIZE, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ)
    _mine_mb = official_micro_batches(_src_rows_view, _n_check, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ)
    _naive_mb = official_micro_batches(list(range(DATASET_EXPECTED_ROWS)), _n_check, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ)
    _same_steps = sum(a == b for a, b in zip(_official_mb, _mine_mb))
    _naive_same = sum(a == b for a, b in zip(_official_mb, _naive_mb))
    # 公式 dataset クラスで、並べ替え後の file の各行が元の行と同じ tensor（advantage 以外）になることを確認
    DSV = OurSFTDataset(parquet_files=_vpath, tokenizer=TOKENIZER, config=_data_cfg)
    _order = emulate_official_order(DATASET_EXPECTED_ROWS, OFFICIAL_WORLD_SIZE, _n_check, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ)
    _tensor_bad = 0
    for _p in list(range(0, 512)) + list(range(DATASET_EXPECTED_ROWS - 256, DATASET_EXPECTED_ROWS)):   # 最初の 2 step と最後の 1 step
        _a, _b = DSV[_p], DS[_order[_p]]
        if not all(torch.equal(_a[k], _b[k]) for k in ("input_ids", "attention_mask", "position_ids", "loss_mask")):
            _tensor_bad += 1
        if float(_a["advantages"][0]) != TRAIN_VIEW_CHECK["advantage"]:
            _tensor_bad += 1
    del DSV
    VIEW_CHECK_OK = (_same_steps == len(_official_mb) == len(_mine_mb)) and _tensor_bad == 0
    print(f"\n=== 公式 {OFFICIAL_WORLD_SIZE} GPU の再現（GPU {_n_check} 台）: {'OK' if VIEW_CHECK_OK else 'NG'} ({time.time() - t0:.1f}s) ===")
    print(f"  学習用 file: {_vpath} (sha256 {_vsha[:12]})")
    print(f"  各 step の micro batch（4 行の組）が公式と一致した step: {_same_steps}/{len(_official_mb)}（並べ替えない場合: {_naive_same}/{len(_official_mb)}）")
    print(f"  公式 dataset クラスで作った tensor の一致（advantage 以外）と advantage={TRAIN_VIEW_CHECK['advantage']}: 不一致 {_tensor_bad} 行")
    print(f"  trainer.importance_sampling_mode=adv-only: loss = -{TRAIN_VIEW_CHECK['advantage']} × log_prob の token 平均。"
          f"{_n_check} GPU の勾配（micro batch 64 個の和 ÷ {_n_check}）× {TRAIN_VIEW_CHECK['advantage']} = 公式（64 個の和 ÷ {OFFICIAL_WORLD_SIZE}）")

LENGTH_RECORD = dict(max_total_len=MAX_TOTAL_LEN, auto_fit_max_length=AUTO_FIT_MAX_LENGTH, chosen_max_length=int(MAX_LENGTH), official_max_length=OFFICIAL_MAX_LENGTH,
                     rows_truncated_at_chosen=_n_trunc, prompt=dict(min=int(PROMPT_LENS.min()), mean=float(PROMPT_LENS.mean()), max=int(PROMPT_LENS.max())),
                     target_eos=dict(min=int(RESP_LENS.min()), mean=float(RESP_LENS.mean()), max=int(RESP_LENS.max())), mask_check_ok=MASK_CHECK_OK,
                     mask_scan_rows=len(_scan_rows), mask_failures=dict(_bad), ckpt_size_est_gb=CKPT_SIZE_EST_GB, tokenizer_eos=TOKENIZER.eos_token, tokenizer_pad=TOKENIZER.pad_token,
                     official_world_emulation=dict(n_gpus_checked=_n_check, view=TRAIN_VIEW_CHECK, view_check_ok=VIEW_CHECK_OK))
with open(f"{RECORD_DIR}/length_and_mask_record.json", "w") as f:
    json.dump(LENGTH_RECORD, f, indent=2, ensure_ascii=False)
del DS

## 5. ドライラン（既定の実行モード）

In [ ]:
%%ao
# @title 5. ドライラン: データ・設定・コマンド・想定 step 数・保存先を確認する（学習も HF 書き込みも開始しない）
import shutil, json, os


def plan_runs():
    """RUN_KIND に応じて実行予定の (lr, epochs, scheduler) を返す"""
    if RUN_KIND == "baseline":
        return [(OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], "cosine")]
    return [normalize_run(r) for r in SEARCH_RUN_LIST]


DRY_RUN_SPECS = []
if not plan_runs():
    print("実行予定の候補が無い（RUN_KIND='search' なら SEARCH_RUN_LIST を設定する）")
for _lr, _ep, _sched in plan_runs():
    _spec = build_run_spec(RUN_KIND, _lr, _ep, data_path=AO_PARQUET, data_sha256=AO_SHA256, n_rows=DATASET_EXPECTED_ROWS,
                           max_length=MAX_LENGTH, save_freq=SAVE_FREQ, n_gpus=max(N_GPUS, 1), lr_scheduler=_sched)
    DRY_RUN_SPECS.append(_spec)
    print("=" * 100)
    print_run_spec(_spec)
    _cmd = ["torchrun", "--nnodes=1", f"--nproc_per_node={_spec['n_gpus']}", "--node_rank=0", "--master_addr=127.0.0.1", "--master_port=<random>",
            "-m", _spec["module"]] + [f"{k}={v}" for k, v in _spec["overrides"].items()]
    print("  command:\n    " + " \\\n    ".join(_cmd))
    print("  env:", _spec["env"])
    _nfull = len(_spec["upload_full_steps"])
    _nlight = len(_spec["upload_steps"]) - _nfull
    print(f"  checkpoint: 保存 {len(_spec['expected_save_steps'])} 回、HF 転送 {len(_spec['upload_steps'])} 回（再開用 {_nfull} x {CKPT_SIZE_EST_GB:.1f} GB + "
          f"重みのみ {_nlight} x {CKPT_MODEL_ONLY_EST_GB:.1f} GB ≈ {hf_usage_gb(_nfull, _nlight):.0f} GB。HF PRO の非公開枠は 1 TB）、"
          f"ローカルは転送・検証済み {LOCAL_KEEP_LAST_N_VERIFIED_CKPTS} 個を保持")
    _local_exists = os.path.exists(_spec["overrides"]["trainer.default_local_dir"])
    _remote = hf_run_paths(HF_CKPT_REPO_ID, _spec["run_id"]) if HF_API else {}
    print(f"  衝突確認: local dir exists={_local_exists} | HF runs/{_spec['run_id']}: {sorted(_remote) if _remote else 'なし'}")
    if _local_exists or _remote:
        print("  *** 同名 run が既に存在する。上書きしない。再開なら『再開』セル、別 run なら RUN_ID_SUFFIX を設定 ***")
    with open(f"{RECORD_DIR}/dry_run_{_spec['run_id']}.json", "w") as f:
        json.dump(dict(spec=_spec, command=_cmd, local_exists=_local_exists, remote_steps=_remote), f, indent=2, ensure_ascii=False)

print("=" * 100)
print("torchrun:", shutil.which("torchrun"), "| GPUs:", N_GPUS, "| AO_DATA_READY:", AO_DATA_READY, "| MASK_CHECK_OK:", MASK_CHECK_OK, "| flash_attn:", FLASH_ATTN_OK)
print("ドライラン完了。学習・HF 書き込みは行っていない。RUN_MODE を 'trial' / 'train' にして先へ進む。")

## 6. 短い学習・HF 保存・再開の試走

6-a は起動・監視・HF 転送・再開の共通処理（セクション 7〜10 と『再開』でも使う）。6-b が試走本体。

In [ ]:
%%ao
# @title 6-a. 起動・監視・HF 転送・再開のヘルパー（追加処理はすべてここ。公式 trainer は subprocess で無変更のまま実行）
import subprocess, threading, time, json, os, re, glob, shutil, zipfile, hashlib, random, datetime, io, sys, signal, fnmatch
import numpy as np
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

HF_API = HfApi() if HF_LOGGED_IN else None


def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def require_training_env(n_gpus=None):
    """学習（試走・本学習・再開）を始める前の環境条件: GPU・flash-attn・1 GPU の FSDP2 勾配確認"""
    n_gpus = n_gpus or N_GPUS
    assert N_GPUS >= 1 and FLASH_ATTN_OK, "GPU と flash-attn が必要"
    if REQUIRE_FSDP2_GRAD_CHECK and n_gpus == 1:
        assert FSDP2_GRAD_CHECK.get("status") == "ok", (
            f"1 GPU の FSDP2 勾配確認（セクション 2）が OK ではない: {FSDP2_GRAD_CHECK.get('status')}。"
            "world size 1 の NCCL AVG で勾配が壊れる既知の報告があるため学習を開始しない")


def hash_file(path, with_sha256=True):
    """1 回の読み込みで git blob sha1（小さいファイルの照合用）と sha256（LFS 照合用）を計算する"""
    size = os.path.getsize(path)
    h1 = hashlib.sha1(f"blob {size}\0".encode())
    h2 = hashlib.sha256() if with_sha256 else None
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h1.update(b)
            if h2 is not None:
                h2.update(b)
    return h1.hexdigest(), (h2.hexdigest() if h2 is not None else None)


def git_blob_sha1(path):
    return hash_file(path, with_sha256=False)[0]


# ---- checkpoint の完全性 -----------------------------------------------------------------------
OPTIM_FILE_PATTERN = "optim_world_size_*"


def checkpoint_expected_files(world_size, content="full"):
    """content="model_only" は HF へ重みだけ送った checkpoint（optimizer 状態なし。分析・変換用で再開には使えない）"""
    files = []
    for r in range(world_size):
        files += [f"model_world_size_{world_size}_rank_{r}.pt", f"extra_state_world_size_{world_size}_rank_{r}.pt"]
        if content == "full":
            files.append(f"optim_world_size_{world_size}_rank_{r}.pt")
    files += ["fsdp_config.json", "data.pt", "huggingface/config.json", "huggingface/tokenizer_config.json"]
    return files


def checkpoint_is_complete(ckpt_dir, world_size, prev_sizes=None, require_stable=True, content="full"):
    """必要ファイルの存在・非空・torch.save(zip) の完全性・tracker・（連続 2 回のスキャンで）サイズ不変 を確認する。
    ディレクトリの存在や latest tracker だけでは完了と判断しない。"""
    step = int(re.search(r"global_step_(\d+)", ckpt_dir).group(1))
    sizes = {}
    for rel in checkpoint_expected_files(world_size, content):
        p = os.path.join(ckpt_dir, rel)
        if not os.path.isfile(p) or os.path.getsize(p) == 0:
            return False, sizes
        sizes[rel] = os.path.getsize(p)
        if rel.endswith(".pt") and not zipfile.is_zipfile(p):   # torch.save は zip 形式。末尾まで書けていれば central directory がある
            return False, sizes
    tokenizer_ok = any(os.path.isfile(os.path.join(ckpt_dir, "huggingface", f)) for f in ("tokenizer.json", "vocab.json", "tokenizer.model"))
    if not tokenizer_ok:
        return False, sizes
    tracker = os.path.join(os.path.dirname(ckpt_dir), "latest_checkpointed_iteration.txt")
    try:
        if int(open(tracker).read().strip()) < step:
            return False, sizes
    except Exception:
        return False, sizes
    for root, _, fs in os.walk(ckpt_dir):
        for f in fs:
            rel = os.path.relpath(os.path.join(root, f), ckpt_dir)
            sizes[rel] = os.path.getsize(os.path.join(root, f))
    if require_stable and prev_sizes != sizes:
        return False, sizes
    return True, sizes


def checkpoint_content_for(spec, step):
    """HF へ送る内容: 再開用 step（upload_full_steps）は optimizer 状態込み、それ以外は重みのみ。旧版の spec は全 step を full とする"""
    full = spec.get("upload_full_steps")
    return "full" if full is None or step in full else "model_only"


def build_manifest(spec, step, ckpt_dir, with_sha256=None, content="full"):
    with_sha256 = CKPT_MANIFEST_SHA256 if with_sha256 is None else with_sha256
    files = []
    for root, _, fs in os.walk(ckpt_dir):
        for f in sorted(fs):
            p = os.path.join(root, f)
            rel = os.path.relpath(p, ckpt_dir)
            if rel in ("ao_ckpt_manifest.json", "ao_upload_verified.json"):
                continue
            if content == "model_only" and fnmatch.fnmatch(rel, OPTIM_FILE_PATTERN):
                continue
            sha1, sha256 = hash_file(p, with_sha256=with_sha256)
            ent = dict(path=rel, size=os.path.getsize(p), git_blob_sha1=sha1)
            if sha256:
                ent["sha256"] = sha256
            files.append(ent)
    manifest = dict(run_id=spec["run_id"], step=step, world_size=spec["n_gpus"], content=content, files=files, total_bytes=sum(f["size"] for f in files),
                    run_config=spec, created_at=now_iso(), fork_commit=FORK_COMMIT, data_sha256=spec["data_sha256"], base_revision=spec["base_revision"])
    with open(os.path.join(ckpt_dir, "ao_ckpt_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    return manifest


# ---- HF 転送 -----------------------------------------------------------------------------------
def ensure_ckpt_repo():
    assert HF_API is not None, "HF にログインしていない（最初のセルで HF_TOKEN を設定）"
    assert HF_CKPT_REPO_ID, "HF_CKPT_REPO_ID が未設定"
    HF_API.create_repo(HF_CKPT_REPO_ID, private=HF_CKPT_PRIVATE, repo_type="model", exist_ok=True)


def hf_verify_uploaded(repo_id, path_in_repo, manifest, revision):
    """転送後に HF 側のサイズ・LFS sha256・git blob sha1 を manifest と照合する"""
    paths = [f"{path_in_repo}/{e['path']}" for e in manifest["files"]] + [f"{path_in_repo}/ao_ckpt_manifest.json"]
    infos = {i.path: i for i in HF_API.get_paths_info(repo_id, paths=paths, revision=revision, repo_type="model")}
    mismatches = []
    for e in manifest["files"]:
        rp = f"{path_in_repo}/{e['path']}"
        i = infos.get(rp)
        if i is None:
            mismatches.append((e["path"], "missing on HF")); continue
        if getattr(i, "size", None) != e["size"]:
            mismatches.append((e["path"], f"size {getattr(i, 'size', None)} != {e['size']}")); continue
        lfs = getattr(i, "lfs", None)
        if lfs is not None and e.get("sha256") and getattr(lfs, "sha256", None) and lfs.sha256 != e["sha256"]:
            mismatches.append((e["path"], "sha256 mismatch"))
        elif lfs is None and getattr(i, "blob_id", None) and i.blob_id != e["git_blob_sha1"]:
            mismatches.append((e["path"], "git blob sha1 mismatch"))
    if f"{path_in_repo}/ao_ckpt_manifest.json" not in infos:
        mismatches.append(("ao_ckpt_manifest.json", "missing on HF"))
    return mismatches


def upload_checkpoint(spec, step, ckpt_dir, log=print):
    """保存完了した checkpoint を runs/<run_id>/global_step_<step>/ へ転送し、検証後に完了マーカーを置く。既存の完了済み step は上書きしない"""
    t0 = time.time()
    ensure_ckpt_repo()
    pir = f"runs/{spec['run_id']}/global_step_{step}"
    remote = hf_run_paths(HF_CKPT_REPO_ID, spec["run_id"]).get(step)
    if remote and remote["complete"]:
        log(f"collision: {pir} は HF に完了済みで存在する。上書きしない")
        return dict(status="collision_remote_complete", path_in_repo=pir)
    if remote:
        log(f"{pir} に未完了の転送が残っている。同じ step を再転送する")
    content = checkpoint_content_for(spec, step)
    manifest = build_manifest(spec, step, ckpt_dir, content=content)
    ignore = ["ao_upload_verified.json"] + ([OPTIM_FILE_PATTERN] if content == "model_only" else [])
    last_err = None
    for attempt in range(5):
        try:
            info = HF_API.upload_folder(folder_path=ckpt_dir, path_in_repo=pir, repo_id=HF_CKPT_REPO_ID, repo_type="model",
                                        commit_message=f"{spec['run_id']} global_step_{step} ({content})", ignore_patterns=ignore)
            break
        except Exception as e:  # noqa
            last_err = e
            wait = 2 ** attempt * 5
            log(f"upload attempt {attempt + 1} failed: {type(e).__name__}: {str(e)[:200]} -> retry in {wait}s")
            time.sleep(wait)
    else:
        return dict(status="upload_failed", error=repr(last_err)[:500], path_in_repo=pir, duration_sec=time.time() - t0)
    mism = hf_verify_uploaded(HF_CKPT_REPO_ID, pir, manifest, info.oid)
    if mism:
        log(f"verify failed for {pir}: {mism[:5]}")
        return dict(status="verify_failed", commit=info.oid, mismatches=mism, path_in_repo=pir, duration_sec=time.time() - t0)
    marker = dict(run_id=spec["run_id"], step=step, content=content, folder_commit=info.oid, verified_at=now_iso(), n_files=len(manifest["files"]),
                  total_bytes=manifest["total_bytes"], world_size=spec["n_gpus"], total_steps=spec["total_steps"])
    with open(os.path.join(ckpt_dir, "ao_upload_verified.json"), "w") as f:
        json.dump(marker, f, indent=2)
    info2 = HF_API.upload_file(path_or_fileobj=os.path.join(ckpt_dir, "ao_upload_verified.json"), path_in_repo=f"{pir}/ao_upload_verified.json",
                               repo_id=HF_CKPT_REPO_ID, repo_type="model", commit_message=f"{spec['run_id']} global_step_{step} verified")
    dur = time.time() - t0
    log(f"uploaded+verified {pir} [{content}] ({manifest['total_bytes'] / 1e9:.2f} GB, {dur:.0f}s) commit={info.oid[:10]} marker={info2.oid[:10]}")
    return dict(status="verified", content=content, commit=info.oid, marker_commit=info2.oid, path_in_repo=pir, total_bytes=manifest["total_bytes"], duration_sec=dur,
                url=f"https://huggingface.co/{HF_CKPT_REPO_ID}/tree/{info2.oid}/{pir}")


# ---- 監視スレッド ------------------------------------------------------------------------------
class TrainingMonitor(threading.Thread):
    """学習プロセスと並行して checkpoint ディレクトリを監視し、保存完了ごとに HF へ転送・検証する。GPU メモリも記録する。
    kill_after_step が指定された場合（試走）: その step の保存完了を検知したら学習プロセス群を SIGSTOP で一時停止し、
    HF 転送・検証が終わってから SIGTERM で終了する（保存 → HF 転送 → プロセス終了 の順を保証する）。"""

    def __init__(self, proc, spec, kill_after_step=None, upload=True, poll_sec=10, keep_local=None, stable_scans=True):
        super().__init__(daemon=True)
        self.proc, self.spec, self.kill_after_step, self.upload, self.poll_sec = proc, spec, kill_after_step, upload, poll_sec
        self.keep_local = LOCAL_KEEP_LAST_N_VERIFIED_CKPTS if keep_local is None else keep_local
        self.stable_scans = stable_scans   # True: 連続 2 回のスキャンでサイズ不変を要求（通常）。試走では tracker + zip 検証のみで即時判定
        self.ckpt_root = spec["overrides"]["trainer.default_local_dir"]
        self.state = dict(uploaded={}, failed={}, complete_local=[], gpu_mem_max_mb={}, events=[], deleted_local=[], skipped=[])
        self._sizes, self._attempts, self.killed, self.paused = {}, {}, False, False

    def _signal_group(self, sig):
        try:
            os.killpg(os.getpgid(self.proc.pid), sig)
            return True
        except ProcessLookupError:
            return False

    def _terminate(self):
        alive = self.proc.poll() is None
        if not alive:
            self.log("プロセスは既に終了していた（kill は不要）")
            return False
        if self.paused:
            self._signal_group(signal.SIGCONT)
            self.paused = False
        self._signal_group(signal.SIGTERM)
        try:
            self.proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            self._signal_group(signal.SIGKILL)
        self.killed = True
        return True

    def log(self, msg):
        line = f"[monitor {time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True)
        self.state["events"].append(dict(t=time.time(), msg=msg))

    def _sample_gpu(self):
        if shutil.which("nvidia-smi") is None:
            return
        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True)
        for line in r.stdout.splitlines():
            try:
                i, used = [x.strip() for x in line.split(",")]
                self.state["gpu_mem_max_mb"][i] = max(self.state["gpu_mem_max_mb"].get(i, 0), int(float(used)))
            except ValueError:
                pass

    def _cleanup_local(self):
        verified = sorted(int(s) for s in self.state["uploaded"] if self.state["uploaded"][s]["status"] == "verified")
        resume_src = self.spec.get("resume_from_path")
        for s in verified[: max(0, len(verified) - self.keep_local)]:
            d = os.path.join(self.ckpt_root, f"global_step_{s}")
            if resume_src and os.path.abspath(d) == os.path.abspath(resume_src):
                continue   # 再開元は消さない
            if os.path.isdir(d):
                shutil.rmtree(d, ignore_errors=True)
                self.state["deleted_local"].append(s)
                self.log(f"local cleanup: removed global_step_{s} (HF 転送・検証済み)")

    def _scan(self, final=False):
        dirs = glob.glob(os.path.join(self.ckpt_root, "global_step_*"))
        steps = sorted(int(re.search(r"global_step_(\d+)$", d).group(1)) for d in dirs if re.search(r"global_step_(\d+)$", d))
        for step in steps:
            if step in self.state["uploaded"]:
                continue
            d = os.path.join(self.ckpt_root, f"global_step_{step}")
            marker = os.path.join(d, "ao_upload_verified.json")
            if os.path.isfile(marker):   # 以前の実行で転送・検証済み（再開のために HF から取得したもの）。再転送しない
                try:
                    self.state["uploaded"][step] = dict(status="verified", previously_verified=True, **json.load(open(marker)))
                    self.log(f"global_step_{step} は転送・検証済み（マーカーあり）。スキップ")
                except Exception:
                    pass
                continue
            complete, sizes = checkpoint_is_complete(d, self.spec["n_gpus"], self._sizes.get(step), require_stable=(self.stable_scans and not final))
            self._sizes[step] = sizes
            if not complete:
                continue
            if step not in self.state["complete_local"]:
                self.state["complete_local"].append(step)
                self.log(f"checkpoint global_step_{step} complete locally ({sum(sizes.values()) / 1e9:.2f} GB, {len(sizes)} files)")
            kill_now = self.kill_after_step is not None and step >= self.kill_after_step and not self.killed
            upload_steps = self.spec.get("upload_steps")
            if upload_steps is not None and step not in upload_steps and step not in self.state["skipped"]:
                # 転送対象外の step（公開 CoT 学生と同じ step でも再開用の間隔でもない）。HF に無いので再開候補にならず、ローカルからも消す
                self.state["skipped"].append(step)
                shutil.rmtree(d, ignore_errors=True)
                self.log(f"global_step_{step} は転送対象外（HF_UPLOAD_STEPS={self.spec.get('upload_policy')}）。ローカルから削除")
                continue
            if step in self.state["skipped"]:
                continue
            if kill_now and not self.paused and self.proc.poll() is None:
                self.paused = self._signal_group(signal.SIGSTOP)   # 転送中に学習が先へ進まないよう一時停止（切断前の状態を固定）
                self.log(f"kill_after_step={self.kill_after_step}: global_step_{step} の保存完了を検知。学習プロセスを一時停止して HF 転送する")
            if self.upload:
                if self._attempts.get(step, 0) >= 3:
                    continue
                self._attempts[step] = self._attempts.get(step, 0) + 1
                res = upload_checkpoint(self.spec, step, d, log=self.log)
                if res["status"] == "verified":
                    self.state["uploaded"][step] = res
                    self.state["failed"].pop(step, None)
                    self._cleanup_local()
                elif res["status"].startswith("collision"):
                    # 同じ step が HF に完了済み（例: 再開元）。上書きしない。転送済みとして扱う
                    self.state["uploaded"][step] = dict(res, status="verified", previously_verified=True)
                    self._attempts[step] = 99
                else:
                    self.state["failed"][step] = res
            if kill_now and (not self.upload or step in self.state["uploaded"]):
                self.log(f"global_step_{step} の{'HF 転送・検証' if self.upload else '保存'}を確認したのでプロセスを終了する（切断の模擬）")
                self._terminate()

    def run(self):
        while True:
            alive = self.proc.poll() is None
            self._sample_gpu()
            try:
                self._scan(final=not alive)
            except Exception as e:  # noqa
                self.log(f"monitor error: {type(e).__name__}: {str(e)[:300]}")
                if self.paused:   # 転送に失敗しても停止したまま放置しない
                    self._terminate()
            if not alive:
                try:
                    self._scan(final=True)
                except Exception as e:  # noqa
                    self.log(f"monitor error: {type(e).__name__}: {str(e)[:300]}")
                break
            time.sleep(self.poll_sec)


# ---- 起動 --------------------------------------------------------------------------------------
_METRIC_RE = re.compile(r"(?:^|\s)step:(\d+)((?: - [\w/]+:\S+)+)")


def parse_metric_line(line):
    """console logger の `step:N - train/loss:0.5 - train/lr:1e-05 ...` を辞書にする（nan/inf も float として保持）"""
    m = _METRIC_RE.search(line)
    if not m:
        return None
    d = dict(step=int(m.group(1)))
    for kv in m.group(2).split(" - ")[1:]:
        k, v = kv.split(":", 1)
        try:
            d[k] = float(v)
        except ValueError:
            pass
    return d


_INTERESTING = ("Saving checkpoint", "Updated checkpoint tracker", "Loaded", "loaded", "resume", "Traceback", "Error", "error", "OutOfMemory", "Total training steps",
                "Number of steps/epoch", "Normalize batch", "Using FSDP rank", "WARN", "Killed", "Signal", "Found checkpoint", "grad_norm is not finite")


def launch_training(spec, kill_after_step=None, upload=None, keep_local=None, quiet=False, poll_sec=10, stable_scans=True):
    """公式エントリーポイントを torchrun (python -m torch.distributed.run) で起動し、ログを流しつつ監視スレッドで checkpoint を HF へ転送する。終了まで戻らない。"""
    upload = HF_UPLOAD_CHECKPOINTS if upload is None else upload
    if upload:
        ensure_ckpt_repo()
    run_id = spec["run_id"]
    run_log_dir = f"{LOG_DIR}/{run_id}"
    os.makedirs(run_log_dir, exist_ok=True)
    ts = time.strftime("%Y%m%d-%H%M%S")
    log_path = f"{run_log_dir}/{ts}.log"
    with open(f"{run_log_dir}/{ts}.spec.json", "w") as f:
        json.dump(spec, f, indent=2, ensure_ascii=False)
    port = random.randint(20000, 21000)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nnodes=1", f"--nproc_per_node={spec['n_gpus']}", "--node_rank=0", "--master_addr=127.0.0.1",
           f"--master_port={port}", "-m", spec["module"]] + [f"{k}={v}" for k, v in spec["overrides"].items()]
    env = os.environ.copy()
    env.update(spec["env"])
    env["PYTHONPATH"] = REPO_DIR + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    with open(f"{run_log_dir}/{ts}.cmd.txt", "w") as f:
        f.write(" \\\n  ".join(cmd) + "\n\nenv: " + json.dumps(spec["env"]) + "\n")
    print(f"launch {run_id}: log={log_path}")
    print("  " + " ".join(cmd[:10]) + " ...")
    t_launch = time.time()
    # start_new_session=True: torchrun とワーカーを 1 つのプロセスグループにし、試走の一時停止/終了をグループごと送れるようにする
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace",
                            start_new_session=True)
    with open(f"{WORK_DIR}/ao-trainer.pgid", "w") as f:   # セル 1 を再実行したとき、残った学習プロセスをこの process group ごと終了する
        f.write(str(proc.pid))
    mon = TrainingMonitor(proc, spec, kill_after_step=kill_after_step, upload=upload, keep_local=keep_local, poll_sec=poll_sec, stable_scans=stable_scans)
    mon.start()
    metrics, events = [], []
    try:
        with open(log_path, "w") as lf:
            for line in proc.stdout:
                t = time.time()
                lf.write(f"[{t - t_launch:9.1f}s] {line}")
                m = parse_metric_line(line)
                if m:
                    m["t"] = t
                    _scale = float(spec.get("logged_loss_scale", 1.0))
                    if "train/loss" in m:
                        m["train/loss_official_scale"] = m["train/loss"] / _scale   # adv-only で N/8 倍されたログを公式スケールへ戻した値
                    metrics.append(m)
                    if not quiet:
                        print(f"[{t - t_launch:7.0f}s] step {m['step']}/{spec['total_steps']} loss={m.get('train/loss_official_scale', float('nan')):.4f}"
                              f"{'' if _scale == 1.0 else f' (logged {_scale:g}x: ' + format(m.get('train/loss', float('nan')), '.4f') + ')'} "
                              f"lr={m.get('train/lr', float('nan')):.3e} grad_norm={m.get('train/grad_norm', float('nan')):.3f} entropy={m.get('train/entropy/mean', float('nan')):.3f}", flush=True)
                elif any(k in line for k in _INTERESTING) and "ratio_list_all" not in line:
                    events.append(dict(t=t, line=line.rstrip()[:300]))
                    if not quiet:
                        print(f"[{t - t_launch:7.0f}s] {line.rstrip()[:200]}", flush=True)
    except KeyboardInterrupt:
        print("KeyboardInterrupt -> terminating training process group")
        mon._terminate()
    rc = proc.wait()
    try:
        mon.join(timeout=3600 * 6)   # プロセス終了後も、最後の checkpoint の転送が終わるまで待つ
    except KeyboardInterrupt:
        print("KeyboardInterrupt during final upload -> 結果は転送未完了のまま記録する")
    t_end = time.time()
    result = dict(run_id=run_id, spec=spec, exit_code=rc, killed_by_monitor=mon.killed, log_path=log_path, launched_at=t_launch, ended_at=t_end,
                  wall_sec=t_end - t_launch, metrics=metrics, events=events, monitor=dict(uploaded=mon.state["uploaded"], failed=mon.state["failed"],
                  complete_local=mon.state["complete_local"], gpu_mem_max_mb=mon.state["gpu_mem_max_mb"], deleted_local=mon.state["deleted_local"], skipped=mon.state["skipped"],
                  events=mon.state["events"]))
    with open(f"{run_log_dir}/{ts}.result.json", "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False, default=str)
    status = "killed(simulated disconnect)" if mon.killed else ("OK" if rc == 0 else f"FAILED rc={rc}")
    print(f"finished {run_id}: {status} | steps logged={len(metrics)} | wall={t_end - t_launch:.0f}s | GPU mem max MB={mon.state['gpu_mem_max_mb']} "
          f"| uploaded={sorted(mon.state['uploaded'])} | failed={ {k: v['status'] for k, v in mon.state['failed'].items()} }")
    if mon.state["failed"]:
        print("*** HF 転送に失敗した checkpoint がある。ローカルには残している。ネットワーク復旧後に upload_checkpoint() を再実行する ***")
    return result


def summarize_timing(result):
    """初期化時間・optimizer step 時間（勾配蓄積込み）・保存時間・転送時間を分けて集計する"""
    ms = result["metrics"]
    spec = result["spec"]
    out = dict(n_steps=len(ms))
    if not ms:
        return out
    saves = set(spec["expected_save_steps"])
    intervals = [(ms[i]["step"], ms[i]["t"] - ms[i - 1]["t"]) for i in range(1, len(ms))]
    plain = [d for s, d in intervals if (s - 1) not in saves]      # 直前 step で保存が走った区間を除く
    out["step_sec_median"] = float(np.median(plain)) if plain else None
    out["step_sec_all"] = [round(d, 2) for _, d in intervals]
    first_step_t = ms[0]["t"]
    out["init_sec_incl_first_step"] = first_step_t - result["launched_at"]
    out["init_sec_est"] = (first_step_t - result["launched_at"] - out["step_sec_median"]) if out["step_sec_median"] else None
    ev = result["events"]
    save_starts = [e["t"] for e in ev if "Saving checkpoint to" in e["line"]]
    save_ends = [e["t"] for e in ev if "Updated checkpoint tracker" in e["line"]]
    out["save_sec"] = [round(b - a, 1) for a, b in zip(save_starts, save_ends)]
    out["upload_sec"] = {s: round(v.get("duration_sec", 0), 1) for s, v in result["monitor"]["uploaded"].items()}
    out["gpu_mem_max_mb"] = result["monitor"]["gpu_mem_max_mb"]
    return out


# ---- HF からの再開 ------------------------------------------------------------------------------
def list_hf_checkpoints(run_id):
    """HF 上の run の checkpoint 一覧。完了マーカーのあるものだけを再開候補にする"""
    paths = hf_run_paths(HF_CKPT_REPO_ID, run_id)
    rows = []
    for step in sorted(paths):
        row = dict(step=step, complete=paths[step]["complete"], n_files=len(paths[step]["files"]))
        if row["complete"]:
            try:
                mp = hf_hub_download(HF_CKPT_REPO_ID, f"runs/{run_id}/global_step_{step}/ao_upload_verified.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
                row.update(json.load(open(mp)))
            except Exception as e:  # noqa
                row["complete"] = False
                row["marker_error"] = str(e)[:200]
        row.setdefault("content", "full")   # 旧版のマーカーには content が無い（全ファイルを送っていた）
        row["resumable"] = row["complete"] and row["content"] == "full"
        rows.append(row)
    return rows


def download_hf_checkpoint(run_id, step, revision=None):
    """HF から checkpoint を取得し manifest で検証する。戻り値: (local_path, manifest, revision)"""
    pir = f"runs/{run_id}/global_step_{step}"
    marker_path = hf_hub_download(HF_CKPT_REPO_ID, f"{pir}/ao_upload_verified.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
    marker = json.load(open(marker_path))
    revision = revision or marker["folder_commit"]
    local = f"{CKPT_DIR}/{run_id}/global_step_{step}"
    tmp = f"{WORK_DIR}/hf_download/{run_id}_{step}"
    shutil.rmtree(tmp, ignore_errors=True)
    snapshot_download(HF_CKPT_REPO_ID, repo_type="model", revision=revision, allow_patterns=[f"{pir}/*", f"{pir}/**"], local_dir=tmp)
    src = os.path.join(tmp, pir)
    if os.path.isdir(local):
        shutil.rmtree(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    shutil.move(src, local)
    shutil.copy(marker_path, os.path.join(local, "ao_upload_verified.json"))   # マーカーは folder_commit より後の commit なので別途置く（監視が再転送しないための目印）
    manifest = json.load(open(os.path.join(local, "ao_ckpt_manifest.json")))
    bad = []
    for e in manifest["files"]:
        p = os.path.join(local, e["path"])
        if not os.path.isfile(p) or os.path.getsize(p) != e["size"]:
            bad.append((e["path"], "missing/size")); continue
        if e.get("sha256") and sha256_of(p) != e["sha256"]:
            bad.append((e["path"], "sha256"))
    if bad:
        raise RuntimeError(f"取得した checkpoint が manifest と一致しない: {bad[:5]}")
    # 公式の latest tracker も揃えておく（resume_path モードでは使われないが、ディレクトリ構成を学習時と同じにする）。
    # 既存の値より大きいときだけ書く（小さい値にすると、ローカルに残した新しい step の checkpoint が「未完了」と判定される）
    _tracker = os.path.join(os.path.dirname(local), "latest_checkpointed_iteration.txt")
    try:
        _prev = int(open(_tracker).read().strip())
    except (OSError, ValueError):
        _prev = -1
    if step > _prev:
        with open(_tracker, "w") as f:
            f.write(str(step))
    ok, _ = checkpoint_is_complete(local, manifest["world_size"], require_stable=False, content=manifest.get("content", "full"))
    assert ok, "必要ファイル（model/optim/extra の各 rank shard, data.pt, fsdp_config.json, huggingface/）が揃っていない"
    manifest["folder_commit"] = marker.get("folder_commit")
    print(f"downloaded {pir} [{manifest.get('content', 'full')}] @ {revision[:10]} -> {local} ({manifest['total_bytes'] / 1e9:.2f} GB, verified against manifest)")
    return local, manifest, revision


def current_data_sha256_for(saved_data_path):
    """保存時の data_path に対応する現在のファイルの sha256（本学習は AO parquet、試走は subset parquet）"""
    if os.path.basename(saved_data_path) == os.path.basename(AO_PARQUET):
        return AO_SHA256, AO_PARQUET
    if os.path.isfile(saved_data_path):
        return sha256_of(saved_data_path), saved_data_path
    return None, None


def saved_training_view(saved, world_size):
    """保存時の run 設定にある training_view。旧版（公式 8 GPU 再現を入れる前）の checkpoint には無いので、並べ替えなし・vanilla とみなす"""
    if saved.get("training_view"):
        return saved["training_view"]
    return dict(enabled=False, official_world=OFFICIAL_WORLD_SIZE, actual_world=int(world_size), advantage=1.0,
                importance_sampling_mode=saved["overrides"].get("trainer.importance_sampling_mode", "vanilla"), order_sha256=None)


def check_resume_compatibility(manifest, n_gpus=None):
    """コード・データ・tokenizer・学習設定・GPU 台数/FSDP 構成の互換性を確認し、問題を列挙する"""
    n_gpus = n_gpus or N_GPUS
    saved = manifest["run_config"]
    problems, warnings_ = [], []
    if manifest["world_size"] != n_gpus:
        problems.append(f"FSDP world_size {manifest['world_size']} != 現在の GPU 台数 {n_gpus}（sharded checkpoint は台数依存。条件を黙って変えない）")
    if saved["base_revision"] != MODEL_INFO["base_revision"] or saved["model_key"] != MODEL_KEY:
        problems.append(f"Base モデル/revision が異なる: saved={saved['model_key']}@{saved['base_revision'][:8]} now={MODEL_KEY}@{MODEL_INFO['base_revision'][:8]}")
    cur_sha, cur_path = current_data_sha256_for(saved["data_path"])
    if cur_sha is None:
        problems.append(f"学習データ {saved['data_path']} が現在の環境に無い（セクション 3 / 試走セルで再作成する）")
    elif saved["data_sha256"] != cur_sha:
        problems.append(f"学習データの sha256 が異なる: saved={saved['data_sha256'][:12]} now={cur_sha[:12]} ({cur_path})")
    if saved["target_style"] != AO_TARGET_STYLE:
        problems.append("AO_TARGET_STYLE が異なる")
    # 公式 8 GPU 再現用の学習 file（並べ替えと advantage）が保存時と同じになるか
    sv = saved_training_view(saved, manifest["world_size"])
    if not saved.get("training_view"):
        warnings_.append("保存時の設定に training_view が無い（公式 8 GPU 再現を入れる前の版の checkpoint）。並べ替えなし・vanilla として扱う。"
                         "再開するには EMULATE_OFFICIAL_WORLD_SIZE=False にする（その run は公式 8 GPU の条件と異なる）")
    if cur_sha is not None and saved["data_sha256"] == cur_sha:
        _, tsha, tv = prepare_train_file(cur_path, cur_sha, manifest["world_size"])
        keys = ("enabled", "official_world", "actual_world", "advantage", "importance_sampling_mode", "order_sha256")
        diff = {k: (sv.get(k), tv.get(k)) for k in keys if sv.get(k) != tv.get(k)}
        if diff:
            problems.append(f"公式 8 GPU 再現の設定が保存時と異なる（EMULATE_OFFICIAL_WORLD_SIZE / GPU 台数を保存時に合わせる）: {diff}")
        elif sv.get("enabled") and tsha != saved.get("train_file_sha256"):
            warnings_.append("学習用 file の sha256 が保存時と異なる（行順・advantage は同一。parquet の書き込みメタデータの差の可能性）")
    if saved.get("fork_commit") != FORK_COMMIT:
        if OFFICIAL_CODE_UNCHANGED is True:
            warnings_.append(f"Fork commit が異なる: saved={str(saved.get('fork_commit'))[:10]} now={FORK_COMMIT[:10]}（verl/・training_scripts/ は参照 commit と同一なので続行可）")
        else:
            problems.append(f"Fork commit が異なり、公式コードの無変更を確認できない (OFFICIAL_CODE_UNCHANGED={OFFICIAL_CODE_UNCHANGED})")
    if saved["kind"] != "trial" and "trainer.total_training_steps" in saved["overrides"]:
        problems.append("保存時の設定に step 上限 (trainer.total_training_steps) がある。本学習の checkpoint ではない")
    if saved.get("max_length") != int(MAX_LENGTH):
        warnings_.append(f"現在の MAX_LENGTH {MAX_LENGTH} と保存時 {saved.get('max_length')} が異なる。再開は保存時の設定を使う")
    if not FLASH_ATTN_OK and n_gpus > 0:
        problems.append("flash-attn が使えない")
    if manifest.get("content", "full") != "full":
        problems.append("この checkpoint は重みのみ（optimizer 状態なし）。再開には使えない")
    if saved.get("gpu_profile", "official") != GPU_PROFILE:
        problems.append(f"GPU プロファイル（torch の版）が保存時と異なる: saved={saved.get('gpu_profile', 'official')} now={GPU_PROFILE}。"
                        "同じ種類の GPU（A100 なら A100）で再開する")
    return problems, warnings_


def build_resume_spec(manifest, local_ckpt_path):
    """保存時の run 設定をそのまま使い、resume 引数だけを付けた spec を作る（総 epoch 数・総 step 数を維持）"""
    saved = manifest["run_config"]
    spec = json.loads(json.dumps(saved))
    ov = spec["overrides"]
    ov["trainer.resume_mode"] = "resume_path"
    ov["trainer.resume_from_path"] = local_ckpt_path
    ov["model.partial_pretrain"] = get_base_model_local()          # 同じ revision を再取得（パスは環境で変わり得る）
    cur_sha, cur_path = current_data_sha256_for(saved["data_path"])
    assert cur_sha == saved["data_sha256"], "学習データが保存時と一致しない"
    train_path, train_sha, view = prepare_train_file(cur_path, cur_sha, manifest["world_size"])   # 保存時と同じ並べ替え・advantage を再作成
    sv = saved_training_view(saved, manifest["world_size"])
    assert view.get("enabled") == sv.get("enabled") and view.get("order_sha256") == sv.get("order_sha256") and view.get("advantage") == sv.get("advantage"), \
        "公式 8 GPU 再現の学習 file が保存時と一致しない"
    ov["data.train_files"] = train_path
    ov["data.val_files"] = ov["data.train_files"]
    ov["trainer.default_local_dir"] = f"{CKPT_DIR}/{spec['run_id']}"
    spec["base_local"] = ov["model.partial_pretrain"]
    spec["data_path"] = cur_path
    spec["train_file"], spec["train_file_sha256"] = train_path, train_sha
    spec["training_view"] = dict(view)
    spec.setdefault("logged_loss_scale", view.get("logged_loss_scale", 1.0))
    spec.setdefault("upload_steps", list(saved.get("expected_save_steps", [])))   # 旧版の spec は保存した全 step を転送していた
    spec.setdefault("upload_policy", "all")
    spec.setdefault("official_world_size", OFFICIAL_WORLD_SIZE)
    spec.setdefault("residual_differences", residual_differences(manifest["world_size"], saved.get("max_length", OFFICIAL_MAX_LENGTH), view.get("enabled", False)))
    spec["resume_from_path"] = local_ckpt_path
    spec["resumed_from"] = dict(step=manifest["step"], hf_repo=HF_CKPT_REPO_ID, folder_commit=manifest.get("folder_commit"))
    spec["fork_commit_at_resume"] = FORK_COMMIT
    spec["env"]["VERL_SFT_LOGGING_LEVEL"] = "INFO"
    assert ov["trainer.total_epochs"] == str(saved["epochs"]) and spec["total_steps"] == saved["total_steps"], "総 epoch 数・総 step 数は保存時の値を維持する"
    return spec


def resume_run_from_hf(run_id, step="latest", kill_after_step=None, upload=None):
    """HF の完了済み checkpoint から新プロセスで再開する。未保存 step は再実行になる"""
    require_training_env(None)
    rows = list_hf_checkpoints(run_id)
    complete = [r for r in rows if r["resumable"]]   # optimizer 状態込みで転送・検証済みのものだけが再開候補
    print(f"HF checkpoints for {run_id}:")
    for r in rows:
        print(f"  step {r['step']:6d} complete={r['complete']} content={r['content']} resumable={r['resumable']} files={r['n_files']} "
              f"commit={str(r.get('folder_commit', ''))[:10]} verified_at={r.get('verified_at', '')}")
    if not complete:
        raise SystemExit("再開できる checkpoint（optimizer 状態込みで転送・検証済み）が無い。不完全な転送や重みだけの step は候補にしない")
    if step == "latest":
        chosen = max(r["step"] for r in complete)
    else:
        chosen = int(step)
        assert chosen in [r["step"] for r in complete], f"step {chosen} は再開できる checkpoint ではない（未完了、または重みのみ）"
        newer = [r["step"] for r in complete if r["step"] > chosen]
        if newer:
            raise SystemExit(f"step {chosen} より新しい完了済み step {newer} がある。古い step から再開すると新しい step を上書きする恐れがあるので拒否する。"
                             "別 run として続ける場合は run_id を変えて手動で設定する")
    local, manifest, rev = download_hf_checkpoint(run_id, chosen)
    problems, warns = check_resume_compatibility(manifest)
    for w in warns:
        print("WARN:", w)
    if problems:
        for p in problems:
            print("PROBLEM:", p)
        raise SystemExit("再開条件を満たさない。条件を黙って変更しない")
    spec = build_resume_spec(manifest, local)
    total = spec["total_steps"]
    print(f"resume {run_id} from global_step_{chosen} (HF revision {rev[:10]}). 総 step {total} は維持。step {chosen + 1}..{total} を再実行（最後の保存以降の未保存 step は再実行になる）")
    print_run_spec(spec)
    return launch_training(spec, kill_after_step=kill_after_step, upload=upload)


# ---- FSDP checkpoint → HF 形式（公式 verl.model_merger をそのまま使う） ---------------------------
def merge_checkpoint(run_id, step, local_ckpt=None):
    """python -m verl.model_merger merge --backend fsdp（README の手順）で HF 形式へ変換する。戻り値: 変換先ディレクトリ"""
    ckpt = local_ckpt or f"{CKPT_DIR}/{run_id}/global_step_{step}"
    ok = False

    def _content(d):
        mp = os.path.join(d, "ao_ckpt_manifest.json")
        return json.load(open(mp)).get("content", "full") if os.path.isfile(mp) else "full"

    if os.path.isfile(os.path.join(ckpt, "fsdp_config.json")):
        world = json.load(open(os.path.join(ckpt, "fsdp_config.json")))["world_size"]
        ok, _ = checkpoint_is_complete(ckpt, world, require_stable=False, content=_content(ckpt))
    if not ok:   # ローカルに無い、または書きかけ → HF の完了済み checkpoint（重みのみでも変換はできる）を取得する
        ckpt, _, _ = download_hf_checkpoint(run_id, step)
        world = json.load(open(os.path.join(ckpt, "fsdp_config.json")))["world_size"]
        ok, _ = checkpoint_is_complete(ckpt, world, require_stable=False, content=_content(ckpt))
    assert ok, f"checkpoint が不完全: {ckpt}"
    target = f"{WORK_DIR}/merged/{run_id}/merged_step{step}"
    if os.path.isfile(os.path.join(target, "config.json")) and glob.glob(os.path.join(target, "*.safetensors")):
        print("merged model exists:", target)
        return target
    os.makedirs(target, exist_ok=True)
    cmd = [sys.executable, "-m", "verl.model_merger", "merge", "--backend", "fsdp", "--local_dir", ckpt, "--target_dir", target, "--trust_remote_code"]
    env = dict(os.environ, PYTHONPATH=REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", ""))
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-4000:])
        raise RuntimeError("verl.model_merger failed")
    assert os.path.isfile(os.path.join(target, "config.json")) and glob.glob(os.path.join(target, "*.safetensors")), "変換結果に config/safetensors が無い"
    assert any(os.path.isfile(os.path.join(target, f)) for f in ("tokenizer.json", "tokenizer_config.json")), "変換結果に tokenizer が無い"
    print(f"merged {ckpt} -> {target} ({time.time() - t0:.0f}s)")
    return target


def get_manifest(run_id, step):
    """checkpoint の manifest（保存時の run 設定を含む）。ローカルに無ければ HF から manifest だけ取得する"""
    local = f"{CKPT_DIR}/{run_id}/global_step_{step}/ao_ckpt_manifest.json"
    if os.path.isfile(local):
        return json.load(open(local))
    p = hf_hub_download(HF_CKPT_REPO_ID, f"runs/{run_id}/global_step_{step}/ao_ckpt_manifest.json", repo_type="model", local_dir=f"{WORK_DIR}/hf_markers")
    return json.load(open(p))


# ---- 実験記録 ----------------------------------------------------------------------------------
def write_experiment_record(spec, results, evaluations=None, final_model=None, unverified=None, upload=True):
    rec = dict(
        run_id=spec["run_id"], kind=spec["kind"], written_at=now_iso(), session_started_at=SESSION_STARTED_AT,
        models=dict(teacher=TEACHER, base=dict(repo=spec["base_repo"], revision=spec["base_revision"]),
                    cot_student=dict(repo=spec["cot_repo"], revision=spec["cot_revision"], subfolder=spec["cot_subfolder"])),
        data=dict(repo=spec["dataset_repo"], revision=spec["dataset_revision"], raw_sha256=DATASET_EXPECTED_SHA256, ao_parquet_sha256=spec["data_sha256"],
                  rows=spec["n_rows"], target_style=spec["target_style"], ao_audit=AO_AUDIT_SUMMARY, prompt_note="prompt は公式と同一（step by step の指示を含む）。target は最後の \\boxed{} のみ"),
        length_and_mask=LENGTH_RECORD, training=dict(overrides=spec["overrides"], changes_vs_official=spec["changes_vs_official"], env=spec["env"],
                                                     total_steps=spec["total_steps"], steps_per_epoch=spec["steps_per_epoch"], warmup_steps=spec["warmup_steps"],
                                                     save_freq=spec["save_freq"], official_script=spec["official_script"], n_gpus=spec["n_gpus"],
                                                     official_world_size=spec.get("official_world_size"), training_view=spec.get("training_view"),
                                                     train_file=spec.get("train_file"), train_file_sha256=spec.get("train_file_sha256"),
                                                     logged_loss_scale=spec.get("logged_loss_scale", 1.0),
                                                     residual_differences_vs_official=spec.get("residual_differences")),
        code=dict(fork_repo=FORK_REPO_URL, fork_commit=FORK_COMMIT, upstream_reference_commit=UPSTREAM_REFERENCE_COMMIT, official_code_unchanged=OFFICIAL_CODE_UNCHANGED),
        environment=ENV_RECORD,
        runs=[dict(exit_code=r["exit_code"], killed_by_monitor=r["killed_by_monitor"], log_path=r["log_path"], wall_sec=r["wall_sec"], n_steps_logged=len(r["metrics"]),
                   first_step=(r["metrics"][0]["step"] if r["metrics"] else None), last_step=(r["metrics"][-1]["step"] if r["metrics"] else None),
                   last_loss_logged=(r["metrics"][-1].get("train/loss") if r["metrics"] else None),
                   last_loss_official_scale=(r["metrics"][-1].get("train/loss_official_scale") if r["metrics"] else None), timing=summarize_timing(r),
                   checkpoints_uploaded={s: dict(commit=v.get("commit"), marker_commit=v.get("marker_commit"), url=v.get("url")) for s, v in r["monitor"]["uploaded"].items()},
                   upload_failures=r["monitor"]["failed"], resumed_from=r["spec"].get("resumed_from")) for r in results],
        evaluations=evaluations or [], final_model=final_model,
        unverified=unverified or [],
        caveat="loss 低下だけで能力向上は主張しない。評価欄が空なら性能は未測定。",
    )
    path = f"{RECORD_DIR}/experiment_record_{spec['run_id']}.json"
    with open(path, "w") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False, default=str)
    print("experiment record:", path)
    if upload and HF_API is not None and HF_CKPT_REPO_ID:
        ensure_ckpt_repo()
        ts = time.strftime("%Y%m%d-%H%M%S")
        for name in (f"experiment_record_{ts}.json", "experiment_record_latest.json"):
            HF_API.upload_file(path_or_fileobj=path, path_in_repo=f"runs/{spec['run_id']}/{name}", repo_id=HF_CKPT_REPO_ID, repo_type="model",
                               commit_message=f"{spec['run_id']} experiment record {ts}")
        print(f"uploaded to https://huggingface.co/{HF_CKPT_REPO_ID}/tree/main/runs/{spec['run_id']}")
    return rec


print("helpers ready: launch_training / TrainingMonitor / upload_checkpoint / list_hf_checkpoints / download_hf_checkpoint / resume_run_from_hf / write_experiment_record")

In [ ]:
%%ao
# @title 6-b. 短い試走: 学習 → HF 保存 → プロセス終了 → HF から取得 → 新プロセスで再開（RUN_MODE="trial" のときだけ実行）
import pyarrow.parquet as pq, json, os, math, numpy as np

TRIAL_RESULTS = {}
if RUN_MODE != "trial":
    print("RUN_MODE != 'trial' なのでスキップ")
else:
    require_training_env()
    assert AO_DATA_READY, "AO データが未確定（セクション 3 のゲート）。試走でも同じデータ条件を使う"
    # 試走データ: AO parquet の先頭 TRIAL_NUM_ROWS 行（行順維持）。本学習と混ざらないよう別ファイル・別 run_id
    _trial_rows = (TRIAL_NUM_ROWS // OFFICIAL_TBS) * OFFICIAL_TBS
    assert _trial_rows >= OFFICIAL_TBS, "TRIAL_NUM_ROWS は global batch 256 以上"
    TRIAL_PARQUET = f"{DATA_DIR}/trial/Math-AO-trial{_trial_rows}.parquet"
    os.makedirs(os.path.dirname(TRIAL_PARQUET), exist_ok=True)
    pq.write_table(AO_TABLE.slice(0, _trial_rows), TRIAL_PARQUET)
    TRIAL_SHA256 = sha256_of(TRIAL_PARQUET)
    _ts = time.strftime("%Y%m%d-%H%M%S")
    _trial_id = f"trial-{MODEL_KEY}_Math-AO-{_trial_rows}_lr{fmt_lr(OFFICIAL_BASELINE['lr'])}_ep{TRIAL_EPOCHS}_bs{OFFICIAL_TBS}_{_ts}"
    _spe = steps_per_epoch(_trial_rows, OFFICIAL_TBS, N_GPUS)
    _total = _spe * TRIAL_EPOCHS
    assert TRIAL_KILL_AFTER_STEP in range(TRIAL_SAVE_FREQ, _total, TRIAL_SAVE_FREQ), f"TRIAL_KILL_AFTER_STEP は保存 step（{TRIAL_SAVE_FREQ} の倍数）かつ最終 step {_total} 未満"
    TRIAL_SPEC = build_run_spec("trial", OFFICIAL_BASELINE["lr"], TRIAL_EPOCHS, data_path=TRIAL_PARQUET, data_sha256=TRIAL_SHA256, n_rows=_trial_rows,
                                max_length=MAX_LENGTH, save_freq=TRIAL_SAVE_FREQ, upload_policy="all", run_id=_trial_id,
                                extra_note=f"試走。データ {_trial_rows} 行 / {TRIAL_EPOCHS} epoch / step 上限なし（scheduler 期間 = {_total} step）。本学習の設定ではない")
    print_run_spec(TRIAL_SPEC)
    if TRIAL_SPEC["training_view"]["enabled"]:
        # 試走 subset でも、各 step の micro batch が公式 8 GPU と同じ 4 行の組になることを確認（並べ替えは 256 行 block 内なので本学習の先頭 step とも同じ）
        _tv_rows = pq.read_table(TRIAL_SPEC["train_file"], columns=["ao_source_row"]).column("ao_source_row").to_pylist()
        assert official_micro_batches(_tv_rows, N_GPUS, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ) == \
            official_micro_batches(list(range(_trial_rows)), OFFICIAL_WORLD_SIZE, OFFICIAL_TBS, OFFICIAL_MICRO_BSZ), "試走 subset の micro batch 構成が公式と一致しない"
        print(f"試走 subset の micro batch 構成: 公式 {OFFICIAL_WORLD_SIZE} GPU と一致（{_trial_rows // OFFICIAL_TBS} step 分）")
    print(f"\n[試走 A] step {TRIAL_KILL_AFTER_STEP} の checkpoint 保存完了を検知 → 学習プロセスを一時停止 → HF へ転送・検証 → プロセス終了（切断の模擬）")
    TRIAL_RESULTS["A_interrupted"] = launch_training(TRIAL_SPEC, kill_after_step=TRIAL_KILL_AFTER_STEP, upload=True, poll_sec=5, stable_scans=False)
    _a = TRIAL_RESULTS["A_interrupted"]
    assert _a["killed_by_monitor"], "監視スレッドによる終了が起きていない（プロセスが先に完走した、または HF 転送が完了していない）"
    assert TRIAL_KILL_AFTER_STEP in _a["monitor"]["uploaded"], "kill step の checkpoint が HF に転送・検証されていない"
    _a_last = max(m["step"] for m in _a["metrics"])
    assert _a_last < _total, f"run A が最終 step {_total} まで進んでしまい、中断の模擬になっていない（last step {_a_last}）"
    print(f"run A: step {_a_last} で終了（総 step {_total}）。step {TRIAL_KILL_AFTER_STEP + 1}〜{_a_last} は再実行される")
    _hf_before = {r["step"]: r.get("folder_commit") for r in list_hf_checkpoints(_trial_id) if r["complete"]}

    # ローカル checkpoint を消して、HF からの取得だけで再開できることを確かめる
    shutil.rmtree(f"{CKPT_DIR}/{_trial_id}", ignore_errors=True)
    print(f"\n[試走 B] ローカル checkpoint を削除し、HF から step {TRIAL_KILL_AFTER_STEP} を取得して新プロセスで再開（総 epoch {TRIAL_EPOCHS} / 総 step {_total} を維持）")
    TRIAL_RESULTS["B_resumed"] = resume_run_from_hf(_trial_id, step=TRIAL_KILL_AFTER_STEP, upload=True)
    _b = TRIAL_RESULTS["B_resumed"]
    assert _b["exit_code"] == 0, "再開 run が異常終了"
    _b_steps = [m["step"] for m in _b["metrics"]]
    assert _b_steps and _b_steps[0] == TRIAL_KILL_AFTER_STEP + 1 and _b_steps[-1] == _total, f"再開後の step 列が想定外: {_b_steps}"
    _b_log = open(_b["log_path"]).read()
    _hf_after = {r["step"]: r.get("folder_commit") for r in list_hf_checkpoints(_trial_id) if r["complete"]}
    RESUME_CHECKS = dict(
        global_step_continued=(_b_steps[0] == TRIAL_KILL_AFTER_STEP + 1),
        model_loaded=("Loaded model from" in _b_log), optimizer_loaded=("Loaded optimizer from" in _b_log),
        lr_scheduler_loaded=("Loaded lr_scheduler from" in _b_log), rng_loaded=("Loaded rng from" in _b_log),
        dataloader_state_loaded=("Successfully loaded dataloader state" in _b_log),
        final_ckpt_uploaded=(_total in _b["monitor"]["uploaded"] and not _b["monitor"]["uploaded"][_total].get("previously_verified")),
        kill_step_ckpt_not_overwritten=(_hf_before.get(TRIAL_KILL_AFTER_STEP) == _hf_after.get(TRIAL_KILL_AFTER_STEP) is not None),
        no_upload_failures=(not _b["monitor"]["failed"]),
    )
    print("再開チェック:", RESUME_CHECKS)

    # 解析的な cosine schedule と比較（warmup = int(total*0.1)、verl get_cosine_schedule_with_warmup と同式）
    def expected_lr(step, total, base_lr, warmup_ratio=float(YAML_DEFAULTS.optim.warmup_steps_ratio)):
        warm = int(total * warmup_ratio)
        if step < warm:
            return base_lr * step / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return base_lr * max(0.0, 0.5 * (1 + math.cos(math.pi * prog)))

    LR_CHECK = []
    for m in _b["metrics"]:
        exp = expected_lr(m["step"], _total, OFFICIAL_BASELINE["lr"])   # train/lr は step 後の get_last_lr
        LR_CHECK.append(dict(step=m["step"], logged=m.get("train/lr"), expected_after_step=exp, ok=abs(m.get("train/lr", -1) - exp) <= 1e-9 + 1e-3 * abs(exp)))
    print("lr の継続（再開後）:", LR_CHECK)
    RESUME_CHECKS["lr_schedule_continued"] = all(x["ok"] for x in LR_CHECK)

    if TRIAL_RUN_REFERENCE:
        print(f"\n[試走 C] 中断なしの参照 run（同じ設定）。再開後の loss / データ位置を比較する")
        _ref_spec = build_run_spec("trial", OFFICIAL_BASELINE["lr"], TRIAL_EPOCHS, data_path=TRIAL_PARQUET, data_sha256=TRIAL_SHA256, n_rows=_trial_rows,
                                   max_length=MAX_LENGTH, save_freq=TRIAL_SAVE_FREQ, upload_policy="all", run_id=_trial_id + "-ref", extra_note="試走の参照 run（中断なし）")
        TRIAL_RESULTS["C_reference"] = launch_training(_ref_spec, upload=False, keep_local=99)
        _c = {m["step"]: m for m in TRIAL_RESULTS["C_reference"]["metrics"]}
        _cmp = []
        for m in _b["metrics"]:
            r = _c.get(m["step"])
            if r:
                _cmp.append(dict(step=m["step"], loss_resumed=m.get("train/loss"), loss_reference=r.get("train/loss"),
                                 rel_diff=abs(m.get("train/loss", 0) - r.get("train/loss", 0)) / max(1e-8, abs(r.get("train/loss", 0))),
                                 lr_equal=abs(m.get("train/lr", 0) - r.get("train/lr", 0)) < 1e-12))
        print("再開 run と参照 run の比較（同じ step で同じバッチなら loss は近い。大きくずれればデータ位置が復元されていない）:")
        for x in _cmp:
            print("  ", x)
        RESUME_CHECKS["loss_matches_reference_within_5pct"] = all(x["rel_diff"] < 0.05 for x in _cmp) if _cmp else None
        RESUME_CHECKS["lr_equals_reference"] = all(x["lr_equal"] for x in _cmp) if _cmp else None

    # ---- 時間計測と本学習の概算 ----
    _tA, _tB = summarize_timing(_a), summarize_timing(_b)
    _tC = summarize_timing(TRIAL_RESULTS["C_reference"]) if TRIAL_RUN_REFERENCE else None
    _step_sec = np.median([x for t in (_tA, _tB, _tC) if t and t.get("step_sec_median") for x in [t["step_sec_median"]]])
    _init_sec = _tA.get("init_sec_est") or _tA.get("init_sec_incl_first_step")
    _save_sec = float(np.median([s for t in (_tA, _tB) for s in t.get("save_sec", [])])) if any(t.get("save_sec") for t in (_tA, _tB)) else None
    _upload_sec = float(np.median([s for t in (_tA, _tB) for s in t.get("upload_sec", {}).values()])) if any(t.get("upload_sec") for t in (_tA, _tB)) else None
    print("\n=== 試走の計測 ===")
    print(f"  初期化（起動〜最初の step 前）≈ {_init_sec:.0f}s | optimizer step（勾配蓄積 {TRIAL_SPEC['grad_accum_micro_batches']} micro batch 込み）中央値 {_step_sec:.1f}s")
    print(f"  checkpoint 保存 {_save_sec}s | HF 転送+検証 {_upload_sec}s（バックグラウンド） | GPU メモリ最大 {_a['monitor']['gpu_mem_max_mb']} MB")
    _base_steps = steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * OFFICIAL_BASELINE["epochs"]
    _base_saves = len(set(list(range(SAVE_FREQ, _base_steps + 1, SAVE_FREQ)) + [_base_steps]))
    _base_est = _init_sec + _base_steps * _step_sec + _base_saves * (_save_sec or 0)
    print(f"  baseline 概算: init {_init_sec:.0f}s + {_base_steps} steps x {_step_sec:.1f}s + {_base_saves} saves x {_save_sec or 0}s ≈ {_base_est / 3600:.1f} h（HF 転送は並行。評価・変換・最終アップロードは未計測）")
    _grid = [normalize_run(c) for c in (SEARCH_GRID_CONFIRMED or [])]
    if _grid:
        _grid_steps = sum(steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * e for _, e, _ in _grid)
        print(f"  探索全体（{len(_grid)} 条件 = {_grid_steps} steps）概算 ≈ {(_grid_steps * _step_sec + len(_grid) * _init_sec) / 3600:.1f} h（保存・評価は含まない）")
    print("  注意: 試走は step 上限を使わず epoch 数で短くしたので scheduler 期間は試走内で完結している。試走の checkpoint・重み・データ削減は本学習に使わない")
    TRIAL_TIMING = dict(init_sec=_init_sec, step_sec_median=float(_step_sec), save_sec=_save_sec, upload_sec=_upload_sec, gpu_mem_max_mb=_a["monitor"]["gpu_mem_max_mb"],
                        baseline_est_hours=_base_est / 3600, timing_A=_tA, timing_B=_tB, timing_C=_tC, resume_checks=RESUME_CHECKS, lr_check=LR_CHECK)
    with open(f"{RECORD_DIR}/trial_{_trial_id}.json", "w") as f:
        json.dump(dict(spec=TRIAL_SPEC, timing=TRIAL_TIMING, results={k: dict(exit_code=v["exit_code"], log=v["log_path"], uploaded=list(v["monitor"]["uploaded"])) for k, v in TRIAL_RESULTS.items()}),
                  f, indent=2, ensure_ascii=False, default=str)
    if not all(v for v in RESUME_CHECKS.values() if v is not None):
        print("*** 再開チェックに失敗項目がある。原因を解決するまで本学習へ進まない ***")
    else:
        print("試走の再開チェックはすべて合格")

## 7. baseline の確認または実行

In [ ]:
%%ao
# @title 7. baseline（公式 CoT 設定 lr 5e-5 / 8 epoch の AO 学習）の確認または実行
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})


def preflight_train(spec):
    """本学習の直前確認: データ条件・ゲート・試走設定の混入・衝突"""
    assert RUN_MODE == "train", "RUN_MODE='train' でのみ本学習を開始する"
    require_training_env(spec["n_gpus"])
    assert AO_DATA_READY, "AO データに未解決の抽出/照合問題がある（セクション 3）"
    assert MASK_CHECK_OK, "loss mask の確認に失敗している（セクション 4）"
    assert spec["data_path"] == AO_PARQUET and spec["n_rows"] == DATASET_EXPECTED_ROWS and spec["data_sha256"] == AO_SHA256, "本学習は全 20,480 行の AO データで行う"
    assert not spec["run_id"].startswith("trial-") and spec["total_training_steps_override"] is None, "試走設定が本学習に混入している"
    assert "trainer.total_training_steps" not in spec["overrides"], "step 上限は本学習で使わない"
    assert spec["overrides"]["trainer.resume_mode"] == "disable", "新規 run は同じ Base から独立に開始する（再開は『再開』セル）"
    if spec["n_gpus"] != OFFICIAL_WORLD_SIZE:
        if spec["training_view"]["enabled"]:
            assert VIEW_CHECK_OK and TRAIN_VIEW_CHECK.get("actual_world") == spec["n_gpus"], \
                "公式 8 GPU の再現確認（セクション 4-b の 4.4）が未実施または失敗。同じ GPU 台数で 4-b を実行し直す"
            assert spec["training_view"]["order_sha256"] == TRAIN_VIEW_CHECK["order_sha256"]
        else:
            print(f"*** 注意: EMULATE_OFFICIAL_WORLD_SIZE=False。GPU {spec['n_gpus']} 台では micro batch 構成と勾配の大きさが公式 CoT 学習と異なる ***")
    if HF_UPLOAD_CHECKPOINTS:
        assert HF_API is not None, "HF 未ログイン。学習中の HF 保存が要件なので開始しない"
        remote = hf_run_paths(HF_CKPT_REPO_ID, spec["run_id"])
        assert not remote, f"HF に同名 run {spec['run_id']} が存在する（steps {sorted(remote)}）。上書きしない。再開は『再開』セル、別 run は RUN_ID_SUFFIX"
    assert not os.path.exists(spec["overrides"]["trainer.default_local_dir"]), f"ローカルに同名 run のディレクトリがある: {spec['overrides']['trainer.default_local_dir']}"
    _disk = shutil.disk_usage(WORK_DIR)
    print(f"preflight OK: disk free {_disk.free / 1e9:.0f} GB (ckpt ≈ {CKPT_SIZE_EST_GB:.1f} GB x local keep {LOCAL_KEEP_LAST_N_VERIFIED_CKPTS})")


if RUN_MODE != "train" or RUN_KIND != "baseline":
    print("RUN_MODE='train' かつ RUN_KIND='baseline' のときだけ実行する（現在: %s / %s）" % (RUN_MODE, RUN_KIND))
    if HF_API is not None and HF_CKPT_REPO_ID:
        _bid = make_run_id("baseline", OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], suffix=RUN_ID_SUFFIX)
        _remote = hf_run_paths(HF_CKPT_REPO_ID, _bid)
        print(f"HF 上の baseline run {_bid}: {'steps ' + str(sorted(_remote)) if _remote else 'なし'}")
        print("  注意: 既存成果物があっても、その run_config（manifest）で AO target と設定を確認するまで再利用可能とは判断しない")
else:
    BASELINE_SPEC = build_run_spec("baseline", OFFICIAL_BASELINE["lr"], OFFICIAL_BASELINE["epochs"], data_path=AO_PARQUET, data_sha256=AO_SHA256,
                                   n_rows=DATASET_EXPECTED_ROWS, max_length=MAX_LENGTH, save_freq=SAVE_FREQ,
                                   extra_note="baseline: 公開 CoT 設定 (lr 5e-5, 8 epoch) を AO データで実行")
    print_run_spec(BASELINE_SPEC)
    preflight_train(BASELINE_SPEC)
    TRAIN_RESULTS[BASELINE_SPEC["run_id"]] = launch_training(BASELINE_SPEC)
    _r = TRAIN_RESULTS[BASELINE_SPEC["run_id"]]
    print("timing:", json.dumps({k: v for k, v in summarize_timing(_r).items() if k != "step_sec_all"}, default=str))
    write_experiment_record(BASELINE_SPEC, [_r], unverified=["論文と同じ評価（セクション 9）未実施", "最終モデル未変換"])

## 8. LR / epoch 探索（保留中）

In [ ]:
%%ao
# @title 8. LR / epoch / scheduler 探索（保留中。baseline の性能が出ないときに使う。候補は論文の最適化条件。各候補を同じ Base から独立に学習。8 epoch run の途中 checkpoint を短い epoch 設定の代用にしない）
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})
SEARCH_SPECS = {}
if RUN_MODE != "train" or RUN_KIND != "search":
    print("RUN_MODE='train' かつ RUN_KIND='search' のときだけ実行する（現在: %s / %s）" % (RUN_MODE, RUN_KIND))
    print("探索は保留中。候補:", SEARCH_GRID_PROPOSED, "| 確定グリッド:", SEARCH_GRID_CONFIRMED, "| 実行リスト:", SEARCH_RUN_LIST)
elif not SEARCH_RUN_LIST:
    print("SEARCH_RUN_LIST が空。実行する (lr, epochs, scheduler) を設定セルで指定する")
else:
    _confirmed = {normalize_run(c) for c in (SEARCH_GRID_CONFIRMED or [])}
    for _lr, _ep, _sched in (normalize_run(r) for r in SEARCH_RUN_LIST):
        _tag = "confirmed" if (_lr, _ep, _sched) in _confirmed else "proposed"
        _spec = build_run_spec("search", _lr, _ep, data_path=AO_PARQUET, data_sha256=AO_SHA256, n_rows=DATASET_EXPECTED_ROWS, max_length=MAX_LENGTH,
                               save_freq=SAVE_FREQ, lr_scheduler=_sched,
                               extra_note=f"探索候補 ({_tag})。総 epoch {_ep} に応じて scheduler 期間 = {steps_per_epoch(DATASET_EXPECTED_ROWS, OFFICIAL_TBS, N_GPUS) * _ep} step")
        SEARCH_SPECS[_spec["run_id"]] = _spec
    for _rid, _spec in SEARCH_SPECS.items():
        print("=" * 100)
        print_run_spec(_spec)
        _remote = hf_run_paths(HF_CKPT_REPO_ID, _rid) if HF_API else {}
        if _remote:
            print(f"HF に {_rid} が既にある (steps {sorted(_remote)})。完了済みなら再実行しない。途中なら『再開』セルで RESUME_RUN_ID を指定する")
            continue
        preflight_train(_spec)
        TRAIN_RESULTS[_rid] = launch_training(_spec)
        write_experiment_record(_spec, [TRAIN_RESULTS[_rid]], unverified=["論文と同じ評価（セクション 9）未実施", "最終モデル未変換"])
    print("探索 run の完了状況:", {k: ("OK" if v["exit_code"] == 0 else f"rc={v['exit_code']}") for k, v in TRAIN_RESULTS.items()})

## 再開: Colab 切断後に HF の checkpoint から続ける

切断後は **セル 0（認証）→ 1（環境構築）→ 設定セル → 2 → 3 → 4-a → 4-b → 6-a** を実行してから、`RESUME_RUN_ID` を設定してこのセルを実行する。

In [ ]:
%%ao
# @title 再開: Colab 切断後に HF の完了済み checkpoint から再開する（RESUME_RUN_ID を設定。セル 0〜4 と 6-a を先に実行）
# 手順: 1) 環境と Fork の commit を復元（セル 1・2） 2) HF の checkpoint 一覧 3) 最新の完了済み or 指定 step を選択
#       4) HF revision を記録して取得・manifest で検証 5) trainer.resume_mode=resume_path / resume_from_path で再開
TRAIN_RESULTS = globals().get("TRAIN_RESULTS", {})
if not RESUME_RUN_ID:
    print("RESUME_RUN_ID が未設定。再開しない。HF 上の run 一覧:")
    if HF_API is not None and HF_CKPT_REPO_ID and hf_repo_exists(HF_CKPT_REPO_ID):
        _runs = sorted({f.split("/")[1] for f in HF_API.list_repo_files(HF_CKPT_REPO_ID, repo_type="model") if f.startswith("runs/")})
        for _r in _runs:
            print("  ", _r)
else:
    assert RUN_MODE in ("train", "trial"), "再開は RUN_MODE='train'（本学習）または 'trial'（試走 run）で行う"
    assert AO_DATA_READY and MASK_CHECK_OK
    TRAIN_RESULTS[RESUME_RUN_ID] = resume_run_from_hf(RESUME_RUN_ID, step=RESUME_STEP)
    _r = TRAIN_RESULTS[RESUME_RUN_ID]
    print("timing:", json.dumps({k: v for k, v in summarize_timing(_r).items() if k != "step_sec_all"}, default=str))
    write_experiment_record(_r["spec"], [_r], unverified=["再開 run。それ以前の run 結果は HF の experiment_record を参照"])

## 9. 評価（論文と同じ方法）

公式の `evaluation/math_eval/math_eval_budget.py` を無変更で実行する（MATH500 avg@3, AIME24 avg@10, temperature 0.6, 最大 32768 token）。論文と同じく各 step の推移を記録し、条件や checkpoint の選択はしない。

In [ ]:
%%ao
# @title 9. 論文と同じ評価（公式 evaluation/math_eval/math_eval_budget.py を無変更で実行。各 step の推移を記録し、選択はしない）
import json, os, sys, time, glob, shutil, subprocess, hashlib

EVAL_ENV_DIR = ENV_BOOTSTRAP["eval_env_dir"]
EVAL_PY = f"{EVAL_ENV_DIR}/bin/python"
EVAL_DIR = f"{WORK_DIR}/eval"
MATH_EVAL_DIR = f"{REPO_DIR}/evaluation/math_eval"
# 公式の評価スクリプト（math_eval_budget.py の DATASET_CONFIGS、utils/utils.py の DATASET_KEYS）はデータの場所を作者の環境の絶対パスで持つ。
# コードは変えず、このパスに Fork の clone へのシンボリックリンクを置く
OFFICIAL_EVAL_ROOT = "/mnt/shared-storage-user/renqihan/sft_generalization"
PAPER_EVAL_DATASET_PATHS = {"MATH500": f"{OFFICIAL_EVAL_ROOT}/evaluation/data/math/MATH500",
                            "AIME24": f"{OFFICIAL_EVAL_ROOT}/evaluation/data/math/converted_aime_dataset"}
PAPER_EVAL_OUTPUT_KEYS = {"MATH500": ("math500", 2000), "AIME24": ("aime", 5000)}   # (save_name, budget_list[-1])。結果ファイルの場所
PAPER_EVAL_K = {"MATH500": 3, "AIME24": 10}                                         # DATASET_CONFIGS の test_n（論文 App. B.4 と同じ）
PAPER_EVAL_DEVIATIONS = ["tensor_parallel_size 1（論文と公式スクリプトは 2 GPU で 2）と GPU の種類（論文は H200）。数値がわずかに変わると temperature 0.6 の"
                         "サンプリング結果も変わるので、論文値とはサンプリングの揺らぎの範囲（AIME24 は 30 問なので数ポイント）で異なり得る。"
                         "CoT 学生と厳密に比べるときは同じ環境で CoT も評価する（PAPER_EVAL_COT_STEPS）"]
# ipykernel は MPLBACKEND を inline 用の値に設定する。評価用の環境には matplotlib_inline が無く、公式スクリプトの import matplotlib が失敗するので
# subprocess には描画しない既定の Agg を渡す（評価の値には影響しない）
EVAL_SUBPROCESS_ENV = {"MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}


def eval_subprocess_env(**extra):
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "VIRTUAL_ENV") and not k.startswith("UV_")}
    env.update(EVAL_SUBPROCESS_ENV, **extra)
    return env


def ensure_eval_env():
    """論文の数学評価用の環境（Python 3.12 + vLLM 0.8.5 + 公式 pin）を作る。学習用の公式環境とは別"""
    if GPU_PROFILE != "official":
        raise SystemExit(f"GPU プロファイル {GPU_PROFILE}: vLLM 0.8.5（公式 pin）はこの GPU 用のコードを持たない（sm_70〜sm_90 のみ）。"
                         "評価は A100 のランタイムで行う（学習済みの checkpoint は HF にある）")
    pins = ENV_BOOTSTRAP["eval_pip_pinned"]
    cons = ENV_BOOTSTRAP["eval_constraints"]   # 公式 requirements.txt（矛盾する protobuf / typer を除く）
    h = hashlib.sha256((json.dumps(pins) + open(cons).read()).encode()).hexdigest()[:16]
    marker = f"{EVAL_ENV_DIR}/ao_eval_env_ok.json"
    if os.path.isfile(marker) and json.load(open(marker)).get("hash") == h and os.path.isfile(EVAL_PY):
        return json.load(open(marker))
    uv_env = eval_subprocess_env(UV_CACHE_DIR=ENV_BOOTSTRAP["uv_cache_dir"], UV_PYTHON_INSTALL_DIR=ENV_BOOTSTRAP["uv_python_install_dir"])
    uv = ENV_BOOTSTRAP["uv_cmd"]
    t0 = time.time()
    shutil.rmtree(EVAL_ENV_DIR, ignore_errors=True)
    for cmd in (uv + ["venv", "--quiet", "--python", "3.12", "--python-preference", "only-managed", EVAL_ENV_DIR],
                uv + ["pip", "install", "--quiet", "--python", EVAL_PY, "-c", cons] + pins):
        r = subprocess.run(cmd, capture_output=True, text=True, env=uv_env)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
            raise RuntimeError("評価用の環境の作成に失敗")
    r = subprocess.run([EVAL_PY, "-c", "import json, vllm, torch, transformers, math_verify, xformers, matplotlib, pandas, datasets; "
                                       "print(json.dumps(dict(vllm=vllm.__version__, torch=torch.__version__, transformers=transformers.__version__, "
                                       "xformers=xformers.__version__, datasets=datasets.__version__)))"],
                       capture_output=True, text=True, env=uv_env)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("評価用の環境の import 確認に失敗")
    freeze = subprocess.run(uv + ["pip", "freeze", "--python", EVAL_PY], capture_output=True, text=True, env=uv_env).stdout
    with open(f"{RECORD_DIR}/eval_env_freeze.txt", "w") as f:   # 評価用環境の全パッケージの版（記録）
        f.write(freeze)
    info = dict(hash=h, pins=pins, versions=json.loads(r.stdout.strip().splitlines()[-1]), created_sec=round(time.time() - t0),
                freeze_file=f"{RECORD_DIR}/eval_env_freeze.txt")
    json.dump(info, open(marker, "w"), indent=2)
    print(f"評価用の環境を作成した: {EVAL_ENV_DIR}（{info['created_sec']} 秒）{info['versions']}")
    return info


def ensure_official_eval_paths():
    """公式スクリプトが参照する絶対パスに Fork の clone へのリンクを置き、データがあることを確認する"""
    os.makedirs(os.path.dirname(OFFICIAL_EVAL_ROOT), exist_ok=True)
    if os.path.islink(OFFICIAL_EVAL_ROOT):
        if os.path.realpath(OFFICIAL_EVAL_ROOT) != os.path.realpath(REPO_DIR):
            os.remove(OFFICIAL_EVAL_ROOT)
            os.symlink(REPO_DIR, OFFICIAL_EVAL_ROOT)
    elif os.path.exists(OFFICIAL_EVAL_ROOT):
        raise SystemExit(f"{OFFICIAL_EVAL_ROOT} がリンク以外で存在する。消さずに止める")
    else:
        os.symlink(REPO_DIR, OFFICIAL_EVAL_ROOT)
    for ds, p in PAPER_EVAL_DATASET_PATHS.items():
        assert os.path.isdir(p), f"{ds} のデータが無い: {p}"


def read_official_math_result(out_dir, dataset):
    """math_eval_budget.py が書く results/<name>/<save_name>_budget<B>/no_instruct_512.json の native（打ち切りなし）を読む。
    論文の値 = native の average_pass_rate（MATH500 は avg@3、AIME24 は avg@10）"""
    save_name, budget = PAPER_EVAL_OUTPUT_KEYS[dataset]
    hits = glob.glob(f"{out_dir}/results/*/{save_name}_budget{budget}/no_instruct_512.json")
    if len(hits) != 1:
        return None
    native = json.load(open(hits[0]))["native"]
    return {f"avg@{PAPER_EVAL_K[dataset]}": 100 * float(native["average_pass_rate"]), "pass@1(first sample)": 100 * float(native["pass@1"]),
            "maj@k": 100 * float(native["pass@k(majority)"]), "avg_length_tokens": float(native["average_length"]), "result_file": hits[0]}


def run_official_math_eval(model_dir, dataset, out_dir, log_path):
    """python math_eval_budget.py --model_path ... --dataset ... --types=no_instruct --output_dir ... --num_gpus=1（evaluation/scripts/run_math.sh と同じ引数。GPU 数だけ 1）"""
    cached = read_official_math_result(out_dir, dataset)
    if cached:
        return cached
    save_name, budget = PAPER_EVAL_OUTPUT_KEYS[dataset]
    for f in glob.glob(f"{out_dir}/new_outputs/*/{save_name}_budget{budget}/no_instruct_512.json"):
        try:
            json.load(open(f))
        except ValueError:   # 生成結果の書き込み中に止まった壊れたファイル。残すと公式スクリプトが毎回読み込みで失敗する
            print(f"    壊れた生成結果を削除して生成し直す: {f}")
            os.remove(f)
    check_gpu_free_for_vllm()
    cmd = [EVAL_PY, "math_eval_budget.py", f"--model_path={model_dir}", f"--dataset={PAPER_EVAL_DATASET_PATHS[dataset]}", "--types=no_instruct",
           f"--output_dir={out_dir}", "--num_gpus=1"]
    env = eval_subprocess_env(VLLM_ATTENTION_BACKEND="XFORMERS", VLLM_WORKER_MULTIPROC_METHOD="spawn", CUDA_VISIBLE_DEVICES="0",   # run_math.sh と同じ
                              VIRTUAL_ENV=EVAL_ENV_DIR, PATH=f"{EVAL_ENV_DIR}/bin" + os.pathsep + os.environ.get("PATH", ""))
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as lf:
        lf.write(" ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=MATH_EVAL_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace")
        for line in p.stdout:
            lf.write(line)
            if any(k in line for k in ("Evaluating model", "Test:", "Time taken", "Error", "Traceback", "out of memory")):
                print("   ", line.rstrip()[:300], flush=True)
            elif time.time() - last > 600:
                print(f"    ... {dataset} 生成中 {(time.time() - t0) / 60:.0f} 分", flush=True)
                last = time.time()
        rc = p.wait()
    if rc != 0:
        print(open(log_path).read()[-3000:])
        raise RuntimeError(f"math_eval_budget.py が失敗 (rc={rc})。ログ: {log_path}")
    res = read_official_math_result(out_dir, dataset)
    assert res, f"結果ファイルが見つからない: {out_dir}"
    res["elapsed_sec"] = round(time.time() - t0)
    return res


def check_gpu_free_for_vllm(max_used_frac=0.10):
    """vLLM（V0）は GPU 全体の 85% を確保しようとし、他のプロセスが使っている分を差し引かない。他が 10% 以上使っていれば止める"""
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
    if r.returncode != 0:
        return
    used, total = (float(x) for x in r.stdout.splitlines()[0].split(","))
    if used / total > max_used_frac:
        raise SystemExit(f"GPU メモリを他のプロセスが {used:.0f}/{total:.0f} MiB 使っている。学習や他の評価が終わってから実行する"
                         "（このカーネルでモデルを GPU に載せた場合はセル 1 から実行し直す）")


def cot_public_step_dir(step):
    """公開 CoT 学生の stepN/（HF 形式）を取得してローカルパスを返す（vLLM は HF の subfolder を直接読めない）"""
    d = f"{MODELS_DIR}/cot_{MODEL_KEY}_{MODEL_INFO['cot_revision'][:8]}"
    snapshot_download(repo_id=MODEL_INFO["cot_repo"], revision=MODEL_INFO["cot_revision"], allow_patterns=[f"step{step}/*"], local_dir=d)
    assert os.path.isfile(f"{d}/step{step}/config.json"), f"{MODEL_INFO['cot_repo']} に step{step} が無い（公開されている step: 10, 20, 40, 80, 160, 320, 480, 640）"
    return f"{d}/step{step}"


def paper_eval_targets():
    targets = []
    if PAPER_EVAL_BASE:
        targets.append(dict(kind="base", name=f"{MODEL_KEY}-Base", step=0))
    for s in PAPER_EVAL_COT_STEPS:
        targets.append(dict(kind="cot", name=f"{MODEL_KEY}_Math-CoT(public)", step=int(s)))
    runs = PAPER_EVAL_AO_RUNS or [rid for rid, r in globals().get("TRAIN_RESULTS", {}).items() if r["exit_code"] == 0]
    if not runs:
        print("  WARN: 評価する AO run が無い。このセッションで学習していない場合（切断後など）は PAPER_EVAL_AO_RUNS に run_id を指定する")
    for rid in runs:
        done = [c["step"] for c in list_hf_checkpoints(rid) if c["complete"]]   # 重みだけの step も評価できる
        steps = [s for s in PAPER_EVAL_STEPS if s in done] if PAPER_EVAL_AO_STEPS == "paper" else [int(s) for s in PAPER_EVAL_AO_STEPS]
        missing = [s for s in steps if s not in done]
        assert not missing, f"{rid}: HF に無い step {missing}"
        cond = get_manifest(rid, steps[0])["run_config"].get("paper_condition") if steps else None
        targets += [dict(kind="ao", name=rid, run_id=rid, step=s, paper_condition=cond) for s in steps]
    return targets


def paper_reference(kind, step, dataset, paper_condition=None):
    """論文の値（App. D）: Base なら Base の値、それ以外は同じ step の公開 CoT 学生の値。無ければ None。
    1.7B/4B の CoT 学生は論文の既定条件（lr 5e-5, 8 epoch, cosine）しか無いので、AO も既定条件の run だけに並べる"""
    ref, j = PAPER_REFERENCE.get(MODEL_KEY, {}), {"MATH500": 0, "AIME24": 1}.get(dataset)
    if j is None:
        return None
    if kind == "base":
        return ref.get("base", {}).get(dataset)
    if kind == "ao" and not str(paper_condition or "").startswith("default"):
        return None
    row = ref.get("Math-CoT", {}).get(step)
    return None if row is None else row[j]


def evaluate_paper_target(t):
    """1 つのモデル（run と step）を論文と同じ方法で評価する。AO は HF の checkpoint を変換してから評価し、この関数が作った変換物だけを評価後に消せる"""
    tag = f"{t['name']}_step{t['step']}"
    out_dir = f"{EVAL_DIR}/paper/{tag}"
    row = dict(t, results={}, deviations=PAPER_EVAL_DEVIATIONS, evaluated_at=now_iso())
    cached = {ds: read_official_math_result(out_dir, ds) for ds in PAPER_EVAL_DATASETS}
    created, model_dir = [], None
    if not all(cached.values()):
        if t["kind"] == "base":
            model_dir = get_base_model_local()
        elif t["kind"] == "cot":
            model_dir = cot_public_step_dir(t["step"])
        else:
            _ck = f"{CKPT_DIR}/{t['run_id']}/global_step_{t['step']}"
            _merged = f"{WORK_DIR}/merged/{t['run_id']}/merged_step{t['step']}"
            _had = (os.path.isdir(_ck), os.path.isdir(_merged))
            model_dir = merge_checkpoint(t["run_id"], t["step"])
            created = ([] if _had[1] else [model_dir]) + ([] if _had[0] else [_ck])
    row["model_dir"] = model_dir
    for ds in PAPER_EVAL_DATASETS:
        print(f"  [{tag}] {ds} ...", flush=True)
        row["results"][ds] = cached[ds] or run_official_math_eval(model_dir, ds, out_dir, f"{LOG_DIR}/eval/{tag}_{ds}.log")
        row["results"][ds]["paper_reference"] = paper_reference(t["kind"], t["step"], ds, t.get("paper_condition"))
    if PAPER_EVAL_DELETE_MERGED and t["kind"] == "ao":
        for d in created:
            shutil.rmtree(d, ignore_errors=True)
    return row


def save_paper_eval_row(row):
    """評価ごとに保存する（切断で失わないよう HF にも時刻付きの新しいファイルで置く。上書きしない）"""
    with open(f"{RECORD_DIR}/paper_eval_results.json", "w") as f:
        json.dump(PAPER_EVAL_RESULTS, f, indent=2, ensure_ascii=False)
    if HF_API is not None and HF_CKPT_REPO_ID:
        ts = time.strftime("%Y%m%d-%H%M%S")
        tag = f"{row['name']}_step{row['step']}".replace("/", "_")
        path_in_repo = f"runs/{row['run_id']}/paper_eval/step{row['step']}_{ts}.json" if row["kind"] == "ao" else f"paper_eval/{tag}_{ts}.json"
        local = f"{RECORD_DIR}/paper_eval_{tag}_{ts}.json"
        json.dump(row, open(local, "w"), indent=2, ensure_ascii=False)
        HF_API.upload_file(path_or_fileobj=local, path_in_repo=path_in_repo, repo_id=HF_CKPT_REPO_ID, repo_type="model",
                           commit_message=f"paper-protocol evaluation {tag}")


PAPER_EVAL_RESULTS = globals().get("PAPER_EVAL_RESULTS", [])
print("=== 論文と同じ評価 ===")
print(f"  スクリプト: evaluation/math_eval/math_eval_budget.py（無変更）| データ: {PAPER_EVAL_DATASETS} | "
      f"MATH500 avg@3, AIME24 avg@10, temperature 0.6, top_p 0.95, 最大 32768 token, math-verify")
print("  論文との差:", PAPER_EVAL_DEVIATIONS)
if not PAPER_EVAL_ENABLED:
    print("  PAPER_EVAL_ENABLED=False のため実行しない（学習が終わってから True にする）")
else:
    assert N_GPUS >= 1, "GPU が必要"
    _eval_env = ensure_eval_env()
    ensure_official_eval_paths()
    _targets = paper_eval_targets()
    print(f"  評価対象 {len(_targets)} 件: {[(t['name'], t['step']) for t in _targets]}")
    for _t in _targets:
        _row = evaluate_paper_target(_t)
        _row["eval_env"] = _eval_env.get("versions")
        _key = (_row["kind"], _row["name"], _row["step"])
        PAPER_EVAL_RESULTS[:] = [r for r in PAPER_EVAL_RESULTS if (r["kind"], r["name"], r["step"]) != _key] + [_row]   # 再実行で重複させない
        save_paper_eval_row(_row)
    print(f"\n  {'model':60s} {'step':>5s} " + " ".join(f"{d + ' avg@' + str(PAPER_EVAL_K[d]):>14s} {'(論文 CoT)':>10s} {'長さ':>7s}" for d in PAPER_EVAL_DATASETS))
    for _r in PAPER_EVAL_RESULTS:
        _cells = []
        for d in PAPER_EVAL_DATASETS:
            _x = _r["results"].get(d)
            if _x is None:
                _cells.append(f"{'-':>14s} {'':>10s} {'':>7s}")
                continue
            _ref = _x.get("paper_reference")
            _cells.append(f"{_x[f'avg@{PAPER_EVAL_K[d]}']:14.1f} {('' if _ref is None else f'{_ref:.1f}'):>10s} {_x['avg_length_tokens']:7.0f}")
        print(f"  {_r['name'][:60]:60s} {_r['step']:5d} " + " ".join(_cells))
    print("  （論文 CoT）は同じ step の公開 CoT 学生の論文値（App. D。Base の行は Base の論文値。論文の既定条件以外の AO run には並べない）。NoCoT は PAPER_REFERENCE")
    print("  結果は評価ごとに records/paper_eval_results.json と HF（runs/<run_id>/paper_eval/, paper_eval/）に保存した")

## 10. 最終モデルの変換・HF 保存・読み込み確認・Model Card

In [ ]:
%%ao
# @title 10. 最終モデル: verl.model_merger で HF 形式へ変換 → HF 保存 → 読み込み・短い生成の確認 → Model Card
import json, os, time, torch, glob
from transformers import AutoTokenizer, AutoModelForCausalLM

_final_rid = FINAL_RUN_ID or next(iter(globals().get("TRAIN_RESULTS", {}) or {}), None)
if not _final_rid:
    print("FINAL_RUN_ID が未設定で、このセッションに学習 run も無い。設定セルで FINAL_RUN_ID を指定する")
elif RUN_MODE == "dry_run":
    print("dry_run では変換・アップロードを行わない")
else:
    assert HF_API is not None and HF_CKPT_REPO_ID, "HF にログインしていない"
    _ckpts = list_hf_checkpoints(_final_rid)
    _complete = [c["step"] for c in _ckpts if c["complete"]]
    print(f"{_final_rid}: HF 完了済み steps = {_complete}")
    if FINAL_STEP == "last":
        _tot = _ckpts[-1].get("total_steps") if _ckpts else None
        _step = max(_complete) if _complete else None
        if _tot and _step != _tot:
            print(f"WARN: 最終 step {_tot} の checkpoint が HF に無い（最新は {_step}）。学習が完了していない可能性がある")
    else:
        _step = int(FINAL_STEP)
    assert _step in _complete, f"step {_step} は HF に完了済みでない"
    MERGED_DIR = merge_checkpoint(_final_rid, _step)
    _saved_spec = get_manifest(_final_rid, _step)["run_config"]
    FINAL_REPO = HF_FINAL_MODEL_REPO_ID or f"{HF_ACCOUNT_NAME}/{_final_rid}"
    _sub = f"step{_step}"   # 公開 CoT モデルと同じ stepNNN/ サブディレクトリ構成
    HF_API.create_repo(FINAL_REPO, private=HF_FINAL_MODEL_PRIVATE, repo_type="model", exist_ok=True)
    _existing = [f for f in HF_API.list_repo_files(FINAL_REPO, repo_type="model") if f.startswith(_sub + "/")]
    if _existing:
        raise SystemExit(f"{FINAL_REPO}/{_sub} は既に存在する。上書きしない。HF_FINAL_MODEL_REPO_ID を変える")

    _paper_evals = [r for r in globals().get("PAPER_EVAL_RESULTS", []) if r.get("run_id") == _final_rid]
    if not _paper_evals:   # 別のセッションで評価した場合は HF の runs/<run_id>/paper_eval/ から step ごとに最新の結果を読む
        _latest = {}
        for _f in sorted(f for f in HF_API.list_repo_files(HF_CKPT_REPO_ID, repo_type="model") if f.startswith(f"runs/{_final_rid}/paper_eval/")):
            _latest[_f.split("/")[-1].split("_")[0]] = _f   # step<N>_<時刻>.json → 時刻順で最後が最新
        _paper_evals = [json.load(open(hf_hub_download(HF_CKPT_REPO_ID, _f, repo_type="model", local_dir=f"{WORK_DIR}/hf_markers"))) for _f in _latest.values()]
        if _paper_evals:
            print(f"論文と同じ評価の結果を HF から読んだ: {len(_paper_evals)} step")
    _pe_md = "\n".join(f"| {r['step']} | " + " | ".join(f"{r['results'][d][f'avg@{PAPER_EVAL_K[d]}']:.1f} ({'' if r['results'][d].get('paper_reference') is None else r['results'][d]['paper_reference']})"
                                                         for d in PAPER_EVAL_DATASETS) + f" | {r['results'][PAPER_EVAL_DATASETS[0]]['avg_length_tokens']:.0f} |"
                        for r in sorted(_paper_evals, key=lambda r: r["step"])) or "| (未実施) |" + " - |" * (len(PAPER_EVAL_DATASETS) + 1)
    _changes_md = "\n".join(f"| `{c['key']}` | `{c['official']}` | `{c['new']}` | {c['reason']} |" for c in _saved_spec["changes_vs_official"])
    _audit = AO_AUDIT_SUMMARY
    _tv = _saved_spec.get("training_view") or dict(enabled=False)
    if _tv.get("enabled"):
        _align_md = (f"- 学習は {_saved_spec['n_gpus']} GPU。公式 trainer は 1 GPU 内の micro batch の勾配を和で蓄積し GPU 間は平均するので、そのままでは micro batch の組み合わせと勾配の大きさが公式と変わる。\n"
                     f"- 学習用 file の行を並べ替え（`{os.path.basename(_saved_spec['train_file'])}`, sha256 {_saved_spec['train_file_sha256'][:12]}）、各 step の各 micro batch を公式 {_tv['official_world']} GPU と同じ 4 行にした。\n"
                     f"- `trainer.importance_sampling_mode=adv-only` と `advantage={_tv['advantage']}` で loss を {_tv['advantage']} 倍し、勾配を公式と同じ大きさにした（ログの train/loss も {_tv['advantage']} 倍）。\n")
    elif _saved_spec["n_gpus"] == _saved_spec.get("official_world_size", 8):
        _align_md = f"- 公式と同じ {_saved_spec['n_gpus']} GPU で学習した。\n"
    else:
        _align_md = f"- **合わせ込みなし**: {_saved_spec['n_gpus']} GPU で学習し、micro batch の組み合わせと勾配の大きさが公式と異なる。\n"
    _align_md += "- 残る差（公式コードを変えずには揃えられない）:\n" + "\n".join(f"  - {r}" for r in _saved_spec.get("residual_differences", []))
    MODEL_CARD = f"""---
license: apache-2.0
base_model: {_saved_spec['base_repo']}
tags: [answer-only-distillation, sft, math, rethink-sft-generalization]
---
# {_final_rid} (Answer-only distillation student, step {_step})

Answer-only (AO) 蒸留の学生モデル。公開 CoT 学生 `{_saved_spec['cot_repo']}/{_saved_spec['cot_subfolder']}` と比較するために、
同じ Base・同じ prompt・同じ問題で、教師 `{TEACHER['repo']}` の**最終回答だけ**を target に Full-parameter SFT した。
学習コードは公式実装 (`verl.trainer.fsdp_sft_trainer_ours`) を無変更で使用。重みは `{_sub}/` にある（公開 CoT モデルと同じ配置）。

## 対応関係と revision
| 役割 | repo | revision |
| --- | --- | --- |
| Teacher | {TEACHER['repo']} | {TEACHER['revision']} |
| Base | {_saved_spec['base_repo']} | {_saved_spec['base_revision']} |
| CoT student (比較対象) | {_saved_spec['cot_repo']} (subfolder {_saved_spec['cot_subfolder']}) | {_saved_spec['cot_revision']} |
| 学習データ | {_saved_spec['dataset_repo']} | {_saved_spec['dataset_revision']} (raw sha256 {DATASET_EXPECTED_SHA256[:12]}, AO parquet sha256 {_saved_spec['data_sha256'][:12]}) |
| コード (Fork) | {FORK_REPO_URL} | {_saved_spec['fork_commit']} (参照元 {UPSTREAM_REFERENCE_COMMIT}) |
| tokenizer | Base と同一（checkpoint 内 huggingface/ から復元） | - |

## AO データの作り方と監査
- target = `response`（教師 CoT+回答）の `</think>` 以降の最後の `\\boxed{{...}}` を原文のまま（style: `{_saved_spec['target_style']}`）。CoT・説明文は含めない。参照正解 `answer` は照合のみ。
- 抽出: `verl/utils/reward_score/math_verify_ours.py` の `last_boxed_only_string`、照合: 同 `compute_score`（math-verify）。
- 行数 {_audit['rows']}、抽出状況 {json.dumps(_audit['extraction_status'])}、照合 {json.dumps(_audit['verify_status'])}、
  複数の異なる box を持つ行 {_audit['rows_multi_distinct_boxes']}、照合不一致で目視確認のうえ保持した行 {AO_ACKNOWLEDGED_VERIFY_MISMATCH_ROWS}。
- AO parquet では元の 20,480 行・行順・`message`・`advantage` を維持。抽出失敗時の `answer` 代用や行削除はしない。
- prompt は公式と同一で「step by step」の指示を含むが、target は回答のみ。

## 公式 CoT 学習（{_saved_spec.get('official_world_size', 8)} GPU）との条件合わせ
{_align_md}

## 学習設定
- run_id `{_final_rid}`, lr {_saved_spec['lr_str']}, epochs {_saved_spec['epochs']}, global batch {_saved_spec['tbs']}, micro batch/GPU {_saved_spec['micro_bsz']}, GPU 数 {_saved_spec['n_gpus']},
  steps/epoch {_saved_spec['steps_per_epoch']}, total steps {_saved_spec['total_steps']}, warmup {_saved_spec['warmup_steps']}, max_length {_saved_spec['max_length']} (公式 {_saved_spec['official_max_length']}), save_freq {_saved_spec['save_freq']}
- 公式スクリプト `{os.path.basename(_saved_spec['official_script'])}` との差分:

| key | 公式 | 本 run | 理由 |
| --- | --- | --- | --- |
{_changes_md}

- 環境: {json.dumps(ENV_RECORD.get('gpus'))}, torch {PKG_VERSIONS.get('torch')}, transformers {PKG_VERSIONS.get('transformers')}, flash-attn {FLASH_ATTN_VERSION}, Colab={IN_COLAB}, GPU プロファイル {_saved_spec.get('gpu_profile', 'official')}{('（公式からの変更: ' + _saved_spec['gpu_profile_deviation'] + '）') if _saved_spec.get('gpu_profile_deviation') else ''}
- 再開用 checkpoint（model/optimizer/extra/dataloader 状態）: `{HF_CKPT_REPO_ID}` の `runs/{_final_rid}/global_step_*`

## 評価（論文と同じ方法）
公式 `evaluation/math_eval/math_eval_budget.py` を無変更で実行（vLLM 0.8.5, temperature 0.6, top_p 0.95, 最大 32768 token, math-verify）。
MATH500 は avg@3、AIME24 は avg@10（%）。括弧内は同じ step の公開 CoT 学生の論文値（App. D）。論文との差: {'; '.join(PAPER_EVAL_DEVIATIONS)}。

| step | {' | '.join(PAPER_EVAL_DATASETS)} | {PAPER_EVAL_DATASETS[0]} の平均出力長 (token) |
| --- | {' | '.join('---' for _ in PAPER_EVAL_DATASETS)} | --- |
{_pe_md}

loss の低下だけで能力向上は主張しない。上の表が「未実施」なら性能は未測定。

## 評価の範囲
- 研究目的は数学能力への CoT の寄与を調べることなので、評価は MATH500 / AIME24 だけ。論文の数学以外の評価（LiveCodeBench, GPQA-Diamond, MMLU-Pro, IFEval, AlpacaEval, HaluEval, TruthfulQA, HEx-PHI）は行っていない。
"""
    with open(os.path.join(MERGED_DIR, "..", "README.md"), "w") as f:
        f.write(MODEL_CARD)
    print("uploading merged model to", FINAL_REPO, "/", _sub)
    _info = HF_API.upload_folder(folder_path=MERGED_DIR, path_in_repo=_sub, repo_id=FINAL_REPO, repo_type="model", commit_message=f"{_final_rid} merged step {_step}")
    _info_rd = HF_API.upload_file(path_or_fileobj=os.path.join(MERGED_DIR, "..", "README.md"), path_in_repo="README.md", repo_id=FINAL_REPO, repo_type="model",
                                  commit_message=f"model card for {_final_rid}")
    FINAL_MODEL = dict(repo=FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, weights_commit=_info.oid, url=f"https://huggingface.co/{FINAL_REPO}/tree/{_info_rd.oid}/{_sub}",
                       source_checkpoint=dict(repo=HF_CKPT_REPO_ID, path=f"runs/{_final_rid}/global_step_{_step}"))
    print("uploaded:", FINAL_MODEL)

    # 読み込みと短い生成の確認（アップロード後の revision から）
    _tok = AutoTokenizer.from_pretrained(FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, trust_remote_code=True)
    _mdl = AutoModelForCausalLM.from_pretrained(FINAL_REPO, subfolder=_sub, revision=_info_rd.oid, torch_dtype=torch.bfloat16, trust_remote_code=True).cuda().eval()
    _probe_prompts = [list(RAW_TABLE.column("message")[0].as_py()),
                      [{"role": "user", "content": "What is 17 + 26?\n Please reason step by step, and put your final answer within \\boxed{}."}]]
    GEN_CHECK = []
    for _p in _probe_prompts:
        _enc = _tok.apply_chat_template(_p, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        with torch.no_grad():
            _g = _mdl.generate(**_enc, max_new_tokens=64, do_sample=False, eos_token_id=_tok.eos_token_id, pad_token_id=_tok.pad_token_id)
        _out = _tok.decode(_g[0, _enc["input_ids"].shape[1]:], skip_special_tokens=False)
        GEN_CHECK.append(dict(prompt=_p[0]["content"][:80], output=_out))
        print("prompt:", _p[0]["content"][:80].replace("\n", " "), "\n  -> output:", repr(_out))
    del _mdl; torch.cuda.empty_cache()
    FINAL_MODEL["generation_check"] = GEN_CHECK
    _own_results = [r for r in globals().get("TRAIN_RESULTS", {}).values() if r["run_id"] == _final_rid]
    write_experiment_record(_saved_spec, _own_results, evaluations=_paper_evals, final_model=FINAL_MODEL,
                            unverified=(["論文と同じ評価（MATH500/AIME24）未実施"] if not _paper_evals else []) +
                                       (["このセッションの学習ログなし（HF の experiment_record を参照）"] if not _own_results else []))